In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Any
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.patches import Rectangle, ConnectionPatch
from matplotlib.colors import TwoSlopeNorm
from pathlib import Path
import re


data_hourly =pd.read_excel('XX.xlsx', index_col=0, parse_dates=True)  


CONSTANTS

In [ ]:
rho = 980 # kg/m3
c = 4.18 # kJ/kgK
mu = 0.001 # Pa.s
k_fluid = 0.6 # W/mK
m_dot_max = 147.0 # kg/s
max_temp =120 # C
buffer_capacity_sizing_factor = 1
buffer_capacity_mwh_max_soc = 75 #mwh
buffer_power_sizing_factor = 1
buffer_power_mw_max = 16 # MW
delta_P_customer = 1.5 # bar
v_water = 0.001 #m3/kg
pump_efficiency = 0.85
cp = 4180.0
charge_efficiency = 0.95
discharge_efficiency = 0.95
# Physical buffer helper functions
RHO_WATER = 980   # kg/m3
CP_WATER = 4180.0    # J/kgK

In [ ]:
#Economic parameters

AVR_price_per_GJ = XX                         #Eur/GJ
AVR_price_per_mwh = AVR_price_per_GJ *3.6       #Eur/MWh
HWC_price_per_GJ = XX                        #Eur/GJ
HWC_price_per_mwh = HWC_price_per_GJ *3.6       #Eur/MWh
price_buffer = XX #eur/m3

cost_params = {
    # Heat production costs
    "avr_heat_cost_eur_per_mwh": AVR_price_per_mwh,      # €/MWh_th
    "gas_heat_cost_eur_per_mwh": HWC_price_per_mwh,      # €/MWh_th delivered
    
    # Electricity cost for pumping and possible HP
    "electricity_cost_eur_per_mwh": XXX,  # €/MWh_el
    
    # O&M
    "fixed_om_eur_per_year": XX,      # €/year
    "variable_om_eur_per_mwh": XX,         # €/MWh_th delivered
}


emission_factors = {
    2030: {"gas_GJ": XX, "avr_GJ": XX, "elec_kWh": XX},
    2040: {"gas_GJ": XX, "avr_GJ": XX, "elec_kWh": XX},
    2050: {"gas_GJ": XX, "avr_GJ": XX, "elec_kWh": XX},
}
co2_gas_GJ = XX #kg/GJheat
co2_avr_GJ = XX #kg/GJheat
co2_electricity_kWh = XX #kg/kWh

co2_gas_mwh = co2_gas_GJ*3.6
co2_avr_mwh = co2_avr_GJ*3.6
co2_electricity_mwh = co2_electricity_kWh*1000

PLOTS TO FOLDER FUNCTION

In [ ]:
output_dir = Path("Figures/BFCS_F=" + str(buffer_capacity_sizing_factor))
output_dir.mkdir(exist_ok=True)

def clean_filename(title):
    title = str(title).strip()
    title = re.sub(r"[^\w\s.-]", "", title)
    title = re.sub(r"\s+", "_", title)
    return title if title else "untitled_plot"

def save_current_plot_from_title(save_title=None, dpi=300):
    fig = plt.gcf()
    ax = plt.gca()

    # Use provided save_title, otherwise use visible plot title
    if save_title is None:
        save_title = ax.get_title()

    if not save_title:
        save_title = "untitled_plot"

    filename = clean_filename(save_title) + "_BFCS_" + str(buffer_capacity_sizing_factor) + ".pdf"
    filepath = output_dir / filename

    fig.savefig(filepath, dpi=dpi, bbox_inches="tight")

    print(f"Saved as: {filepath}")

HEAT LOSS SEPARATION


In [ ]:
heat_loss_table = pd.read_excel('XX.xlsx')
Ts_vals = heat_loss_table["Ts_C"].to_numpy()
hl_vals = heat_loss_table["Heat_loss_mw"].to_numpy() 

def lookup_heat_loss(Ts_day):
    return np.interp(Ts_day, Ts_vals, hl_vals)

Ts_vals_year = heat_loss_table['T_net'].to_numpy()
hl_vals_year = heat_loss_table['Heat_loss_mw_net'].to_numpy()

def lookup_heat_loss_year(Ts_day):
    return np.interp(Ts_day, Ts_vals_year, hl_vals_year)


data_hourly["total_source_production_mw"] = data_hourly["TOT_ARN_MW"]
data_hourly["heat_loss_real_mw"] = data_hourly["AVR_ARN_TA"].apply(lookup_heat_loss_year)
data_hourly["useful_demand_mw"] = (
    data_hourly["TOT_ARN_MW"] - data_hourly["heat_loss_real_mw"]
).clip(lower=0.0)
data_hourly["heat_loss_at_max_temp_mw"] = lookup_heat_loss_year(max_temp)
data_hourly["network_load_at_max_temp_mw"] = (
    data_hourly["useful_demand_mw"] + data_hourly["heat_loss_at_max_temp_mw"]
)

print("Observed production (MWh):", round(data_hourly["TOT_ARN_MW"].sum(), 2))
print("Estimated useful demand (MWh):", round(data_hourly["useful_demand_mw"].sum(), 2))
print("Estimated real heat loss (MWh):", round(data_hourly["heat_loss_real_mw"].sum(), 2))


TES VOLUME AND CAPAXCITY


In [ ]:


def buffer_capacity_mwh_from_volume(buffer_volume_m3, T_hot_C, T_cold_C, rho=RHO_WATER, cp=CP_WATER):
    dT = max(float(T_hot_C) - float(T_cold_C), 0.0)
    return rho * buffer_volume_m3 * cp * dT / 3.6e9

def buffer_charge_limit_mwh(mdot_charge_max_kg_s, T_charge_C, T_tank_cold_C, dt_hours=1.0, cp=CP_WATER):
    dT = max(float(T_charge_C) - float(T_tank_cold_C), 0.0)
    return mdot_charge_max_kg_s * cp * dT * dt_hours * 3600.0 / 3.6e9

def buffer_discharge_limit_mwh(mdot_discharge_max_kg_s, T_tank_hot_C, T_return_C, dt_hours=1.0, cp=CP_WATER):
    dT = max(float(T_tank_hot_C) - float(T_return_C), 0.0)
    return mdot_discharge_max_kg_s * cp * dT * dt_hours * 3600.0 / 3.6e9

In [ ]:
#calculate minimum supply temperature based on useful demand, maximum flow and return temperature
def calculate_min_supply_temp(Q_demand_MW, cp, m_dot_max, Treturn):

    Q = pd.Series(Q_demand_MW)
    Treturn = pd.Series(Treturn)

    #convert mw to w
    Qdot = (Q * 1e6)  # [W]

    # Compute minimum supply temperature [°C]
    Tsupply_min = Treturn + (Qdot / (m_dot_max * cp))

    Tsupply_min.name = "T_in_min_C"
    return Tsupply_min

def calculate_min_supply_temp_with_heat_loss(
    useful_demand_MW,
    cp,
    m_dot_max,
    Treturn,
    heat_loss_lookup=lookup_heat_loss_year,
    min_supply_temp=80.0,
    max_iter=50,
    tol_C=1e-4,
):

    useful = pd.Series(useful_demand_MW).astype(float)
    Treturn = pd.Series(Treturn).astype(float).reindex(useful.index)

    Ts = calculate_min_supply_temp(useful, cp, m_dot_max, Treturn).clip(lower=min_supply_temp)

    for _ in range(max_iter):
        heat_loss_mw = pd.Series(heat_loss_lookup(Ts), index=Ts.index).astype(float)
        total_load_mw = useful + heat_loss_mw
        Ts_new = calculate_min_supply_temp(total_load_mw, cp, m_dot_max, Treturn).clip(lower=min_supply_temp)

        if float((Ts_new - Ts).abs().max()) < tol_C:
            Ts = Ts_new
            break
        Ts = Ts_new

    Ts.name = "T_in_min_C"
    return Ts

min_temp = calculate_min_supply_temp_with_heat_loss(data_hourly['useful_demand_mw'], cp=4180.0, m_dot_max=m_dot_max, Treturn=data_hourly['AVR_ARN_TR_monthly']).apply(lambda x: max(x, 80))
min_temp_130 = calculate_min_supply_temp_with_heat_loss(data_hourly['useful_demand_mw'], cp=4180.0, m_dot_max=130, Treturn=data_hourly['AVR_ARN_TR_monthly']).apply(lambda x: max(x, 80))
min_temp_140 = calculate_min_supply_temp_with_heat_loss(data_hourly['useful_demand_mw'], cp=4180.0, m_dot_max=140, Treturn=data_hourly['AVR_ARN_TR_monthly']).apply(lambda x: max(x, 80))
min_temp_150 = calculate_min_supply_temp_with_heat_loss(data_hourly['useful_demand_mw'], cp=4180.0, m_dot_max=150, Treturn=data_hourly['AVR_ARN_TR_monthly']).apply(lambda x: max(x, 80))
min_temp_160 = calculate_min_supply_temp_with_heat_loss(data_hourly['useful_demand_mw'], cp=4180.0, m_dot_max=160, Treturn=data_hourly['AVR_ARN_TR_monthly']).apply(lambda x: max(x, 80))
#plot min temp against time
#add min temp to data_hourly
data_hourly['Ts_min_no_buffer'] = min_temp
plt.figure(figsize=(12, 6))
plt.plot(min_temp.index, min_temp.values, label='Min Supply Temperature (°C)', color='orange')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.grid(True)
plt.tight_layout()
plot_title='Minimum required supply temperature correct'
save_current_plot_from_title(save_title=plot_title)
plt.show()


In [ ]:
#calculate maximum supply temperature per day over the year
maximum_daily_temp = min_temp.resample('D').max()
maximum_daily_temp_130 = min_temp_130.resample('D').max()
maximum_daily_temp_160 = min_temp_160.resample('D').max()
#fill maximum_daily_temp for every day with the maximum value of that day
maximum_daily_temp_filled = maximum_daily_temp.reindex(data_hourly.index, method='ffill')
#plot maximum daily temp
plt.figure(figsize=(12,6))
plt.plot(maximum_daily_temp.index, maximum_daily_temp.values, label='147 kg/s')
plt.plot(maximum_daily_temp_130.index, maximum_daily_temp_130.values, label='130 kg/s')
plt.plot(maximum_daily_temp_160.index, maximum_daily_temp_160.values, label='160 kg/s')
plt.title('Highest daily minimum supply temperature')
plt.ylabel('Temperature (°C)')
plt.xlabel('Date')
plt.legend()
plt.grid(True)
save_current_plot_from_title()
plt.show()

#make a new profile where the maximum temperature is 120 degrees
maximum_daily_temp_with_cap = maximum_daily_temp.copy()
maximum_daily_temp_with_cap[maximum_daily_temp_with_cap > max_temp] = max_temp
plt.figure(figsize=(12,6))
plt.plot(maximum_daily_temp.index, maximum_daily_temp.values, label='147 kg/s', color='orange')
plot_title='Highest daily minimum supply temperature'
plt.ylabel('Temperature (°C)', fontsize=14)
plt.xlabel('Date', fontsize=14)
plt.grid(True)
save_current_plot_from_title(save_title=plot_title)
plt.show()

In [ ]:


Ts_hourly = maximum_daily_temp_with_cap.reindex(data_hourly.index, method='ffill')
T_return_hourlyxx = data_hourly['AVR_ARN_TR_monthly']
heat_loss_base_hourlyxx = Ts_hourly.apply(lookup_heat_loss_year)
demand_mw_hourlyxx = data_hourly['useful_demand_mw'] + heat_loss_base_hourlyxx


delta_T = Ts_hourly - T_return_hourlyxx

delta_T = delta_T.where(delta_T > 0)

mass_flow_hourly = (demand_mw_hourlyxx * 1e6) / (cp * delta_T)
mass_flow_hourly_capped = mass_flow_hourly.clip(upper=m_dot_max)

# Resultaat als nette Series
mass_flow_hourly = pd.Series(
    mass_flow_hourly,
    index=data_hourly.index,
    name='required_mass_flow_kg_s'
)

plt.figure(figsize=(18, 6))
plt.scatter(mass_flow_hourly_capped.index, mass_flow_hourly_capped.values, label='Required mass flow (kg/s)',s=5)
plt.xlabel('Date', fontsize = 14)
plt.ylabel('Mass flow (kg/s)', fontsize = 14)
plt.ylim(20, 165)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)
plot_title='Required mass flow per hour'
save_current_plot_from_title(save_title=plot_title)
plt.show()

In [ ]:
#maximum energy supply in mw

maximum_heat_supply = (m_dot_max*cp*(max_temp-data_hourly['AVR_ARN_TR_monthly']))/1000000
print(maximum_heat_supply.head())
plt.figure(figsize=(12,6))
plt.plot(maximum_heat_supply.index, maximum_heat_supply.values, color='red', alpha = 0.5, label='Maximum heat supply')
plt.plot(data_hourly['network_load_at_max_temp_mw'].index, data_hourly['network_load_at_max_temp_mw'].values, color='blue', alpha=0.5, label='Useful demand + heat loss at 120°C')
plt.title('Maximum heat supply and hourly heat demand')
plt.ylabel('Heat (MW)')
plt.xlabel('Date')
plt.legend()
plt.grid(True)
save_current_plot_from_title()
plt.show()

#calculate total available heat supply per day
total_available_heat_supply = maximum_heat_supply.resample('D').sum()
plt.figure(figsize=(12,6))
plt.plot(total_available_heat_supply.index, total_available_heat_supply.values, color='red', alpha = 0.5, label='Maximum heat supply per day')
plt.plot(data_hourly['network_load_at_max_temp_mw'].resample('D').sum().index, data_hourly['network_load_at_max_temp_mw'].resample('D').sum().values  , color='blue', alpha=0.5, label='Daily useful demand + heat loss at 120°C')
plt.title('Total available heat supply per day and daily heat demand')
plt.ylabel('Heat (MWh)')
plt.xlabel('Date')
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()


In [ ]:
# Hourly MW values
maximum_heat_supply = (
    m_dot_max * cp * (max_temp - data_hourly["AVR_ARN_TR_monthly"])
) / 1_000_000  # MW

required_heat = data_hourly["network_load_at_max_temp_mw"]  # MW

# Hourly shortage in MW
hourly_shortage = (required_heat - maximum_heat_supply).clip(lower=0)

# Daily MWh values
daily_available_heat_supply = maximum_heat_supply.resample("D").sum()
daily_required_heat = required_heat.resample("D").sum()

# Daily shortage in MWh
daily_shortage = (daily_required_heat - daily_available_heat_supply).clip(lower=0)

fig, ax1 = plt.subplots(figsize=(15, 7.5))

# -------------------------
# Left axis: hourly MW
# -------------------------
line_max_hourly, = ax1.plot(
    maximum_heat_supply.index,
    maximum_heat_supply.values,
    color="red",
    linewidth=1.4,
    alpha=0.9,
    label="Maximum hourly heat supply",
    zorder=4
)

line_required_hourly, = ax1.plot(
    required_heat.index,
    required_heat.values,
    color="steelblue",
    linewidth=0.8,
    alpha=0.35,
    label="Hourly network demand",
    zorder=2
)

# Hourly shortage shading
hourly_shortage_color = "orange"

ax1.fill_between(
    required_heat.index,
    maximum_heat_supply.values,
    required_heat.values,
    where=required_heat.values > maximum_heat_supply.values,
    color=hourly_shortage_color,
    alpha=0.55,
    interpolate=True,
    zorder=5
)

hourly_shortage_patch = Patch(
    facecolor=hourly_shortage_color,
    edgecolor=hourly_shortage_color,
    alpha=0.55,
    label="Hourly shortage"
)

ax1.set_ylabel("Hourly heat [MW]", fontsize=14)
ax1.set_xlabel("Date", fontsize=14)
ax1.set_ylim(0, 80)
ax1.grid(True, alpha=0.45)

# -------------------------
# Right axis: daily MWh
# -------------------------
ax2 = ax1.twinx()

line_max_daily, = ax2.step(
    daily_available_heat_supply.index,
    daily_available_heat_supply.values,
    where="post",
    color="black",
    linewidth=2.0,
    linestyle=":",
    alpha=0.9,
    label="Daily maximum heat supply",
    zorder=1
)

line_required_daily, = ax2.plot(
    daily_required_heat.index,
    daily_required_heat.values,
    color="grey",
    linewidth=2.0,
    linestyle=":",
    alpha=0.95,
    label="Daily network demand",
    zorder=1
)

# Daily shortage shading
daily_shortage_color = "lightcoral"

ax2.fill_between(
    daily_required_heat.index,
    daily_available_heat_supply.values,
    daily_required_heat.values,
    where=daily_required_heat.values > daily_available_heat_supply.values,
    color=daily_shortage_color,
    alpha=0.45,
    interpolate=True,
    zorder=2
)

daily_shortage_patch = Patch(
    facecolor=daily_shortage_color,
    edgecolor=daily_shortage_color,
    alpha=0.45,
    label="Daily shortage"
)

ax2.set_ylabel("Daily heat [MWh]", fontsize=14)
ax2.set_ylim(0, 1100)

# -------------------------
# Legend outside figure
# -------------------------
handles = [
    line_max_hourly,
    line_required_hourly,
    hourly_shortage_patch,
    line_max_daily,
    line_required_daily,
    daily_shortage_patch
]

labels = [h.get_label() for h in handles]

ax1.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(1.12, 0.5),
    fontsize=12,
    framealpha=0.9,
    borderpad=0.25,
    labelspacing=0.25,
    handlelength=1.4,
    handletextpad=0.4,
    borderaxespad=0.3
)

fig.subplots_adjust(left=0.08, right=0.76, bottom=0.11, top=0.96)

plot_title = "Hourly and daily heat supply capacity, demand, and shortage"
save_current_plot_from_title(save_title=plot_title)

plt.show()

Calculate when the AVR + BIO cant supply enough for schuytgraaf and make a new theoretical supply profile.


In [ ]:
#calculate shortage in heat supply
shortage = pd.Series(index=maximum_heat_supply.index, dtype=float)

for i in maximum_heat_supply.index:
    if maximum_heat_supply[i] < data_hourly.loc[i, 'network_load_at_max_temp_mw']:
        shortage[i] = (
            data_hourly.loc[i, 'network_load_at_max_temp_mw']
            - maximum_heat_supply[i]
        )
    else:
        shortage[i] = 0.0
#plot shortage and heat produced with gas with same index
plt.figure(figsize=(12,6))
plt.plot(data_hourly.index, shortage, label='Shortage (MW)', color='red')
plt.title('Hourly shortage in heat supply')
plt.ylabel('Heat (MW)')
plt.xlabel('Date')
plt.grid(True)
save_current_plot_from_title()
plt.show()

base_scenario = pd.DataFrame(index=data_hourly.index)

# Gas use = shortage
base_scenario["gas_use"] = shortage

# AVR heat = total demand - shortage
base_scenario["avr_heat"] = data_hourly["network_load_at_max_temp_mw"] - base_scenario['gas_use']

In [ ]:
#calculate shortage in mwh per day
daily_shortage_mwh = pd.Series(shortage, index=data_hourly.index).resample('D').sum()
plt.figure(figsize=(12,6))
plt.plot(daily_shortage_mwh.index, daily_shortage_mwh.values, color='red')
plt.title('Daily shortage in heat supply')
plt.ylabel('Heat (MWh)')
plt.xlabel('Date')
plt.grid(True)
save_current_plot_from_title()
plt.show()

In [ ]:
# calculate equivalent old MW/MWh sizing first
#buffer_thermal_power_mw_equiv = math.ceil(max(shortage)) * buffer_power_sizing_factor


#buffer_capacity_mwh_equiv = math.ceil(max(shortage.resample('D').sum())) * buffer_capacity_sizing_factor
buffer_capacity_mwh_equiv = buffer_capacity_mwh_max_soc * buffer_capacity_sizing_factor
print(buffer_capacity_mwh_equiv)
buffer_thermal_power_mw_equiv = buffer_capacity_mwh_equiv/5
print(buffer_thermal_power_mw_equiv)

# convert to physical design parameters
T_buffer_hot_design_C = float(max_temp)
T_buffer_cold_design_C = float(data_hourly['AVR_ARN_TR_monthly'].mean())
design_dT_buffer_K = max(T_buffer_hot_design_C - T_buffer_cold_design_C, 1e-6)

buffer_volume_m3 = buffer_capacity_mwh_equiv * 3.6e9 / (RHO_WATER * CP_WATER * design_dT_buffer_K)
buffer_charge_mdot_max_kg_s = buffer_thermal_power_mw_equiv * 1e6 / (CP_WATER * design_dT_buffer_K)
buffer_discharge_mdot_max_kg_s = buffer_thermal_power_mw_equiv * 1e6 / (CP_WATER * design_dT_buffer_K)

print(buffer_volume_m3)
print(buffer_charge_mdot_max_kg_s)
print(buffer_discharge_mdot_max_kg_s)

# calculate if the buffer can be charged the same day of the shortage, by looking at the maximum heat supply and the shortage on the same day
buffer_capacity_mwh_design = buffer_capacity_mwh_from_volume(
    buffer_volume_m3,
    T_buffer_hot_design_C,
    T_buffer_cold_design_C
)

can_charge_same_day = buffer_capacity_mwh_design < (
    maximum_heat_supply.resample('D').sum() - data_hourly['network_load_at_max_temp_mw'].resample('D').sum()
)
plt.figure(figsize=(12,6))
plt.plot(can_charge_same_day.index, can_charge_same_day.values, color='green')
# how many days cant be charged the same day
num_days_cant_charge_same_day = sum(~can_charge_same_day)
print(f"Number of days that can't be charged the same day: {num_days_cant_charge_same_day}")

In [ ]:
def buffer_behaviour_replace_gas(
    buffer_volume_m3,
    buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s,
    maximum_heat_supply_mw,
    ts_day,
    data_hourly,
    shortage_mw,
    dt_hours=1.0
):
    idx = data_hourly.index
    soc = pd.Series(index=idx, dtype=float)
    soc.iloc[0] = 0.0

    charge = pd.Series(index=idx, dtype=float)
    discharge = pd.Series(index=idx, dtype=float)
    backup = pd.Series(index=idx, dtype=float)
    curtail = pd.Series(index=idx, dtype=float)
    charge_mdot_kg_s = pd.Series(0.0, index=idx, dtype=float)
    discharge_mdot_kg_s = pd.Series(0.0, index=idx, dtype=float)
    for i in range(1, len(idx)):
        prev = soc.iloc[i-1]

        Ts_i = float(ts_day.iloc[i])
        Tret_i = float(data_hourly['AVR_ARN_TR_monthly'].iloc[i])

        heat_loss_mw = lookup_heat_loss_year(Ts_i)
        demand_mwh = (data_hourly['useful_demand_mw'].iloc[i] + heat_loss_mw) * dt_hours
        max_supply_mwh = maximum_heat_supply_mw.iloc[i] * dt_hours
        shortage_mwh = shortage_mw.iloc[i] * dt_hours
        dT_buffer_K = max(Ts_i - Tret_i, 1e-9)
        cap_buffer_mwh = buffer_capacity_mwh_from_volume(buffer_volume_m3, Ts_i, Tret_i)
        charge_limit_mwh = buffer_charge_limit_mwh(
            buffer_charge_mdot_max_kg_s, Ts_i, Tret_i, dt_hours, cp
        )
        discharge_limit_mwh = buffer_discharge_limit_mwh(
            buffer_discharge_mdot_max_kg_s, Ts_i, Tret_i, dt_hours, cp
        )
        
        if shortage_mwh > 0:
            dis = min(shortage_mwh, discharge_limit_mwh, prev)
            discharge_mdot_kg_s.iloc[i] = dis * 1e6 / (cp * dT_buffer_K * dt_hours)
            new_soc = prev - dis

            discharge.iloc[i] = dis
            charge.iloc[i] = 0.0
            backup.iloc[i] = shortage_mwh - dis
            curtail.iloc[i] = 0.0
            charge_mdot_kg_s.iloc[i] = 0.0
        else:
            surplus_mwh = max_supply_mwh - demand_mwh

            if surplus_mwh > 0:
                ch = min(surplus_mwh, charge_limit_mwh, max(0.0, cap_buffer_mwh - prev))
                charge_mdot_kg_s.iloc[i] = ch * 1e6 / (cp * dT_buffer_K * dt_hours)
                new_soc = prev + ch

                charge.iloc[i] = ch
                discharge.iloc[i] = 0.0
                curtail.iloc[i] = surplus_mwh - ch
                backup.iloc[i] = 0.0
                discharge_mdot_kg_s.iloc[i] = 0.0
            else:
                new_soc = prev
                charge.iloc[i] = discharge.iloc[i] = backup.iloc[i] = curtail.iloc[i] = 0.0

        soc.iloc[i] = max(0.0, min(cap_buffer_mwh, new_soc))

    return soc, charge, discharge, backup, curtail, charge_mdot_kg_s, discharge_mdot_kg_s

soc, charge, discharge, backup, curtail, charge_mdot_kg_s, discharge_mdot_kg_s = buffer_behaviour_replace_gas(
    buffer_volume_m3=buffer_volume_m3,
    buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
    maximum_heat_supply_mw=maximum_heat_supply,
    ts_day=maximum_daily_temp_filled,
    data_hourly=data_hourly,
    shortage_mw=shortage
)
plt.figure(figsize=(20,6))
#plt.plot(data_hourly.index, soc.values, label='SOC', color='blue')
plt.plot(data_hourly.index, discharge.values, label='Discharge', color='red')
plt.title('Buffer discharge over the year')
plt.xlabel('Date')
plt.ylabel('Discharge (MW)')
plt.grid(True)
save_current_plot_from_title()
plt.show()
print(soc.head())



In [ ]:
#daily supply temp reality AVR
AVR_Ts = data_hourly['AVR_ARN_TA'].resample('D').mean()


In [ ]:
def simulate_day_soc_feasibility(
    day_df,
    maximum_heat_supply_mw,
    Ts_day,
    soc_start_mwh,
    buffer_volume_m3,
    buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s,
    m_dot_max,
    cp=cp,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw',
    dt_hours=1.0,
    charge_efficiency=charge_efficiency,
    discharge_efficiency=discharge_efficiency
):
    idx = day_df.index
    soc = pd.Series(index=idx, dtype=float)
    soc.iloc[0] = soc_start_mwh

    unmet_mwh = 0.0
    heat_loss_mw = float(lookup_heat_loss_year(Ts_day))

    for i in range(1, len(idx)):
        prev = soc.iloc[i-1]

        Tret = float(day_df[T_return_col].iloc[i])
        demand_mw = float(day_df[demand_col].iloc[i])

        cap_mw = (m_dot_max * cp * max(Ts_day - Tret, 0.0)) / 1e6
        plant_mw = cap_mw

        cap_buffer_mwh = buffer_capacity_mwh_from_volume(
            buffer_volume_m3=buffer_volume_m3,
            T_hot_C=Ts_day,
            T_cold_C=Tret
        )

        charge_limit_mwh = buffer_charge_limit_mwh(
            mdot_charge_max_kg_s=buffer_charge_mdot_max_kg_s,
            T_charge_C=Ts_day,
            T_tank_cold_C=Tret,
            dt_hours=dt_hours,
            cp=cp
        )

        discharge_limit_mwh = buffer_discharge_limit_mwh(
            mdot_discharge_max_kg_s=buffer_discharge_mdot_max_kg_s,
            T_tank_hot_C=Ts_day,
            T_return_C=Tret,
            dt_hours=dt_hours,
            cp=cp
        )

        net_mw = plant_mw - demand_mw - heat_loss_mw
        new_soc = prev

        if net_mw >= 0:
            surplus_mwh = net_mw * dt_hours
            ch = min(
                surplus_mwh,
                charge_limit_mwh,
                max(0.0, (cap_buffer_mwh - prev) / max(charge_efficiency, 1e-12))
            )
            new_soc = prev + ch * charge_efficiency
        else:
            deficit_mwh = (-net_mw) * dt_hours
            dis = min(deficit_mwh, discharge_limit_mwh, prev * discharge_efficiency)
            new_soc = prev - dis / max(discharge_efficiency, 1e-12)
            unmet_mwh += (deficit_mwh - dis)

        soc.iloc[i] = max(0.0, min(cap_buffer_mwh, new_soc))

    feasible = (unmet_mwh <= 1e-9)
    soc_end = float(soc.iloc[-1])

    return feasible, soc_end, unmet_mwh

def find_minimum_daily_Ts(
    day_df,
    maximum_heat_supply_mw,
    soc_start_mwh,
    buffer_volume_m3,
    buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s,
    m_dot_max,
    cp=4180.0,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw',
    dt_hours=1.0,
    Ts_hi=max_temp,
    Ts_lo=80,
    tol_C=0.05,
    max_iter=100
):
    Tret = day_df[T_return_col]
    default_lo = float(Tret.max())
    Ts_lo = default_lo if Ts_lo is None else Ts_lo

    if Ts_hi is None:
        if 'Ts_min_no_buffer' in day_df.columns:
            Ts_hi = float(day_df['Ts_min_no_buffer'].max())
        else:
            Ts_hi = 160

    feasible_hi, soc_end_hi, unmet_hi = simulate_day_soc_feasibility(
        day_df=day_df,
        maximum_heat_supply_mw=maximum_heat_supply_mw,
        Ts_day=Ts_hi,
        soc_start_mwh=soc_start_mwh,
        buffer_volume_m3=buffer_volume_m3,
        buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
        buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
        m_dot_max=m_dot_max,
        cp=cp,
        T_return_col=T_return_col,
        demand_col=demand_col,
        dt_hours=dt_hours,
        charge_efficiency=charge_efficiency,
        discharge_efficiency=discharge_efficiency,
    )
    if not feasible_hi:
        return Ts_hi, soc_end_hi, False, unmet_hi

    lo, hi = Ts_lo, Ts_hi
    soc_end_best = soc_end_hi

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        feasible, soc_end, unmet_hi = simulate_day_soc_feasibility(
            day_df=day_df,
            maximum_heat_supply_mw=maximum_heat_supply_mw,
            Ts_day=mid,
            soc_start_mwh=soc_start_mwh,
            buffer_volume_m3=buffer_volume_m3,
            buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
            buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
            m_dot_max=m_dot_max,
            cp=cp,
            T_return_col=T_return_col,
            demand_col=demand_col,
            dt_hours=dt_hours,
            charge_efficiency=charge_efficiency,
            discharge_efficiency=discharge_efficiency,
        )
        if feasible:
            hi = mid
            soc_end_best = soc_end
        else:
            lo = mid

        if (hi - lo) <= tol_C:
            break

    return hi, soc_end_best, True, 0.0

def compute_daily_min_Ts_with_buffer_daily_reset(
    data_hourly,
    maximum_heat_supply_mw,
    buffer_volume_m3,
    buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s,
    m_dot_max,
    cp=4180.0,
    dt_hours=1.0,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw',
    tol_C=1e-4,
    soc_start_each_day_mwh=0.0
):
    maximum_heat_supply_mw = maximum_heat_supply_mw.reindex(data_hourly.index)
    daily_groups = data_hourly.groupby(data_hourly.index.floor('D'))
    results = []

    for day, day_df in daily_groups:
        day_idx = day_df.index
        max_supply_day = maximum_heat_supply_mw.loc[day_idx]

        Ts_day, soc_end, ok, unmet = find_minimum_daily_Ts(
            day_df=day_df,
            maximum_heat_supply_mw=max_supply_day,
            soc_start_mwh=soc_start_each_day_mwh,
            buffer_volume_m3=buffer_volume_m3,
            buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
            buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
            m_dot_max=m_dot_max,
            cp=cp,
            T_return_col=T_return_col,
            demand_col=demand_col,
            dt_hours=dt_hours,
            tol_C=tol_C
        )

        results.append({
            'date': day,
            'Ts_min_day_with_buffer_C': Ts_day,
            'soc_start_mwh': soc_start_each_day_mwh,
            'soc_end_mwh': soc_end,
            'feasible': ok,
            'unmet_mwh': unmet
        })

    return pd.DataFrame(results).set_index('date')

daily_Ts = compute_daily_min_Ts_with_buffer_daily_reset(
    data_hourly=data_hourly,
    maximum_heat_supply_mw=maximum_heat_supply,
    buffer_volume_m3=buffer_volume_m3,
    buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
    m_dot_max=m_dot_max,
    cp=cp,
    tol_C=1e-4,
    soc_start_each_day_mwh=0.0
)
daily_Ts['Ts_min_day_with_buffer_C_80'] = daily_Ts['Ts_min_day_with_buffer_C'].apply(lambda x: max(x, 80))
plt.figure(figsize=(20,8))
plt.plot(daily_Ts.index, daily_Ts['Ts_min_day_with_buffer_C_80'], color='purple', label='Daily minimum supply temperature with buffer')
plt.plot(maximum_daily_temp.index, maximum_daily_temp.values, color='orange', label='Daily minimum supply temperature (No Buffer)')
plt.plot(AVR_Ts.index, AVR_Ts.values, label='Real supply temperatures 2024')
plt.axhline(y=max_temp, color='red', linestyle='--', label='Maximum supply temperature (120°C)')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True)
plot_title='Daily supply temperatures'
save_current_plot_from_title(save_title=plot_title)
plt.show()

Kopie van hierboven, voro 6h charge regulation

In [ ]:
def simulate_day_soc_feasibility(
    day_df,
    maximum_heat_supply_mw,
    Ts_day,
    soc_start_mwh,
    buffer_volume_m3,
    buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s,
    m_dot_max,
    cp=cp,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw',
    dt_hours=1.0,
    charge_efficiency=charge_efficiency,
    discharge_efficiency=discharge_efficiency
):
    idx = day_df.index
    soc = pd.Series(index=idx, dtype=float)
    soc.iloc[0] = soc_start_mwh

    unmet_mwh = 0.0
    
    heat_loss_mw = float(lookup_heat_loss_year(Ts_day))

    for i in range(1, len(idx)):
        prev = soc.iloc[i-1]

        Tret = float(day_df[T_return_col].iloc[i])
        demand_mw = float(day_df[demand_col].iloc[i])

        cap_mw = (m_dot_max * cp * max(Ts_day - Tret, 0.0)) / 1e6
        plant_mw = cap_mw

        cap_buffer_mwh = buffer_capacity_mwh_from_volume(
            buffer_volume_m3=buffer_volume_m3,
            T_hot_C=Ts_day,
            T_cold_C=Tret
        )

        charge_limit_mwh = buffer_charge_limit_mwh(
            mdot_charge_max_kg_s=buffer_charge_mdot_max_kg_s,
            T_charge_C=Ts_day,
            T_tank_cold_C=Tret,
            dt_hours=dt_hours,
            cp=cp
        )

        discharge_limit_mwh = buffer_discharge_limit_mwh(
            mdot_discharge_max_kg_s=buffer_discharge_mdot_max_kg_s,
            T_tank_hot_C=Ts_day,
            T_return_C=Tret,
            dt_hours=dt_hours,
            cp=cp
        )

        net_mw = plant_mw - demand_mw - heat_loss_mw
        new_soc = prev

        # CHANGED: only allow charging in the first 6 hours of the day
        can_charge_now = idx[i].hour < 6

        if net_mw >= 0:
            surplus_mwh = net_mw * dt_hours

            # CHANGED: after hour 5, surplus can no longer be stored
            if can_charge_now:
                ch = min(surplus_mwh, charge_limit_mwh, max(0.0, cap_buffer_mwh - prev))
                new_soc = prev + ch * charge_efficiency
            else:
                ch = 0.0
                new_soc = prev

        else:
            deficit_mwh = (-net_mw) * dt_hours
            dis = min(deficit_mwh, discharge_limit_mwh, prev * discharge_efficiency)
            new_soc = prev - dis / max(discharge_efficiency, 1e-12)
            unmet_mwh += (deficit_mwh - dis)

        soc.iloc[i] = max(0.0, min(cap_buffer_mwh, new_soc))

    feasible = (unmet_mwh <= 1e-9)
    soc_end = float(soc.iloc[-1])

    return feasible, soc_end, unmet_mwh


def find_minimum_daily_Ts(
    day_df,
    maximum_heat_supply_mw,
    soc_start_mwh,
    buffer_volume_m3,
    buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s,
    m_dot_max,
    cp=4180.0,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw',
    dt_hours=1.0,
    Ts_hi=max_temp,
    Ts_lo=80,
    tol_C=1e-4,
    max_iter=100
):
    Tret = day_df[T_return_col]
    default_lo = float(Tret.max())
    Ts_lo = default_lo if Ts_lo is None else Ts_lo

    if Ts_hi is None:
        if 'Ts_min_no_buffer' in day_df.columns:
            Ts_hi = float(day_df['Ts_min_no_buffer'].max())
        else:
            Ts_hi = 160

    feasible_hi, soc_end_hi, unmet_hi = simulate_day_soc_feasibility(
        day_df=day_df,
        maximum_heat_supply_mw=maximum_heat_supply_mw,
        Ts_day=Ts_hi,
        soc_start_mwh=soc_start_mwh,
        buffer_volume_m3=buffer_volume_m3,
        buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
        buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
        m_dot_max=m_dot_max,
        cp=cp,
        T_return_col=T_return_col,
        demand_col=demand_col,
        dt_hours=dt_hours,
        charge_efficiency=charge_efficiency,
        discharge_efficiency=discharge_efficiency,
    )
    if not feasible_hi:
        return Ts_hi, soc_end_hi, False, unmet_hi

    lo, hi = Ts_lo, Ts_hi
    soc_end_best = soc_end_hi

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        feasible, soc_end, unmet_hi = simulate_day_soc_feasibility(
            day_df=day_df,
            maximum_heat_supply_mw=maximum_heat_supply_mw,
            Ts_day=mid,
            soc_start_mwh=soc_start_mwh,
            buffer_volume_m3=buffer_volume_m3,
            buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
            buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
            m_dot_max=m_dot_max,
            cp=cp,
            T_return_col=T_return_col,
            demand_col=demand_col,
            dt_hours=dt_hours,
            charge_efficiency=charge_efficiency,
            discharge_efficiency=discharge_efficiency,
        )
        if feasible:
            hi = mid
            soc_end_best = soc_end
        else:
            lo = mid

        if (hi - lo) <= tol_C:
            break

    return hi, soc_end_best, True, 0.0


def compute_daily_min_Ts_with_buffer_daily_reset(
    data_hourly,
    maximum_heat_supply_mw,
    buffer_volume_m3,
    buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s,
    m_dot_max,
    cp=4180.0,
    dt_hours=1.0,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw',
    tol_C=1e-4,
    soc_start_each_day_mwh=0.0
):
    maximum_heat_supply_mw = maximum_heat_supply_mw.reindex(data_hourly.index)
    daily_groups = data_hourly.groupby(data_hourly.index.floor('D'))
    results = []

    for day, day_df in daily_groups:
        day_idx = day_df.index
        max_supply_day = maximum_heat_supply_mw.loc[day_idx]

        Ts_day, soc_end, ok, unmet = find_minimum_daily_Ts(
            day_df=day_df,
            maximum_heat_supply_mw=max_supply_day,
            soc_start_mwh=soc_start_each_day_mwh,
            buffer_volume_m3=buffer_volume_m3,
            buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
            buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
            m_dot_max=m_dot_max,
            cp=cp,
            T_return_col=T_return_col,
            demand_col=demand_col,
            dt_hours=dt_hours,
            tol_C=tol_C
        )

        results.append({
            'date': day,
            'Ts_min_day_with_buffer_C': Ts_day,
            'soc_start_mwh': soc_start_each_day_mwh,
            'soc_end_mwh': soc_end,
            'feasible': ok,
            'unmet_mwh': unmet
        })

    return pd.DataFrame(results).set_index('date')


daily_Ts6h = compute_daily_min_Ts_with_buffer_daily_reset(
    data_hourly=data_hourly,
    maximum_heat_supply_mw=maximum_heat_supply,
    buffer_volume_m3=buffer_volume_m3,
    buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
    m_dot_max=m_dot_max,
    cp=cp,
    tol_C=1e-4,
    soc_start_each_day_mwh=0.0
)

daily_Ts6h['Ts_min_day_with_buffer_C_80'] = daily_Ts6h['Ts_min_day_with_buffer_C'].apply(lambda x: max(x, 80))

plt.figure(figsize=(20,6))
plt.plot(daily_Ts6h.index, daily_Ts6h['Ts_min_day_with_buffer_C_80'], color='purple', label='Daily minimum supply temperature with buffer (6h charge window)')
plt.plot(daily_Ts.index, daily_Ts['Ts_min_day_with_buffer_C_80'], color='orange', label='Daily minimum supply temperature with buffer (24h charge window)')
plt.plot(AVR_Ts.index, AVR_Ts.values, label='Real supply temperatures 2024')
plt.axhline(y=max_temp, color='red', linestyle='--', label='Maximum supply temperature (120°C)')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend(loc='upper right')
plt.grid(True)
plot_title='Daily supply temperatures 6h charge window vs 24h charge window'
save_current_plot_from_title(save_title=plot_title)
plt.show()

TEMPERATURE CALCULATION WITH CARRYOVER

In [ ]:
def simulate_day_soc_feasibility(
    day_df,
    maximum_heat_supply_mw,
    Ts_day,
    soc_start_mwh,
    buffer_volume_m3,
    buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s,
    m_dot_max,
    cp=cp,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw',
    dt_hours=1.0,
    charge_efficiency=charge_efficiency,
    discharge_efficiency=discharge_efficiency
):
    idx = day_df.index
    soc = pd.Series(index=idx, dtype=float)
    soc.iloc[0] = soc_start_mwh

    unmet_mwh = 0.0
    
    heat_loss_mw = float(lookup_heat_loss_year(Ts_day))

    for i in range(1, len(idx)):
        prev = soc.iloc[i-1]

        Tret = float(day_df[T_return_col].iloc[i])
        demand_mw = float(day_df[demand_col].iloc[i])

        cap_mw = (m_dot_max * cp * max(Ts_day - Tret, 0.0)) / 1e6
        plant_mw = cap_mw

        cap_buffer_mwh = buffer_capacity_mwh_from_volume(
            buffer_volume_m3=buffer_volume_m3,
            T_hot_C=Ts_day,
            T_cold_C=Tret
        )

        charge_limit_mwh = buffer_charge_limit_mwh(
            mdot_charge_max_kg_s=buffer_charge_mdot_max_kg_s,
            T_charge_C=Ts_day,
            T_tank_cold_C=Tret,
            dt_hours=dt_hours,
            cp=cp
        )

        discharge_limit_mwh = buffer_discharge_limit_mwh(
            mdot_discharge_max_kg_s=buffer_discharge_mdot_max_kg_s,
            T_tank_hot_C=Ts_day,
            T_return_C=Tret,
            dt_hours=dt_hours,
            cp=cp
        )

        net_mw = plant_mw - demand_mw - heat_loss_mw
        new_soc = prev

        # CHANGED: only allow charging in the first 6 hours of the day
        can_charge_now = idx[i].hour < 6

        if net_mw >= 0:
            surplus_mwh = net_mw * dt_hours

            # CHANGED: after hour 5, surplus can no longer be stored
            if can_charge_now:
                ch = min(surplus_mwh, charge_limit_mwh, max(0.0, cap_buffer_mwh - prev))
                new_soc = prev + ch * charge_efficiency
            else:
                ch = 0.0
                new_soc = prev

        else:
            deficit_mwh = (-net_mw) * dt_hours
            dis = min(deficit_mwh, discharge_limit_mwh, prev * discharge_efficiency)
            new_soc = prev - dis / max(discharge_efficiency, 1e-12)
            unmet_mwh += (deficit_mwh - dis)

        soc.iloc[i] = max(0.0, min(cap_buffer_mwh, new_soc))

    feasible = (unmet_mwh <= 1e-9)
    soc_end = float(soc.iloc[-1])

    return feasible, soc_end, unmet_mwh


def find_minimum_daily_Ts(
    day_df,
    maximum_heat_supply_mw,
    soc_start_mwh,
    buffer_volume_m3,
    buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s,
    m_dot_max,
    cp=4180.0,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw',
    dt_hours=1.0,
    Ts_hi=max_temp,
    Ts_lo=80,
    tol_C=1e-4,
    max_iter=100
):
    Tret = day_df[T_return_col]
    default_lo = float(Tret.max())
    Ts_lo = default_lo if Ts_lo is None else Ts_lo

    if Ts_hi is None:
        if 'Ts_min_no_buffer' in day_df.columns:
            Ts_hi = float(day_df['Ts_min_no_buffer'].max())
        else:
            Ts_hi = 160

    feasible_hi, soc_end_hi, unmet_hi = simulate_day_soc_feasibility(
        day_df=day_df,
        maximum_heat_supply_mw=maximum_heat_supply_mw,
        Ts_day=Ts_hi,
        soc_start_mwh=soc_start_mwh,
        buffer_volume_m3=buffer_volume_m3,
        buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
        buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
        m_dot_max=m_dot_max,
        cp=cp,
        T_return_col=T_return_col,
        demand_col=demand_col,
        dt_hours=dt_hours,
        charge_efficiency=charge_efficiency,
        discharge_efficiency=discharge_efficiency,
    )
    if not feasible_hi:
        return Ts_hi, soc_end_hi, False, unmet_hi

    lo, hi = Ts_lo, Ts_hi
    soc_end_best = soc_end_hi

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        feasible, soc_end, unmet_hi = simulate_day_soc_feasibility(
            day_df=day_df,
            maximum_heat_supply_mw=maximum_heat_supply_mw,
            Ts_day=mid,
            soc_start_mwh=soc_start_mwh,
            buffer_volume_m3=buffer_volume_m3,
            buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
            buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
            m_dot_max=m_dot_max,
            cp=cp,
            T_return_col=T_return_col,
            demand_col=demand_col,
            dt_hours=dt_hours,
            charge_efficiency=charge_efficiency,
            discharge_efficiency=discharge_efficiency,
        )
        if feasible:
            hi = mid
            soc_end_best = soc_end
        else:
            lo = mid

        if (hi - lo) <= tol_C:
            break

    return hi, soc_end_best, True, 0.0


def compute_daily_min_Ts_with_buffer_carryover(
    data_hourly,
    maximum_heat_supply_mw,
    buffer_volume_m3,
    buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s,
    m_dot_max,
    cp=4180.0,
    dt_hours=1.0,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw',
    tol_C=1e-4,
    soc_start_first_day_mwh=0.0
):
    maximum_heat_supply_mw = maximum_heat_supply_mw.reindex(data_hourly.index)
    daily_groups = data_hourly.groupby(data_hourly.index.floor('D'))
    results = []
    soc_start_today_mwh = soc_start_first_day_mwh
    for day, day_df in daily_groups:
        day_idx = day_df.index
        max_supply_day = maximum_heat_supply_mw.loc[day_idx]

        Ts_day, soc_end, ok, unmet = find_minimum_daily_Ts(
            day_df=day_df,
            maximum_heat_supply_mw=max_supply_day,
            soc_start_mwh=soc_start_today_mwh,
            buffer_volume_m3=buffer_volume_m3,
            buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
            buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
            m_dot_max=m_dot_max,
            cp=cp,
            T_return_col=T_return_col,
            demand_col=demand_col,
            dt_hours=dt_hours,
            tol_C=tol_C
        )

        results.append({
            'date': day,
            'Ts_min_day_with_buffer_C': Ts_day,
            'soc_start_mwh': soc_start_today_mwh,
            'soc_end_mwh': soc_end,
            'feasible': ok,
            'unmet_mwh': unmet
        })
        soc_start_today_mwh = soc_end

    return pd.DataFrame(results).set_index('date')


daily_Ts6h_carryover = compute_daily_min_Ts_with_buffer_carryover(
    data_hourly=data_hourly,
    maximum_heat_supply_mw=maximum_heat_supply,
    buffer_volume_m3=buffer_volume_m3,
    buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
    buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
    m_dot_max=m_dot_max,
    cp=cp,
    tol_C=1e-4,
    soc_start_first_day_mwh=0.0
)

daily_Ts6h_carryover['Ts_min_day_with_buffer_C_80'] = daily_Ts6h_carryover['Ts_min_day_with_buffer_C'].apply(lambda x: max(x, 80))

plt.figure(figsize=(20,6))
plt.plot(daily_Ts6h_carryover.index, daily_Ts6h_carryover['Ts_min_day_with_buffer_C_80'], color='purple', label='Daily minimum supply temperature with buffer (carryover)')
plt.plot(daily_Ts.index, daily_Ts['Ts_min_day_with_buffer_C_80'], color='orange', label='Daily minimum supply temperature with buffer (24h charge window)')
plt.plot(AVR_Ts.index, AVR_Ts.values, label='Real supply temperatures 2024')
plt.axhline(y=max_temp, color='red', linestyle='--', label='Maximum supply temperature (120°C)')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend(loc='upper right')
plt.grid(True)
plot_title='Daily supply temperatures 6h charge window (carryover) vs 24h charge window'
save_current_plot_from_title(save_title=plot_title)
plt.show()

In [ ]:
#cap base scenario at 120C  

plt.figure(figsize=(20,6))
plt.plot(AVR_Ts.index, AVR_Ts.values, label='Real supply temperatures 2024')
plt.plot(maximum_daily_temp_with_cap.index, maximum_daily_temp_with_cap.values, color='red', label='Daily minimum supply temperature, base') 
plt.plot(daily_Ts.index, daily_Ts['Ts_min_day_with_buffer_C_80'], color='orange', label='Daily minimum Ts, perfect')
plt.plot(daily_Ts6h.index, daily_Ts6h['Ts_min_day_with_buffer_C_80'], color='green', label='Daily minimum Ts, 6H')
plt.plot(daily_Ts6h_carryover.index, daily_Ts6h_carryover['Ts_min_day_with_buffer_C_80'], color='purple', label='Daily minimum Ts, 6H carryover')
plt.axhline(y=max_temp, color='black', linestyle='--', label='Maximum supply temperature (120°C)')
plt.xlabel('Date',fontsize=14)
plt.ylabel('Temperature (°C)',fontsize=14)
plt.legend(loc='upper center', fontsize=14)
plt.grid(True)
plot_title='Daily supply temperatures of the scenarios vs real supply temperatures'
save_current_plot_from_title(save_title=plot_title)
plt.show()

In [ ]:
#based on the temperautres found per day, calculate the mass flow per hour to fulfill demand
daily_Ts_check = daily_Ts['Ts_min_day_with_buffer_C_80'].reindex(
    data_hourly.index,
    method='ffill'
)
def calculate_mass_flow_for_Ts_day(
    day_df,
    Ts_day,
    cp=cp,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw'
):
    Tret = day_df[T_return_col]
    demand_mw = day_df[demand_col]
    heat_loss_mw = pd.Series(lookup_heat_loss_year(Ts_day), index=day_df.index)
    total_network_load_mw = demand_mw + heat_loss_mw

    # Calculate mass flow rate (kg/s) needed to meet useful demand plus scenario-specific heat loss at Ts_day
    m_dot_needed = (total_network_load_mw * 1e6) / (cp * (Ts_day - Tret))

    return m_dot_needed

m_dot_needed = calculate_mass_flow_for_Ts_day(
    day_df=data_hourly,
    Ts_day=daily_Ts_check,
    cp=cp,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw'
)
plt.figure(figsize=(20,6))
plt.plot(m_dot_needed.index, m_dot_needed.values, label='Mass Flow Needed (kg/s)', color='brown')
plt.title('Mass Flow Needed to Meet Demand at Daily Minimum Ts (from AVR + GAS)')
plt.xlabel('Date')
plt.ylabel('Mass Flow (kg/s)')
plt.grid(True)
plt.legend()

In [ ]:
#for each day, calculate the amound of shortage in energy if we can only supply at Ts_day with max mass flow m_dot_max, and the rest has to be unmet demand (shortage)
def calculate_shortage_for_Ts_day(
    day_df,
    Ts_day,
    m_dot_max,
    cp=cp,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw'
):
    Tret = day_df[T_return_col]
    demand_mw = day_df[demand_col]
    heat_loss_mw = pd.Series(lookup_heat_loss_year(Ts_day), index=day_df.index)
    total_network_load_mw = demand_mw + heat_loss_mw

    # Calculate mass flow rate (kg/s) needed to meet useful demand plus scenario-specific heat loss at Ts_day
    m_dot_needed = (total_network_load_mw * 1e6) / (cp * (Ts_day - Tret))

    # Calculate shortage in energy (MW) for each hour
    shortage_mw = total_network_load_mw - ((m_dot_max * cp * (Ts_day - Tret)) / 1e6)

    # Ensure shortage is not negative
    shortage_mw = shortage_mw.clip(lower=0)

    return shortage_mw

shortage_mw = calculate_shortage_for_Ts_day(
    day_df=data_hourly,
    Ts_day=daily_Ts_check,
    m_dot_max=m_dot_max,
    cp=cp,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw'
)
plt.figure(figsize=(20,6))
plt.plot(shortage_mw.index, shortage_mw.values, label='Shortage at Daily Min Ts (MW)', color='red')
plt.title('Shortage in Heat Supply at Daily Minimum Ts')
plt.xlabel('Date')
plt.ylabel('Shortage (MW)')
plt.grid(True)
save_current_plot_from_title()
plt.legend()

#calculate shortage per day in mwh
shortage_mwh = shortage_mw 
daily_shortage_mwh = shortage_mwh.resample('D').sum()
plt.figure(figsize=(20,6))
plt.plot(daily_shortage_mwh.index, daily_shortage_mwh.values, label='Daily Shortage at Min Ts (MWh)', color='red')
plt.title('Daily Shortage in Heat Supply at Daily Minimum Ts')
plt.xlabel('Date')
plt.ylabel('Shortage (MWh)')
plt.grid(True)
save_current_plot_from_title()
plt.legend()

PERFECT REGIME

In [ ]:
def simulate_day_perfect_foresight_charge(
    day_df: pd.DataFrame,
    Ts_day: float,
    m_dot_max: float,
    source_Ts_max: float = max_temp,
    source_mw_max: float = None,
    buffer_volume_m3: float = 0.0,
    buffer_charge_mdot_max_kg_s: float = 0.0,
    buffer_discharge_mdot_max_kg_s: float = 0.0,
    soc_start_mwh: float = 0.0,
    cp: float = 4180.0,
    demand_col: str = 'useful_demand_mw',
    T_return_col: str = 'AVR_ARN_TR_monthly',
    dt_hours: float = 1.0,
    charge_eff=charge_efficiency,
    discharge_eff=discharge_efficiency,
):
    idx = day_df.index
    demand = day_df[demand_col].astype(float)
    Tret = day_df[T_return_col].astype(float)
    infeas_reasons = []

    Ts_eff = min(float(Ts_day), float(source_Ts_max))
    heat_loss_mw = float(lookup_heat_loss_year(Ts_eff))
    heat_loss = pd.Series(heat_loss_mw, index=idx, dtype=float)
    total_network_load = demand + heat_loss
    dT = (Ts_eff - Tret).clip(lower=1e-6)
    charge_mdot_kg_s = pd.Series(0.0, index=idx, dtype=float)
    discharge_mdot_kg_s = pd.Series(0.0, index=idx, dtype=float)
    q_cap_mw = (m_dot_max * cp * dT) / 1e6
    if source_mw_max is not None:
        q_cap_mw = q_cap_mw.clip(upper=float(source_mw_max))

    shortage_mw = (total_network_load - q_cap_mw).clip(lower=0.0)
    headroom_mw = (q_cap_mw - total_network_load).clip(lower=0.0)

    cap_buffer_mwh_hourly = pd.Series(
        [buffer_capacity_mwh_from_volume(buffer_volume_m3, Ts_eff, tr) for tr in Tret],
        index=idx
    )
    charge_limit_mwh_hourly = pd.Series(
        [buffer_charge_limit_mwh(buffer_charge_mdot_max_kg_s, Ts_eff, tr, dt_hours, cp) for tr in Tret],
        index=idx
    )
    discharge_limit_mwh_hourly = pd.Series(
        [buffer_discharge_limit_mwh(buffer_discharge_mdot_max_kg_s, Ts_eff, tr, dt_hours, cp) for tr in Tret],
        index=idx
    )

    if (cap_buffer_mwh_hourly.max() <= 0 or discharge_limit_mwh_hourly.max() <= 0) and (shortage_mw > 1e-9).any():
        infeas_reasons.append("Shortage occurs but buffer is disabled (capacity/power <= 0).")
        out = pd.DataFrame(index=idx)
        return out, False, infeas_reasons

    required_soc = pd.Series(0.0, index=idx)
    req_next = 0.0

    for i in range(len(idx) - 1, -1, -1):
        need_mw = float(shortage_mw.iloc[i])
        discharge_limit_mw_i = float(discharge_limit_mwh_hourly.iloc[i] / dt_hours)

        if need_mw > discharge_limit_mw_i + 1e-9:
            infeas_reasons.append(
                f"Hour {idx[i]} shortage {need_mw:.3f} MW exceeds buffer discharge power {discharge_limit_mw_i:.3f} MW."
            )

        need_mwh = need_mw * dt_hours
        soc_needed_before = req_next + (need_mwh / max(discharge_eff, 1e-12))

        if soc_needed_before > float(cap_buffer_mwh_hourly.iloc[i]) + 1e-9:
            infeas_reasons.append(
                f"Hour {idx[i]} requires SOC {soc_needed_before:.3f} MWh > capacity {float(cap_buffer_mwh_hourly.iloc[i]):.3f} MWh."
            )

        required_soc.iloc[i] = soc_needed_before
        req_next = soc_needed_before

    soc = pd.Series(np.nan, index=idx)
    soc.iloc[0] = float(min(soc_start_mwh, float(cap_buffer_mwh_hourly.iloc[0])))

    q_source_mw = pd.Series(0.0, index=idx)
    q_charge_mw = pd.Series(0.0, index=idx)
    q_discharge_mw = pd.Series(0.0, index=idx)
    unmet_mw = pd.Series(0.0, index=idx)
    m_dot_used = pd.Series(0.0, index=idx)

    for i in range(len(idx)):
        prev_soc = soc.iloc[i-1] if i > 0 else soc.iloc[0]
        cap_i = float(cap_buffer_mwh_hourly.iloc[i])

        need_mw = float(shortage_mw.iloc[i])
        if need_mw > 0:
            q_source_mw.iloc[i] = float(q_cap_mw.iloc[i])

            discharge_limit_mw_i = float(discharge_limit_mwh_hourly.iloc[i] / dt_hours)
            discharge_mw = min(need_mw, discharge_limit_mw_i)
            discharge_mwh_delivered = discharge_mw * dt_hours

            soc_drop = discharge_mwh_delivered / max(discharge_eff, 1e-12)
            soc_after_discharge = prev_soc - soc_drop

            if soc_after_discharge < -1e-9:
                infeas_reasons.append(f"Hour {idx[i]} SOC would go negative. Not enough stored energy.")
                max_deliver_mwh = max(0.0, min(prev_soc * discharge_eff, float(discharge_limit_mwh_hourly.iloc[i])))
                discharge_mw = max_deliver_mwh / dt_hours
                discharge_mwh_delivered = max_deliver_mwh
                soc_drop = discharge_mwh_delivered / max(discharge_eff, 1e-12)
                soc_after_discharge = prev_soc - soc_drop

            q_discharge_mw.iloc[i] = discharge_mw
            discharge_mdot_kg_s.iloc[i] = discharge_mw * 1e6 / (cp * float(dT.iloc[i]) * dt_hours)
            charge_mdot_kg_s.iloc[i] = 0.0
            unmet_mw.iloc[i] = max(0.0, need_mw - discharge_mw)
            soc_end = soc_after_discharge

        else:
            target_soc_next = float(required_soc.iloc[i+1]) if i < len(idx) - 1 else 0.0
            soc_start = prev_soc
            required_delta_soc = max(0.0, target_soc_next - soc_start)

            effective_headroom_mw = max(0.0, float(headroom_mw.iloc[i]))
            max_input_mwh = min(
                effective_headroom_mw * dt_hours,
                float(charge_limit_mwh_hourly.iloc[i])
            )
            max_soc_increase = max_input_mwh * charge_eff
            max_soc_increase = min(max_soc_increase, max(0.0, cap_i - soc_start))

            delta_soc = min(required_delta_soc, max(0.0, max_soc_increase))

            if required_delta_soc > delta_soc + 1e-9:
                infeas_reasons.append(
                    f"Hour {idx[i]} cannot charge enough. Needed ΔSOC={required_delta_soc:.3f} MWh, "
                    f"max possible={delta_soc:.3f} MWh."
                )

            charge_input_mwh = delta_soc / max(charge_eff, 1e-12)
            charge_mw = charge_input_mwh / dt_hours
            charge_mdot_kg_s.iloc[i] = charge_mw * 1e6 / (cp * float(dT.iloc[i]) * dt_hours)
            discharge_mdot_kg_s.iloc[i] = 0.0
            q_charge_mw.iloc[i] = charge_mw
            q_source_mw.iloc[i] = float(total_network_load.iloc[i]) + charge_mw
            soc_end = soc_start + delta_soc

        soc_end = min(max(soc_end, 0.0), cap_i)
        soc.iloc[i] = soc_end

        m_dot_used.iloc[i] = (q_source_mw.iloc[i] * 1e6) / (cp * float(dT.iloc[i]))
        m_dot_used.iloc[i] = min(m_dot_used.iloc[i], m_dot_max)

    feasible = (len(infeas_reasons) == 0) and (unmet_mw.max() <= 1e-9)

    out = pd.DataFrame({
        "demand_mw": demand,
        "heat_loss_mw": heat_loss,
        "total_network_load_mw": total_network_load,
        "Tret_C": Tret,
        "Ts_day_C": Ts_eff,
        "dT_K": dT,
        "q_cap_mw": q_cap_mw,
        "headroom_mw": headroom_mw,
        "shortage_mw": shortage_mw,
        "buffer_capacity_mwh_hourly": cap_buffer_mwh_hourly,
        "charge_limit_mwh_hourly": charge_limit_mwh_hourly,
        "discharge_limit_mwh_hourly": discharge_limit_mwh_hourly,
        "q_source_mw": q_source_mw,
        "q_charge_mw": q_charge_mw,
        "q_discharge_mw": q_discharge_mw,
        "soc_mwh": soc,
        "required_soc_mwh": required_soc,
        "unmet_mw": unmet_mw,
        "m_dot_used_kg_s": m_dot_used,
        "charge_mdot_kg_s": charge_mdot_kg_s,
        "discharge_mdot_kg_s": discharge_mdot_kg_s,
    }, index=idx)

    return out, feasible, infeas_reasons

results = []
soc = 0.0

Ts_hourly = daily_Ts['Ts_min_day_with_buffer_C'].reindex(data_hourly.index, method='ffill')

for day, day_df in data_hourly.groupby(pd.Grouper(freq='D')):
    if len(day_df) == 0:
        continue

    Ts_day = float(Ts_hourly.loc[day_df.index[0]])

    day_res, feasible, reasons = simulate_day_perfect_foresight_charge(
        day_df=day_df,
        Ts_day=Ts_day,
        m_dot_max=m_dot_max,
        source_Ts_max=max_temp,
        source_mw_max=None,
        buffer_volume_m3=buffer_volume_m3,
        buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
        buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
        soc_start_mwh=soc,
        cp=cp,
        demand_col='useful_demand_mw',
        T_return_col='AVR_ARN_TR_monthly',
        dt_hours=1.0,
        charge_eff=charge_efficiency,
        discharge_eff=discharge_efficiency,
    )

    soc = float(day_res["soc_mwh"].iloc[-1]) if len(day_res) else soc
    day_res["day_feasible"] = feasible
    day_res["day"] = day
    results.append(day_res)

    if not feasible:
        print(f"\nDay {day.date()} infeasible:")
        for r in reasons[:10]:
            print(" -", r)
        if len(reasons) > 10:
            print(f" - ... ({len(reasons)-10} more)")

year_res = pd.concat(results)

print("\nTotal unmet energy (MWh):", float((year_res["unmet_mw"] * 1.0).sum()))
print("Total unmet hours:", int((year_res["unmet_mw"] > 1e-9).sum()))

In [ ]:


# --- 1) Daily mass flow summaries ---
m_dot_hourly = year_res["m_dot_used_kg_s"]

m_dot_daily_mean = m_dot_hourly.resample("D").mean()
m_dot_daily_max  = m_dot_hourly.resample("D").max()
m_dot_daily_p95  = m_dot_hourly.resample("D").quantile(0.95)

plt.figure(figsize=(20,6))
plt.plot(m_dot_hourly.index, m_dot_hourly.values, label="Hourly m_dot (kg/s)", alpha=0.5)
plt.title("Hourly mass flow from AVR under perfect foresight charging strategy")
plt.xlabel("Date")
plt.ylabel("Mass flow (kg/s)")
plt.grid(True)
save_current_plot_from_title()
plt.show()

plt.figure(figsize=(20,6))
plt.plot(m_dot_daily_mean.index, m_dot_daily_mean.values, label="Daily mean m_dot (kg/s)")
plt.plot(m_dot_daily_max.index,  m_dot_daily_max.values,  label="Daily max m_dot (kg/s)")
plt.plot(m_dot_daily_p95.index,  m_dot_daily_p95.values,  label="Daily p95 m_dot (kg/s)")
plt.title("Daily mass flow from AVR under perfect foresight charging strategy summary")
plt.xlabel("Date")
plt.ylabel("Mass flow (kg/s)")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()


# --- 2) SOC over the year (hourly) ---
soc_hourly = year_res["soc_mwh"]

plt.figure(figsize=(20,6))
plt.plot(soc_hourly.index, soc_hourly.values, label="SOC (MWh)")
plt.title("Buffer SOC over the year (hourly)")
plt.xlabel("Date")
plt.ylabel("SOC (MWh)")
plt.grid(True)
save_current_plot_from_title()
plt.show()

#print maximum SOC reached and when
max_soc = soc_hourly.max()
max_discharge = year_res["q_discharge_mw"].max()
max_soc_time = soc_hourly.idxmax()
print(f"Maximum SOC reached: {max_soc:.2f} MWh at {max_soc_time}")
print(f"Maximum discharge power: {max_discharge:.2f} MW")



PLOTTING

In [ ]:

# --- Hourly charge / discharge over the year ---
charge = year_res["q_charge_mw"]          # MW into buffer (charging)
discharge = year_res["q_discharge_mw"]    # MW delivered from buffer (discharging)

plt.figure(figsize=(20,6))
plt.plot(charge.index, charge.values, label="Buffer charge (MW)")
plt.plot(discharge.index, discharge.values, label="Buffer discharge (MW)")
plt.title("Hourly buffer charge and discharge over the year")
plt.xlabel("Date")
plt.ylabel("Power (MW)")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()


# --- Unmet demand (hourly) over the year ---
unmet = year_res["unmet_mw"]              # MW not served (would require backup boilers)

plt.figure(figsize=(20,6))
plt.plot(unmet.index, unmet.values, label="Unmet demand (MW)")
plt.title("Hourly unmet demand over the year")
plt.xlabel("Date")
plt.ylabel("Unmet (MW)")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()


# --- Optional: Daily unmet energy (MWh/day) for a clearer view ---
unmet_mwh_day = (unmet * 1.0).resample("D").sum()  # dt_hours=1 so MW*h = MWh

plt.figure(figsize=(20,6))
plt.plot(unmet_mwh_day.index, unmet_mwh_day.values, label="Unmet energy (MWh/day)")
plt.title("Daily unmet demand over the year")
plt.xlabel("Date")
plt.ylabel("Unmet energy (MWh/day)")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()


# --- Quick stats ---
print("Total unmet energy (MWh):", float(unmet.sum()))  # MW * 1h = MWh
print("Hours with unmet demand:", int((unmet > 1e-9).sum()))
print("Peak unmet demand (MW):", float(unmet.max()))

In [ ]:

charge = year_res["q_charge_mw"]                 # positive
discharge_neg = -year_res["q_discharge_mw"]      # make discharge negative

plt.figure(figsize=(20,6))
plt.plot(charge.index, charge.values, label="Buffer charge (MW)")
plt.plot(discharge_neg.index, discharge_neg.values, label="Buffer discharge (MW, negative)")
plt.axhline(0)
plt.title("Hourly buffer charge (+) and discharge (-) over the year, under perfect foresight charging strategy")
plt.xlabel("Date")
plt.ylabel("Power (MW)")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()

#zoom in on first week and plot charge and discharge
plt.figure(figsize=(20,6))
plt.plot(charge.index[:24*7], charge.values[:24*7], label="Buffer charge (MW)")
plt.plot(discharge_neg.index[:24*7], discharge_neg.values[:24*7], label="Buffer discharge (MW, negative)")
plt.title("Hourly buffer charge (+) and discharge (-) - First Week")
plt.xlabel("Date")
plt.ylabel("Power (MW)")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()

start = "2024-01-08"
end = "2024-01-28"

plt.figure(figsize=(20,6))

plt.plot(
    charge.loc[start:end].index,
    charge.loc[start:end].values,
    label="Buffer charge (MW)"
)

plt.plot(
    discharge_neg.loc[start:end].index,
    discharge_neg.loc[start:end].values,
    label="Buffer discharge (MW, negative)"
)

plt.title("Hourly buffer charge (+) and discharge (-) - Weeks 2 to 4 of January")
plt.xlabel("Date")
plt.ylabel("Power (MW)")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()

#plot mass flows for charging and discharging
charge_mdot = year_res["charge_mdot_kg_s"]          # kg/s for charging
discharge_mdot = year_res["discharge_mdot_kg_s"]    # kg/s for discharging
plt.figure(figsize=(20,6))
plt.plot(charge_mdot.index, charge_mdot.values, label="Charge mass flow (kg/s)")
plt.plot(discharge_mdot.index, discharge_mdot.values, label="Discharge mass flow (kg/s)")
plt.title("Hourly mass flow for charging and discharging over the year")
plt.xlabel("Date")
plt.ylabel("Mass flow (kg/s)")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()

In [ ]:
def plot_buffer_charge_discharge_two_zooms_with_temp_demand(
    charge,
    discharge,
    zoom1_start,
    zoom1_end,
    zoom2_start,
    zoom2_end,
    supply_temp=None,
    useful_heat_demand=None,
    title="Hourly buffer charge (+) and discharge (-) over the year",
    zoom1_title=None,
    zoom2_title=None,
    save_title=None,
    discharge_as_negative=True,
    figsize=(16, 9),
    save_plot=True
):
    charge = charge.copy()
    discharge = discharge.copy()

    charge.index = pd.to_datetime(charge.index)
    discharge.index = pd.to_datetime(discharge.index)

    charge = charge.sort_index()
    discharge = discharge.sort_index()

    if discharge_as_negative:
        discharge_plot = -discharge
        discharge_label = "Discharge (MW)"
    else:
        discharge_plot = discharge
        discharge_label = "Discharge (MW)"

    if supply_temp is not None:
        supply_temp = supply_temp.copy()
        supply_temp.index = pd.to_datetime(supply_temp.index)
        supply_temp = supply_temp.sort_index()

    if useful_heat_demand is not None:
        useful_heat_demand = useful_heat_demand.copy()
        useful_heat_demand.index = pd.to_datetime(useful_heat_demand.index)
        useful_heat_demand = useful_heat_demand.sort_index()

    zoom1_start = pd.Timestamp(zoom1_start)
    zoom1_end = pd.Timestamp(zoom1_end)
    zoom2_start = pd.Timestamp(zoom2_start)
    zoom2_end = pd.Timestamp(zoom2_end)

    # =========================
    # Zoom data
    # =========================
    charge_zoom1 = charge.loc[zoom1_start:zoom1_end]
    discharge_zoom1 = discharge_plot.loc[zoom1_start:zoom1_end]

    charge_zoom2 = charge.loc[zoom2_start:zoom2_end]
    discharge_zoom2 = discharge_plot.loc[zoom2_start:zoom2_end]

    if supply_temp is not None:
        temp_zoom1 = supply_temp.loc[zoom1_start:zoom1_end]
        temp_zoom2 = supply_temp.loc[zoom2_start:zoom2_end]
    else:
        temp_zoom1 = None
        temp_zoom2 = None

    if useful_heat_demand is not None:
        demand_zoom1 = useful_heat_demand.loc[zoom1_start:zoom1_end]
        demand_zoom2 = useful_heat_demand.loc[zoom2_start:zoom2_end]
    else:
        demand_zoom1 = None
        demand_zoom2 = None

    # =========================
    # Y-limits
    # =========================
    def get_ylim(*series_list):
        values = []

        for s in series_list:
            if s is not None and not s.empty:
                values.extend([s.min(), s.max()])

        y_min = min(values)
        y_max = max(values)
        pad = 0.05 * (y_max - y_min) if y_max != y_min else 1

        return y_min - pad, y_max + pad

    # Upper plot only uses charge and discharge
    y_full_min, y_full_max = get_ylim(charge, discharge_plot)

    # Rectangles in upper plot must also only use charge/discharge zoom values
    y1_rect_min, y1_rect_max = get_ylim(charge_zoom1, discharge_zoom1)
    y2_rect_min, y2_rect_max = get_ylim(charge_zoom2, discharge_zoom2)

    # Zoom plots may include useful heat demand
    y1_zoom_min, y1_zoom_max = get_ylim(charge_zoom1, discharge_zoom1, demand_zoom1)
    y2_zoom_min, y2_zoom_max = get_ylim(charge_zoom2, discharge_zoom2, demand_zoom2)

    # =========================
    # Figure layout
    # =========================
    fig = plt.figure(figsize=figsize)

    ax_full = fig.add_axes([0.07, 0.58, 0.88, 0.34])
    ax_zoom1 = fig.add_axes([0.09, 0.20, 0.38, 0.28])
    ax_zoom2 = fig.add_axes([0.57, 0.20, 0.38, 0.28])

    # =========================
    # Full-year plot
    # =========================
    line_charge_full, = ax_full.plot(
        charge.index,
        charge,
        label="Charge (MW)",
        linewidth=1
    )

    line_discharge_full, = ax_full.plot(
        discharge_plot.index,
        discharge_plot,
        label=discharge_label,
        linewidth=1
    )

    #ax_full.set_title(title)
    ax_full.set_ylabel("Power (MW)",fontsize=12)
    ax_full.set_ylim(y_full_min, y_full_max)
    ax_full.grid(True, alpha=0.3)

    ax_full.xaxis.set_major_locator(mdates.MonthLocator())
    ax_full.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

    # Highlight rectangle 1, based only on charge/discharge values
    rect1 = Rectangle(
        (mdates.date2num(zoom1_start), y1_rect_min),
        mdates.date2num(zoom1_end) - mdates.date2num(zoom1_start),
        y1_rect_max - y1_rect_min,
        fill=False,
        edgecolor="black",
        linewidth=2
    )
    ax_full.add_patch(rect1)

    # Highlight rectangle 2, based only on charge/discharge values
    rect2 = Rectangle(
        (mdates.date2num(zoom2_start), y2_rect_min),
        mdates.date2num(zoom2_end) - mdates.date2num(zoom2_start),
        y2_rect_max - y2_rect_min,
        fill=False,
        edgecolor="black",
        linewidth=2
    )
    ax_full.add_patch(rect2)

    # =========================
    # Helper for zoom plots
    # =========================
    legend_handles = [line_charge_full, line_discharge_full]
    legend_labels = ["Charge (MW)", discharge_label]

    def plot_zoom_axis(
        ax,
        charge_zoom,
        discharge_zoom,
        demand_zoom,
        temp_zoom,
        zoom_start,
        zoom_end,
        zoom_title,
        y_min,
        y_max,
        add_to_legend=False
    ):
        ax.plot(
            charge_zoom.index,
            charge_zoom,
            linewidth=1.5,
            label="Charge (MW)"
        )

        ax.plot(
            discharge_zoom.index,
            discharge_zoom,
            linewidth=1.5,
            label=discharge_label
        )

        demand_line = None

        if demand_zoom is not None and not demand_zoom.empty:
            demand_line, = ax.plot(
                demand_zoom.index,
                demand_zoom,
                linewidth=1.2,
                linestyle="--",
                label="Useful heat demand (MW)"
            )

        ax.set_title(zoom_title, fontsize=12)
        ax.set_ylabel("Power (MW)",fontsize=12)
        ax.set_xlabel("Date",fontsize=12)
        ax.set_xlim(zoom_start, zoom_end)
        ax.set_ylim(y_min, y_max)
        ax.grid(True, alpha=0.3)

        ax.xaxis.set_major_locator(mdates.DayLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))

        for spine in ax.spines.values():
            spine.set_edgecolor("black")
            spine.set_linewidth(2)

        temp_line = None
        ax_temp = None

        if temp_zoom is not None and not temp_zoom.empty:
            ax_temp = ax.twinx()

            temp_line, = ax_temp.plot(
                temp_zoom.index,
                temp_zoom,
                linewidth=1.5,
                linestyle=":",
                label=r"Supply temperature ($^\circ$C)"
            )

            ax_temp.set_ylabel(r"Supply temperature ($^\circ$C)", fontsize=12)
            ax_temp.set_xlim(zoom_start, zoom_end)

            temp_min = temp_zoom.min()
            temp_max = temp_zoom.max()
            temp_pad = 0.05 * (temp_max - temp_min) if temp_max != temp_min else 1
            ax_temp.set_ylim(temp_min - temp_pad, temp_max + temp_pad)

        # Add demand and temperature only once to the main legend
        if add_to_legend:
            if demand_line is not None:
                legend_handles.append(demand_line)
                legend_labels.append("Heat demand (MW)")

            if temp_line is not None:
                legend_handles.append(temp_line)
                legend_labels.append(r"Supply temperature ($^\circ$C)")

        return ax_temp

    # =========================
    # Zoom plot 1
    # =========================
    if zoom1_title is None:
        zoom1_title = f"Zoom 1: {zoom1_start.strftime('%d %b')} – {zoom1_end.strftime('%d %b')}"

    ax_zoom1_temp = plot_zoom_axis(
        ax=ax_zoom1,
        charge_zoom=charge_zoom1,
        discharge_zoom=discharge_zoom1,
        demand_zoom=demand_zoom1,
        temp_zoom=temp_zoom1,
        zoom_start=zoom1_start,
        zoom_end=zoom1_end,
        zoom_title=zoom1_title,
        y_min=y1_zoom_min,
        y_max=y1_zoom_max,
        add_to_legend=True
    )

    # =========================
    # Zoom plot 2
    # =========================
    if zoom2_title is None:
        zoom2_title = f"Zoom 2: {zoom2_start.strftime('%d %b')} – {zoom2_end.strftime('%d %b')}"

    ax_zoom2_temp = plot_zoom_axis(
        ax=ax_zoom2,
        charge_zoom=charge_zoom2,
        discharge_zoom=discharge_zoom2,
        demand_zoom=demand_zoom2,
        temp_zoom=temp_zoom2,
        zoom_start=zoom2_start,
        zoom_end=zoom2_end,
        zoom_title=zoom2_title,
        y_min=y2_zoom_min,
        y_max=y2_zoom_max,
        add_to_legend=False
    )

    # =========================
    # One combined legend in upper plot
    # =========================
    ax_full.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        fontsize=12
    )

    # =========================
    # Connection lines
    # =========================
    y_top_zoom1 = ax_zoom1.get_ylim()[1]

    con1a = ConnectionPatch(
        xyA=(mdates.date2num(zoom1_start), y1_rect_min),
        coordsA=ax_full.transData,
        xyB=(mdates.date2num(zoom1_start), y_top_zoom1),
        coordsB=ax_zoom1.transData,
        color="black",
        linewidth=1.5
    )

    con1b = ConnectionPatch(
        xyA=(mdates.date2num(zoom1_end), y1_rect_min),
        coordsA=ax_full.transData,
        xyB=(mdates.date2num(zoom1_end), y_top_zoom1),
        coordsB=ax_zoom1.transData,
        color="black",
        linewidth=1.5
    )

    y_top_zoom2 = ax_zoom2.get_ylim()[1]

    con2a = ConnectionPatch(
        xyA=(mdates.date2num(zoom2_start), y2_rect_min),
        coordsA=ax_full.transData,
        xyB=(mdates.date2num(zoom2_start), y_top_zoom2),
        coordsB=ax_zoom2.transData,
        color="black",
        linewidth=1.5
    )

    con2b = ConnectionPatch(
        xyA=(mdates.date2num(zoom2_end), y2_rect_min),
        coordsA=ax_full.transData,
        xyB=(mdates.date2num(zoom2_end), y_top_zoom2),
        coordsB=ax_zoom2.transData,
        color="black",
        linewidth=1.5
    )

    fig.add_artist(con1a)
    fig.add_artist(con1b)
    fig.add_artist(con2a)
    fig.add_artist(con2b)

    # =========================
    # Save plot
    # =========================
    if save_plot:
        if save_title is None:
            save_title = title

        try:
            save_current_plot_from_title(save_title)
        except TypeError:
            plt.sca(ax_full)
            save_current_plot_from_title()

    plt.show()

    return fig, ax_full, ax_zoom1, ax_zoom2, ax_zoom1_temp, ax_zoom2_temp

In [ ]:
plot_buffer_charge_discharge_two_zooms_with_temp_demand(
    charge=year_res["q_charge_mw"],
    discharge=year_res["q_discharge_mw"],
    supply_temp=year_res["Ts_day_C"],
    useful_heat_demand=data_hourly["useful_demand_mw"],
    zoom1_start="2024-01-08 00:00:00",
    zoom1_end="2024-01-18 23:00:00",
    zoom2_start="2024-03-01 00:00:00",
    zoom2_end="2024-03-11 23:00:00",
    title="Hourly buffer charge (+) and discharge (-) over the year, under perfect strategy",
    zoom1_title="Zoom: 8 - 18 January",
    zoom2_title="Zoom: 1 - 11 March",
    save_title="Buffer_charge_discharge_temperature_demand_perfect_BFCS_1"
)

In [ ]:
#at ts_daily, calculate the shortage_mw_hourly.
Ts_hourly_min_80 = daily_Ts['Ts_min_day_with_buffer_C_80'].reindex(data_hourly.index, method='ffill')
print(Ts_hourly_min_80.head())
shortage_mw_hourly = calculate_shortage_for_Ts_day(
    day_df=data_hourly,
    Ts_day=Ts_hourly_min_80,
    m_dot_max=m_dot_max,
    cp=cp,
    T_return_col='AVR_ARN_TR_monthly',
    demand_col='useful_demand_mw'
)
plt.figure(figsize=(20,6))
plt.plot(shortage_mw_hourly.index, shortage_mw_hourly.values, label='Shortage at Daily Min Ts (MW)', color='red')
plt.title('Shortage in Heat Supply at Daily Minimum Ts (Hourly)')
plt.xlabel('Date')
plt.ylabel('Shortage (MW)')
plt.grid(True)
save_current_plot_from_title()
plt.legend()
plt.show()

#calculate shortage per day in mwh
shortage_mwh_hourly = shortage_mw_hourly * 1.0  # MW * 1h = MWh
daily_shortage_mwh_hourly = shortage_mwh_hourly.resample('D').sum()

#calculate the total available heat to charge buffer per day in the first 6 hours of the day, to see if it can cover the shortage in the later hours
heat_loss_at_Ts_hourly_min_80_mw = pd.Series(lookup_heat_loss_year(Ts_hourly_min_80), index=data_hourly.index)
network_load_at_Ts_hourly_min_80_mw = data_hourly['useful_demand_mw'] + heat_loss_at_Ts_hourly_min_80_mw
heat_available_for_charging_mw = (m_dot_max * cp * (Ts_hourly_min_80 - data_hourly['AVR_ARN_TR_monthly'])) / 1e6 - network_load_at_Ts_hourly_min_80_mw
heat_available_for_charging_mw = heat_available_for_charging_mw.clip(lower=0)
heat_available_for_charging_mwh = heat_available_for_charging_mw * 1.0  # MW * 1h = MWh
heat_available_for_charging_mwh_6h = heat_available_for_charging_mwh.groupby(heat_available_for_charging_mwh.index.floor('D')).apply(lambda x: x.iloc[:6].sum())
plt.figure(figsize=(20,6))
plt.plot(daily_shortage_mwh_hourly.index, daily_shortage_mwh_hourly.values, label='Daily Shortage at Min Ts (MWh)', color='red')
plt.plot(heat_available_for_charging_mwh_6h.index, heat_available_for_charging_mwh_6h.values, label='Heat Available for Charging in First 6 Hours (MWh)', color='blue')
plt.title('Daily Shortage at Min Ts vs Heat Available for Charging in First 6 Hours')
plt.xlabel('Date')
plt.ylabel('Energy (MWh)')
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()

6H REGIME


In [ ]:
def simulate_day_charge_first6h_empty_start(
    day_df: pd.DataFrame,
    Ts_day: float,
    m_dot_max: float,
    cp: float,
    buffer_volume_m3: float,
    buffer_charge_mdot_max_kg_s: float,
    buffer_discharge_mdot_max_kg_s: float,
    charge_hours: int = 6,
    charge_eff=charge_efficiency,
    discharge_eff=discharge_efficiency,
    demand_col: str = "useful_demand_mw",
    T_return_col: str = "AVR_ARN_TR_monthly",
    dt_hours: float = 1.0,
):
    idx = day_df.index
    demand = day_df[demand_col].astype(float)
    Tret = day_df[T_return_col].astype(float)
    charge_mdot_kg_s = pd.Series(0.0, index=idx, dtype=float)
    discharge_mdot_kg_s = pd.Series(0.0, index=idx, dtype=float)
    dT = (Ts_day - Tret).clip(lower=1e-6)
    q_cap_mw = (m_dot_max * cp * dT) / 1e6

    heat_loss_mw = float(lookup_heat_loss_year(Ts_day))
    heat_loss = pd.Series(heat_loss_mw, index=idx, dtype=float)
    total_network_load = demand + heat_loss
    shortage_mw = (total_network_load - q_cap_mw).clip(lower=0.0)
    required_discharge_mwh = float((shortage_mw * dt_hours).sum())
    headroom_mw = (q_cap_mw - total_network_load).clip(lower=0.0)

    cap_buffer_mwh_hourly = pd.Series(
        [buffer_capacity_mwh_from_volume(buffer_volume_m3, Ts_day, tr) for tr in Tret],
        index=idx
    )
    charge_limit_mwh_hourly = pd.Series(
        [buffer_charge_limit_mwh(buffer_charge_mdot_max_kg_s, Ts_day, tr, dt_hours, cp) for tr in Tret],
        index=idx
    )
    discharge_limit_mwh_hourly = pd.Series(
        [buffer_discharge_limit_mwh(buffer_discharge_mdot_max_kg_s, Ts_day, tr, dt_hours, cp) for tr in Tret],
        index=idx
    )

    target_soc_needed_mwh = required_discharge_mwh / max(discharge_eff, 1e-12)

    soc = pd.Series(0.0, index=idx)
    q_charge_mw = pd.Series(0.0, index=idx)

    remaining_soc_to_build = max(0.0, min(target_soc_needed_mwh, float(cap_buffer_mwh_hourly.max())))

    for i in range(min(charge_hours, len(idx))):
        if remaining_soc_to_build <= 1e-12:
            break

        prev_soc = soc.iloc[i-1] if i > 0 else 0.0

        max_charge_input_mwh = min(
            float(headroom_mw.iloc[i]) * dt_hours,
            float(charge_limit_mwh_hourly.iloc[i]),
            max(0.0, float(cap_buffer_mwh_hourly.iloc[i]) - prev_soc)
        )

        max_soc_increase = max_charge_input_mwh * charge_eff
        soc_increase = min(remaining_soc_to_build, max(0.0, max_soc_increase))

        charge_input_mwh = soc_increase / max(charge_eff, 1e-12)
        q_charge_mw.iloc[i] = charge_input_mwh / dt_hours
        charge_mdot_kg_s.iloc[i] = q_charge_mw.iloc[i] * 1e6 / (cp * float(dT.iloc[i]) * dt_hours)
        discharge_mdot_kg_s.iloc[i] = 0.0

        soc.iloc[i] = prev_soc + soc_increase
        remaining_soc_to_build -= soc_increase

    for i in range(charge_hours, len(idx)):
        soc.iloc[i] = soc.iloc[i-1]

    q_source_mw = (total_network_load + q_charge_mw).clip(upper=q_cap_mw)

    m_dot_used = (q_source_mw * 1e6) / (cp * dT)
    m_dot_used = m_dot_used.clip(lower=0.0, upper=m_dot_max)

    soc2 = pd.Series(np.nan, index=idx)
    soc2.iloc[0] = soc.iloc[0]

    q_discharge_mw = pd.Series(0.0, index=idx)
    unmet_mw = pd.Series(0.0, index=idx)

    for i in range(len(idx)):
        prev_soc = soc2.iloc[i-1] if i > 0 else 0.0

        charge_in_mwh = q_charge_mw.iloc[i] * dt_hours
        soc_after_charge = min(float(cap_buffer_mwh_hourly.iloc[i]), prev_soc + charge_in_mwh * charge_eff)

        need_mw = float(shortage_mw.iloc[i])

        if need_mw > 0:
            max_deliver_mwh_from_soc = min(
                soc_after_charge * discharge_eff,
                float(discharge_limit_mwh_hourly.iloc[i])
            )
            max_deliver_mw_from_soc = max_deliver_mwh_from_soc / dt_hours

            discharge_mw = min(need_mw, max_deliver_mw_from_soc)
            discharge_mdot_kg_s.iloc[i] = discharge_mw * 1e6 / (cp * float(dT.iloc[i]) * dt_hours)
            charge_mdot_kg_s.iloc[i] = 0.0
            deliver_mwh = discharge_mw * dt_hours

            soc_drop = deliver_mwh / max(discharge_eff, 1e-12)
            soc_end = soc_after_charge - soc_drop

            q_discharge_mw.iloc[i] = discharge_mw
            unmet_mw.iloc[i] = max(0.0, need_mw - discharge_mw)
        else:
            soc_end = soc_after_charge
            unmet_mw.iloc[i] = 0.0

        soc2.iloc[i] = soc_end

    charged_mwh = float((q_charge_mw * dt_hours).sum())
    discharged_mwh = float((q_discharge_mw * dt_hours).sum())
    unmet_mwh = float((unmet_mw * dt_hours).sum())

    summary = {
        "required_discharge_mwh": required_discharge_mwh,
        "target_soc_needed_mwh": target_soc_needed_mwh,
        "charged_mwh_first6h": charged_mwh,
        "discharged_mwh": discharged_mwh,
        "unmet_mwh": unmet_mwh,
        "soc_start_mwh": 0.0,
        "soc_end_mwh": float(soc2.iloc[-1]),
        "max_m_dot_kg_s": float(m_dot_used.max()),
    }

    out = pd.DataFrame({
        "demand_mw": demand,
        "heat_loss_mw": heat_loss,
        "total_network_load_mw": total_network_load,
        "Tret_C": Tret,
        "Ts_day_C": Ts_day,
        "dT_K": dT,
        "q_cap_mw": q_cap_mw,
        "headroom_mw": headroom_mw,
        "shortage_mw": shortage_mw,
        "buffer_capacity_mwh_hourly": cap_buffer_mwh_hourly,
        "charge_limit_mwh_hourly": charge_limit_mwh_hourly,
        "discharge_limit_mwh_hourly": discharge_limit_mwh_hourly,
        "q_charge_mw": q_charge_mw,
        "q_source_mw": q_source_mw,
        "m_dot_used_kg_s": m_dot_used,
        "q_discharge_mw": q_discharge_mw,
        "soc_mwh": soc2,
        "unmet_mw": unmet_mw,
        "charge_mdot_kg_s": charge_mdot_kg_s,
        "discharge_mdot_kg_s": discharge_mdot_kg_s,
    }, index=idx)

    return out, summary


results = []
daily_summaries = []

for day, day_df in data_hourly.groupby(pd.Grouper(freq="D")):
    if len(day_df) == 0:
        continue

    Ts_day = float(Ts_hourly_min_80.loc[day_df.index[0]])

    day_out, summ = simulate_day_charge_first6h_empty_start(
        day_df=day_df,
        Ts_day=daily_Ts6h['Ts_min_day_with_buffer_C_80'].loc[day],
        m_dot_max=m_dot_max,
        cp=cp,
        buffer_volume_m3=buffer_volume_m3,
        buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
        buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
        charge_hours=6,
        charge_eff=charge_efficiency,
        discharge_eff=discharge_efficiency,
        demand_col="useful_demand_mw",
        T_return_col="AVR_ARN_TR_monthly",
        dt_hours=1.0,
    )

    day_out["day"] = day
    results.append(day_out)

    summ["day"] = day
    daily_summaries.append(summ)

year_policy_res = pd.concat(results)
daily_policy_summary = pd.DataFrame(daily_summaries).set_index("day").sort_index()
print(daily_policy_summary.head())
print("Total unmet energy (MWh):", float(daily_policy_summary["unmet_mwh"].sum()))

plt.figure(figsize=(20,6))
plt.plot(year_policy_res.index, year_policy_res["m_dot_used_kg_s"].values, label="m_dot used (kg/s)")
plt.title("Hourly mass flow under 'charge first 6h' policy")
plt.xlabel("Date")
plt.ylabel("kg/s")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()

plt.figure(figsize=(20,6))
plt.plot(daily_policy_summary.index, daily_policy_summary["unmet_mwh"].values, label="Unmet (MWh/day)")
plt.title("Daily unmet energy (backup boiler) under 'charge first 6h' policy")
plt.xlabel("Date")
plt.ylabel("MWh/day")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()

plt.figure(figsize=(20,6))
plt.plot(year_policy_res.index, year_policy_res["charge_mdot_kg_s"].values, label="Charge mass flow (kg/s)")
plt.plot(year_policy_res.index, year_policy_res["discharge_mdot_kg_s"].values, label="Discharge mass flow (kg/s)")
plt.title("Hourly mass flow for charging and discharging under 'charge first 6h' policy")
plt.xlabel("Date")
plt.ylabel("Mass flow (kg/s)")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()

soc_hourly_6h = year_policy_res["soc_mwh"]
max_soc6h = soc_hourly_6h.max()
max_soc_time6h = soc_hourly_6h.idxmax()
max_discharge6h = year_policy_res["q_discharge_mw"].max()
print(f"Maximum SOC reached: {max_soc6h:.2f} MWh at {max_soc_time6h}")
print(f"Maximum discharge power: {max_discharge6h:.2f} MW")

In [ ]:
plot_buffer_charge_discharge_two_zooms_with_temp_demand(
    charge=year_policy_res["q_charge_mw"],
    discharge=year_policy_res["q_discharge_mw"],
    supply_temp=daily_Ts6h['Ts_min_day_with_buffer_C_80'].reindex(year_policy_res.index, method='ffill'),
    useful_heat_demand=data_hourly["useful_demand_mw"],
    zoom1_start="2024-01-08 00:00:00",
    zoom1_end="2024-01-18 23:00:00",
    zoom2_start="2024-03-01 00:00:00",
    zoom2_end="2024-03-11 23:00:00",
    title="Hourly buffer charge (+) and discharge (-) over the year, under 6H strategy",
    zoom1_title="Zoom: 8 - 18 January",
    zoom2_title="Zoom: 1 - 11 March",
    save_title="Buffer_charge_discharge_temperature_demand_6H_BFCS_1"
)

6H CARRYOVER


In [ ]:
def simulate_day_charge_first6h_then_discharge(
    day_df: pd.DataFrame,
    Ts_day: float,
    m_dot_max: float,
    cp: float,
    buffer_volume_m3: float,
    buffer_charge_mdot_max_kg_s: float,
    buffer_discharge_mdot_max_kg_s: float,
    soc_start_mwh: float = 0.0,
    charge_eff=charge_efficiency,
    discharge_eff=discharge_efficiency,
    demand_col: str = "useful_demand_mw",
    T_return_col: str = "AVR_ARN_TR_monthly",
    dt_hours: float = 1.0,
    charge_hours: int = 6,#6
):
    idx = day_df.index
    demand = day_df[demand_col].astype(float)
    Tret = day_df[T_return_col].astype(float)

    dT = (Ts_day - Tret).clip(lower=1e-6)
    q_cap_mw = (m_dot_max * cp * dT) / 1e6
    charge_mdot_kg_s = pd.Series(0.0, index=idx, dtype=float)
    discharge_mdot_kg_s = pd.Series(0.0, index=idx, dtype=float)
    heat_loss_mw = float(lookup_heat_loss_year(Ts_day))
    heat_loss = pd.Series(heat_loss_mw, index=idx, dtype=float)
    total_network_load = demand + heat_loss
    shortage_mw = (total_network_load - q_cap_mw).clip(lower=0.0)
    required_discharge_mwh = float((shortage_mw * dt_hours).sum())
    headroom_mw = (q_cap_mw - total_network_load).clip(lower=0.0)

    cap_buffer_mwh_hourly = pd.Series(
        [buffer_capacity_mwh_from_volume(buffer_volume_m3, Ts_day, tr) for tr in Tret],
        index=idx
    )
    charge_limit_mwh_hourly = pd.Series(
        [buffer_charge_limit_mwh(buffer_charge_mdot_max_kg_s, Ts_day, tr, dt_hours, cp) for tr in Tret],
        index=idx
    )
    discharge_limit_mwh_hourly = pd.Series(
        [buffer_discharge_limit_mwh(buffer_discharge_mdot_max_kg_s, Ts_day, tr, dt_hours, cp) for tr in Tret],
        index=idx
    )

    q_charge_mw = pd.Series(0.0, index=idx)
    soc = pd.Series(np.nan, index=idx)
    soc.iloc[0] = float(min(soc_start_mwh, float(cap_buffer_mwh_hourly.iloc[0])))

    target_soc_needed_mwh = required_discharge_mwh / max(discharge_eff, 1e-12)
    remaining_soc_to_build = max(0.0, target_soc_needed_mwh - soc.iloc[0])

    for i in range(len(idx)):
        prev_soc = soc.iloc[i-1] if i > 0 else soc.iloc[0]

        if i < charge_hours and remaining_soc_to_build > 1e-12:
            max_charge_input_mwh = min(
                float(headroom_mw.iloc[i]) * dt_hours,
                float(charge_limit_mwh_hourly.iloc[i]),
                max(0.0, float(cap_buffer_mwh_hourly.iloc[i]) - prev_soc)
            )

            max_soc_increase = max_charge_input_mwh * charge_eff
            soc_increase = min(remaining_soc_to_build, max(0.0, max_soc_increase))

            charge_input_mwh = soc_increase / max(charge_eff, 1e-12)
            q_charge_mw.iloc[i] = charge_input_mwh / dt_hours
            charge_mdot_kg_s.iloc[i] = q_charge_mw.iloc[i] * 1e6 / (cp * float(dT.iloc[i]) * dt_hours)
            discharge_mdot_kg_s.iloc[i] = 0.0

            soc_now = prev_soc + soc_increase
            remaining_soc_to_build -= soc_increase
        else:
            soc_now = prev_soc

        soc.iloc[i] = min(float(cap_buffer_mwh_hourly.iloc[i]), soc_now)

    q_source_mw = (total_network_load + q_charge_mw).clip(upper=q_cap_mw)

    m_dot_used = (q_source_mw * 1e6) / (cp * dT)
    m_dot_used = m_dot_used.clip(lower=0.0, upper=m_dot_max)

    q_discharge_mw = pd.Series(0.0, index=idx)
    unmet_mw = pd.Series(0.0, index=idx)

    soc2 = pd.Series(np.nan, index=idx)
    soc2.iloc[0] = soc.iloc[0]

    for i in range(len(idx)):
        prev_soc = soc2.iloc[i-1] if i > 0 else soc.iloc[0]

        charge_in_mwh = q_charge_mw.iloc[i] * dt_hours
        soc_after_charge = min(float(cap_buffer_mwh_hourly.iloc[i]), prev_soc + charge_in_mwh * charge_eff)

        need_mw = float(shortage_mw.iloc[i])

        if need_mw > 0:
            max_deliver_mwh_from_soc = min(
                soc_after_charge * discharge_eff,
                float(discharge_limit_mwh_hourly.iloc[i])
            )
            max_deliver_mw_from_soc = max_deliver_mwh_from_soc / dt_hours

            discharge_mw = min(need_mw, max_deliver_mw_from_soc)
            discharge_mdot_kg_s.iloc[i] = discharge_mw * 1e6 / (cp * float(dT.iloc[i]) * dt_hours)
            charge_mdot_kg_s.iloc[i] = 0.0

            deliver_mwh = discharge_mw * dt_hours

            soc_drop = deliver_mwh / max(discharge_eff, 1e-12)
            soc_end = soc_after_charge - soc_drop

            q_discharge_mw.iloc[i] = discharge_mw
            unmet_mw.iloc[i] = max(0.0, need_mw - discharge_mw)
        else:
            soc_end = soc_after_charge
            unmet_mw.iloc[i] = 0.0

        soc2.iloc[i] = soc_end

    charged_mwh = float((q_charge_mw * dt_hours).sum())
    discharged_mwh = float((q_discharge_mw * dt_hours).sum())
    unmet_mwh = float((unmet_mw * dt_hours).sum())

    summary = {
        "required_discharge_mwh": required_discharge_mwh,
        "target_soc_needed_mwh": target_soc_needed_mwh,
        "charged_mwh_first6h": charged_mwh,
        "discharged_mwh": discharged_mwh,
        "unmet_mwh": unmet_mwh,
        "soc_start_mwh": float(soc.iloc[0]),
        "soc_end_mwh": float(soc2.iloc[-1]),
        "max_m_dot_kg_s": float(m_dot_used.max()),
    }

    out = pd.DataFrame({
        "demand_mw": demand,
        "heat_loss_mw": heat_loss,
        "total_network_load_mw": total_network_load,
        "Tret_C": Tret,
        "Ts_day_C": Ts_day,
        "dT_K": dT,
        "q_cap_mw": q_cap_mw,
        "headroom_mw": headroom_mw,
        "shortage_mw": shortage_mw,
        "buffer_capacity_mwh_hourly": cap_buffer_mwh_hourly,
        "charge_limit_mwh_hourly": charge_limit_mwh_hourly,
        "discharge_limit_mwh_hourly": discharge_limit_mwh_hourly,
        "q_charge_mw": q_charge_mw,
        "q_source_mw": q_source_mw,
        "m_dot_used_kg_s": m_dot_used,
        "q_discharge_mw": q_discharge_mw,
        "soc_mwh": soc2,
        "unmet_mw": unmet_mw,
        "charge_mdot_kg_s": charge_mdot_kg_s,
        "discharge_mdot_kg_s": discharge_mdot_kg_s,

    }, index=idx)

    return out, float(soc2.iloc[-1]), summary


results = []
daily_summaries = []
soc = 0.0

for day, day_df in data_hourly.groupby(pd.Grouper(freq="D")):
    if len(day_df) == 0:
        continue

    Ts_day = float(Ts_hourly_min_80.loc[day_df.index[0]])

    day_out, soc, summ = simulate_day_charge_first6h_then_discharge(
        day_df=day_df,
        Ts_day=Ts_day,
        m_dot_max=m_dot_max,
        cp=cp,
        buffer_volume_m3=buffer_volume_m3,
        buffer_charge_mdot_max_kg_s=buffer_charge_mdot_max_kg_s,
        buffer_discharge_mdot_max_kg_s=buffer_discharge_mdot_max_kg_s,
        soc_start_mwh=soc,
        charge_eff=charge_efficiency,
        discharge_eff=discharge_efficiency,
        demand_col="useful_demand_mw",
        T_return_col="AVR_ARN_TR_monthly",
        dt_hours=1.0,
        charge_hours=6,
    )

    day_out["day"] = day
    results.append(day_out)

    summ["day"] = day
    daily_summaries.append(summ)

year_policy_res_with_overdracht = pd.concat(results)
daily_policy_summary = pd.DataFrame(daily_summaries).set_index("day").sort_index()

print(daily_policy_summary.head())
print("Total unmet energy (MWh):", float(daily_policy_summary["unmet_mwh"].sum()))

plt.figure(figsize=(20,6))
plt.plot(year_policy_res_with_overdracht.index, year_policy_res_with_overdracht["m_dot_used_kg_s"].values, label="m_dot used (kg/s)")
plt.title("Hourly mass flow under 'charge first 6h with carryover' policy")
plt.xlabel("Date")
plt.ylabel("kg/s")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()

plt.figure(figsize=(20,6))
plt.plot(daily_policy_summary.index, daily_policy_summary["unmet_mwh"].values, label="Unmet (MWh/day)")
plt.title("Daily unmet energy (backup boiler)")
plt.xlabel("Date")
plt.ylabel("MWh/day")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()

#print mass flows of charge and discharge for the whole year
plt.figure(figsize=(20,6))
plt.plot(year_policy_res_with_overdracht.index, year_policy_res_with_overdracht["charge_mdot_kg_s"].values, label="Charge mass flow (kg/s)")
plt.plot(year_policy_res_with_overdracht.index, year_policy_res_with_overdracht["discharge_mdot_kg_s"].values, label="Discharge mass flow (kg/s)")
plt.title("Hourly mass flow for charging and discharging under 'charge first 6h with carryover' policy")
plt.xlabel("Date")
plt.ylabel("Mass flow (kg/s)")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()

soc_hourly_6hwh = year_policy_res_with_overdracht["soc_mwh"]    
max_soc6hwh = soc_hourly_6hwh.max()
max_soc_time6hwh = soc_hourly_6hwh.idxmax()
max_discharge6h = year_policy_res_with_overdracht["q_discharge_mw"].max()
print(f"Maximum SOC reached: {max_soc6hwh:.2f} MWh at {max_soc_time6hwh}")
print(f"Maximum discharge power: {max_discharge6h:.2f} MW")

In [ ]:
plot_buffer_charge_discharge_two_zooms_with_temp_demand(
    charge=year_policy_res_with_overdracht["q_charge_mw"],
    discharge=year_policy_res_with_overdracht["q_discharge_mw"],
    supply_temp=daily_Ts6h_carryover['Ts_min_day_with_buffer_C_80'].reindex(year_policy_res_with_overdracht.index, method='ffill'),
    useful_heat_demand=data_hourly["useful_demand_mw"],
    zoom1_start="2024-01-04 00:00:00",
    zoom1_end="2024-01-13 23:00:00",
    zoom2_start="2024-03-01 00:00:00",
    zoom2_end="2024-03-11 23:00:00",
    title="Hourly buffer charge (+) and discharge (-) over the year, under 6H carryover strategy",
    zoom1_title="Zoom: 8 - 18 January",
    zoom2_title="Zoom: 1 - 11 March",
    save_title="Buffer_charge_discharge_temperature_demand_6H_Carryover_BFCS_1"
)

In [ ]:
#massflow dot charts
plt.figure(figsize=(18, 6))
plt.scatter(year_policy_res_with_overdracht.index, year_policy_res_with_overdracht["m_dot_used_kg_s"].values, label="m_dot used (kg/s)",s=5)
plt.title('Mass flow per hour from waste incineration, 6H carryover', fontsize = 16)
plt.xlabel('Date', fontsize = 14)
plt.ylabel('Mass flow (kg/s)', fontsize = 14)
plt.ylim(20, 165)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)
save_current_plot_from_title()
plt.show()

plt.figure(figsize=(18, 6))
plt.scatter(year_policy_res.index, year_policy_res["m_dot_used_kg_s"].values, label="m_dot used (kg/s)",s=5)
plt.title('Mass flow per hour from waste incineration, 6H', fontsize = 16)
plt.xlabel('Date', fontsize = 14)
plt.ylabel('Mass flow (kg/s)', fontsize = 14)
plt.ylim(20, 165)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)
save_current_plot_from_title()
plt.show()

plt.figure(figsize=(18, 6))
plt.scatter(year_res.index, year_res["m_dot_used_kg_s"].values, label="m_dot used (kg/s)",s=5)
plt.title('Mass flow per hour from waste incineration, perfect', fontsize = 16)
plt.xlabel('Date', fontsize = 14)
plt.ylabel('Mass flow (kg/s)', fontsize = 14)
plt.ylim(20, 165)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)
save_current_plot_from_title()
plt.show()

plt.figure(figsize=(18, 6))
plt.scatter(mass_flow_hourly_capped.index, mass_flow_hourly_capped.values, label='Required mass flow (kg/s)',s=5)
plt.title('Mass flow per hour from waste incineration, base', fontsize = 16)
plt.xlabel('Date', fontsize = 14)
plt.ylabel('Mass flow (kg/s)', fontsize = 14)
plt.ylim(20, 165)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)
save_current_plot_from_title()
plt.show()

In [ ]:
def plot_mass_flow_with_zoom(
    perfect_series,
    base_series,
    zoom_start,
    zoom_end,
    y_min=20,
    y_max=165,
    save=True
):


    zoom_start = pd.to_datetime(zoom_start)
    zoom_end = pd.to_datetime(zoom_end)

    perfect_zoom = perfect_series.loc[zoom_start:zoom_end]
    base_zoom = base_series.loc[zoom_start:zoom_end]

    fig, axes = plt.subplots(
        2, 2,
        figsize=(20, 9),
        gridspec_kw={"width_ratios": [2.2, 1.2]},
        sharey="row"
    )

    scenarios = [
        {
            "name": "Base",
            "series": base_series,
            "zoom": base_zoom,
            "row": 0
        },
        {
            "name": "Perfect",
            "series": perfect_series,
            "zoom": perfect_zoom,
            "row": 1
        }
    ]

    for scenario in scenarios:
        row = scenario["row"]
        name = scenario["name"]
        series = scenario["series"]
        zoom = scenario["zoom"]

        # Full-year scatterplot
        ax_full = axes[row, 0]
        ax_full.scatter(
            series.index,
            series.values,
            s=5,
            label=f"{name} mass flow"
        )

        ax_full.axvline(zoom_start, linestyle="--", linewidth=1.5)
        ax_full.axvline(zoom_end, linestyle="--", linewidth=1.5)
        ax_full.axvspan(zoom_start, zoom_end, alpha=0.08)

        ax_full.set_title(f"Mass flow per hour from waste incineration, {name.lower()}", fontsize=14)
        ax_full.set_ylabel("Mass flow (kg/s)", fontsize=14)
        ax_full.set_ylim(y_min, y_max)
        ax_full.grid(True)
        ax_full.tick_params(axis="both", labelsize=10)

        # Zoomed line chart
        ax_zoom = axes[row, 1]
        ax_zoom.plot(
            zoom.index,
            zoom.values,
            linewidth=1.8,
            label=f"{name} zoom"
        )

        ax_zoom.set_title(f"Zoom: {zoom_start:%d %b}--{zoom_end:%d %b}", fontsize=14)
        ax_zoom.set_ylim(y_min, y_max)
        ax_zoom.grid(True)
        ax_zoom.tick_params(axis="both", labelsize=10)
        ax_zoom.tick_params(axis="x", rotation=45)

    axes[1, 0].set_xlabel("Date", fontsize=14)
    axes[1, 1].set_xlabel("Date", fontsize=14)

    #fig.suptitle("Hourly mass flow from waste incineration: full year and zoom", fontsize=16)
    plt.tight_layout()

    if save:
        save_current_plot_from_title()

    plt.show()

perfect_mass_flow = year_res["m_dot_used_kg_s"]
base_mass_flow = mass_flow_hourly_capped

plot_mass_flow_with_zoom(
    perfect_series=perfect_mass_flow,
    base_series=base_mass_flow,
    zoom_start="2024-03-01",
    zoom_end="2024-03-08"
)

In [ ]:
#plot year_policy_res_with_overdracht["soc_mwh"]
plt.figure(figsize=(20,6))
plt.plot(year_policy_res_with_overdracht.index, year_policy_res_with_overdracht["soc_mwh"].values, label="SOC (MWh)")
plt.title("Buffer state of charge (SOC) over the year under 'charge first 6h with overdracht' policy")
plt.xlabel("Date")
plt.ylabel("SOC (MWh)")
plt.grid(True)
save_current_plot_from_title()
plt.show()

plt.figure(figsize=(20,6))
plt.plot(year_policy_res.index, year_policy_res["soc_mwh"].values, label="SOC (MWh)")
plt.title("Buffer state of charge (SOC) over the year under 'charge first 6h' policy")
plt.xlabel("Date")
plt.ylabel("SOC (MWh)")
plt.grid(True)
save_current_plot_from_title()
plt.show()

In [ ]:
#zoom in on mass flow of first week of the year
plt.figure(figsize=(20,6))
plt.plot(year_policy_res.index[:24*7], year_policy_res["m_dot_used_kg_s"].values[:24*7], label="m_dot used (kg/s)")
plt.title("Hourly mass flow under 'charge first 6h' policy (First Week)")
plt.xlabel("Date")
plt.ylabel("kg/s")
plt.grid(True)
plt.legend()
save_current_plot_from_title()
plt.show()

ASSESSMENT INDICATOR CALCULATIONS


Pressure loss and pumping power over the year

In [ ]:
import import_ipynb
from pressure_loss_max_m import PipeNetwork
net = PipeNetwork(
    lengths_m=[,
    diameters_m=,
    fractions_out=[],
    rho=980.0,
    f=0.016,
    minor_loss_factor=1.10,
    P0_bar=,
    P1_max_bar=,
    dP_total_set_bar=
)

In [ ]:

def pressure_timeseries_year(
    net: PipeNetwork,
    m0_hourly: pd.Series,
    store_boundaries: bool = False,
    store_flows: bool = False,
) -> pd.DataFrame:

    if not isinstance(m0_hourly, pd.Series):
        raise TypeError("m0_hourly must be a pandas Series (hourly index).")

    n = len(net.lengths_m)

    # Pre-allocate arrays (fast + memory-safe)
    dP_sec = np.full((len(m0_hourly), n), np.nan, dtype=float)
    P_end  = np.full(len(m0_hourly), np.nan, dtype=float)

    if store_boundaries:
        P_b = np.full((len(m0_hourly), n + 1), np.nan, dtype=float)
    if store_flows:
        m_in  = np.full((len(m0_hourly), n), np.nan, dtype=float)
        m_out = np.full((len(m0_hourly), n), np.nan, dtype=float)

    # Loop over hours (8760 is prima)
    for k, (ts, m0) in enumerate(m0_hourly.items()):
        if pd.isna(m0):
            continue

        P, dP_sections, m_in_sections, m_out_sections = net.pressure_profile_bar(float(m0))

        dP_sec[k, :] = dP_sections
        P_end[k] = P[-1]

        if store_boundaries:
            P_b[k, :] = P
        if store_flows:
            m_in[k, :]  = m_in_sections
            m_out[k, :] = m_out_sections

    # Build output DataFrame
    out = pd.DataFrame(index=m0_hourly.index)
    out["m0_kg_s"] = m0_hourly.values
    out["dP_total_bar"] = np.nansum(dP_sec, axis=1)
    out["P_end_bar"] = P_end

    # Per-section ΔP columns
    for i in range(n):
        out[f"dP_s{i+1}_bar"] = dP_sec[:, i]

    # Optional extras
    if store_boundaries:
        for i in range(n + 1):
            out[f"P_b{i}_bar"] = P_b[:, i]  # boundary pressures (b0 = P0)
    if store_flows:
        for i in range(n):
            out[f"m_in_s{i+1}_kg_s"]  = m_in[:, i]
            out[f"m_out_s{i+1}_kg_s"] = m_out[:, i]

    return out

In [ ]:
dfP_supply = pressure_timeseries_year(
    net,
    year_policy_res["m_dot_used_kg_s"],
    store_boundaries=False,
    store_flows=False
)

dfP_supply_return = dfP_supply.copy()
dfP_supply_return["dP_total_bar"] = dfP_supply_return["dP_total_bar"] * 2 + delta_P_customer


plt.figure(figsize=(14,5))
plt.plot(dfP_supply_return.index, dfP_supply_return["dP_total_bar"], label="Total Pressure Drop (Supply + Return)")
plt.title("Total Pressure Drop Over the Year")
plt.ylabel("ΔP_total [bar]")
plt.xlabel("Time")
plt.grid(True)
plt.tight_layout()
save_current_plot_from_title()
plt.show()

In [ ]:
dfP_supply_return["pump_power_W"] = (dfP_supply_return["dP_total_bar"] * 1e5 * year_policy_res["m_dot_used_kg_s"]/ (rho * pump_efficiency))
dfP_supply_return["pump_power_MW"] = dfP_supply_return["pump_power_W"] / 1e6
daily_pump_energy_mwh = (dfP_supply_return["pump_power_MW"] * 1.0).resample("D").sum()  # MWh/day
plt.figure(figsize=(14,5))
plt.plot(daily_pump_energy_mwh.index, daily_pump_energy_mwh.values, label="Daily Pump Energy (MWh)")
plt.title("Daily Pump Energy Consumption Over the Year")
plt.ylabel("Energy (MWh)")
plt.xlabel("Time")
plt.grid(True)
plt.tight_layout()
save_current_plot_from_title()
plt.show()


In [ ]:
#calculate the pumping costs needed to discharge the buffer
#pressure going into buffer
pressure_at_end_supply = (dfP_supply_return["dP_total_bar"]+3-delta_P_customer)/2
pumping_energy_discharging_W = (pressure_at_end_supply-6)*1e5 * year_policy_res["discharge_mdot_kg_s"]/ (rho * pump_efficiency)
pumping_energy_discharging_MW = pumping_energy_discharging_W / 1e6
pressure_at_begin_return = pressure_at_end_supply -1.5
pumping_energy_charging_W= (pressure_at_begin_return-6)*1e5 * year_policy_res["charge_mdot_kg_s"]/ (rho * pump_efficiency)
pumping_energy_charging_MW= pumping_energy_charging_W/1e6

daily_pump_energy_charge = pumping_energy_charging_MW.resample('D').sum()
daily_pump_energy_discharge = pumping_energy_discharging_MW.resample('D').sum()

plt.figure(figsize=(14,5))
plt.plot(daily_pump_energy_charge.index, daily_pump_energy_charge.values, label="Daily Pump Energy charging (MWh)")
plt.title("Daily Pump Energy Consumption Over the Year for charging")
plt.ylabel("Energy (MWh)")
plt.xlabel("Time")
plt.grid(True)
plt.tight_layout()
save_current_plot_from_title()
plt.show()

plt.figure(figsize=(14,5))
plt.plot(daily_pump_energy_discharge.index, daily_pump_energy_discharge.values, label="Daily Pump Energy discharging (MWh)")
plt.title("Daily Pump Energy Consumption Over the Year for discharging")
plt.ylabel("Energy (MWh)")
plt.xlabel("Time")
plt.grid(True)
plt.tight_layout()
save_current_plot_from_title()
plt.show()

# Sum pump energy over the whole year
total_pump_energy_charge = daily_pump_energy_charge.sum()
total_pump_energy_discharge = daily_pump_energy_discharge.sum()

print("Total yearly pump energy (charging) [MWh]:", total_pump_energy_charge)
print("Total yearly pump energy (discharging) [MWh]:", total_pump_energy_discharge)

print("Total yearly pump energy (combined) [MWh]:", 
      total_pump_energy_charge + total_pump_energy_discharge)

total_gj_charged = year_policy_res['q_charge_mw'].sum()*3.6
print(total_gj_charged)
print(total_gj_charged/(total_pump_energy_charge*1000))








HEAT LOSS

In [ ]:
mass_flow_2024 = pd.read_excel('xx.xlsx', index_col=0, parse_dates=True)
mass_flow_2024_avr = mass_flow_2024['Mass flow AVR']
mass_flow_2024_veolia = mass_flow_2024['Mass flow Veolia']
mass_flow_2024_HWC = mass_flow_2024['Mass flow HWC']
mass_flow_2024_total = mass_flow_2024['Mass flow total']
mass_flow_2024_total_min_HWC = mass_flow_2024_total - mass_flow_2024_HWC


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

# Left y-axis: mass flow
ax1.plot(
    mass_flow_2024_total_min_HWC.index,
    mass_flow_2024_total_min_HWC.values,
    label='Mass flow',
)
ax1.set_title('Mass flow in 2024 with gas dispatch')
ax1.set_xlabel('Date')
ax1.set_ylabel('Mass flow (kg/s)')
ax1.grid(True)

# Right y-axis: gas power
ax2 = ax1.twinx()
ax2.plot(
    data_hourly.index,
    data_hourly['GAS_SCH_MW'],
    label='Gas dispatch',
    color='orange'
)
ax2.set_ylabel('Gas power (MW)')
ax2.set_ylim(0, 100)

# Combine legends from both axes
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right')
save_current_plot_from_title()
plt.show()




In [ ]:
plt.figure(figsize=(14,5))
plt.plot(data_hourly['GAS_SCH_MW'], label='Gas schedule (MW)')
plt.title('GAS power HWC 2024')


In [ ]:
print(colnames := data_hourly.columns)

In [ ]:
pressure_hwc_relation = pd.read_excel('xx.xlsx', index_col=0)

In [ ]:

pressure_no_gas_avr = pressure_hwc_relation['AVR_kPa_supply'].where(pressure_hwc_relation['HWC_KG_S']==0)
pressure_with_gas_avr = pressure_hwc_relation['AVR_kPa_supply'].where(pressure_hwc_relation['HWC_KG_S']>0)


pressure_no_gas_hwc = pressure_hwc_relation['HWC_kPa_supply'].where(pressure_hwc_relation['HWC_KG_S']==0)
pressure_with_gas_hwc = pressure_hwc_relation['HWC_kPa_supply'].where(pressure_hwc_relation['HWC_KG_S']>0)


massflow_when_no_gas = mass_flow_2024_total_min_HWC.where(data_hourly['GAS_SCH_MW'] == 0)
massflow_when_gas = mass_flow_2024_total_min_HWC.where(data_hourly['GAS_SCH_MW'] > 0)
avr_mw_when_gas = data_hourly['AVR_ARN_MW'].where(data_hourly['GAS_SCH_MW'] > 0)
t_out_gas = data_hourly['T_buiten'].where(data_hourly['GAS_SCH_MW'] > 0)
t_out_no_gas = data_hourly['T_buiten'].where(data_hourly['GAS_SCH_MW'] == 0)


plt.figure(figsize=(15,7.5))
plt.scatter(massflow_when_no_gas.index,massflow_when_no_gas,s=8, label = "Boiler not active")
plt.scatter(massflow_when_gas.index,massflow_when_gas,s=8, label = "Boiler active")
plot_title = "Mass flow when gas boiler is active vs not active"
plt.ylabel("Mass flow [kg/s]", fontsize=14)
plt.xlabel("Date", fontsize=14)
plt.grid(True)
plt.legend(fontsize=14)
save_current_plot_from_title(plot_title)
plt.show()

# plt.figure(figsize=(20,10))

# plt.title("Mass flow when gas boiler is not active")
# plt.ylabel("Mass flow [kg/s]")
# plt.xlabel("Time")
# plt.grid(True)
# plt.show()


plt.figure(figsize=(20,10))
plt.scatter(avr_mw_when_gas.index, avr_mw_when_gas,s=8)
plt.title("AVR supply when boiler is active")
plt.ylabel("AVR supply (MW)")
plt.xlabel("Date")
plt.grid(True)

save_current_plot_from_title()
plt.show()

plt.figure(figsize=(20,10))
plt.scatter(t_out_gas.index, t_out_gas, s=8, label="With Gas")
plt.scatter(t_out_no_gas.index, t_out_no_gas, s=8, label="Without Gas")
plt.title("Outdoor temperature when boiler is active")
plt.ylabel("Outdoor temperature (°C)")
plt.xlabel("Date")
plt.legend()
plt.grid(True)
save_current_plot_from_title()
plt.show()


#make bars of pressure no gas avr and pressure with gas avr
pressure_no_gas_avr = pressure_no_gas_avr.dropna()/100
pressure_with_gas_avr = pressure_with_gas_avr.dropna()

plt.figure(figsize=(15,7.5))
plt.scatter(x=pd.to_datetime(pressure_no_gas_avr.index), y=pressure_no_gas_avr.values, s=8, label="Without Gas")
plot_title = "Pressure levels at main heat source supply line when boiler is not active"
plt.ylabel("Supply pressure (Bar)", fontsize=14)
plt.xlabel("Date", fontsize=14)
plt.grid(True)
save_current_plot_from_title(plot_title)
plt.show()


plt.figure(figsize=(20,10))
plt.scatter(x=pd.to_datetime(pressure_no_gas_hwc.index), y=pressure_no_gas_hwc.values, s=8, label="Without Gas")
plt.scatter(x=pd.to_datetime(pressure_with_gas_hwc.index), y=pressure_with_gas_hwc.values, s=8, label="With Gas")
plt.title("AVR supply pressure when boiler is active")
plt.ylabel("AVR supply pressure (kPa)")
plt.xlabel("Date")
plt.legend()
plt.grid(True)
save_current_plot_from_title()
plt.show()





In [ ]:
average_supply_temperature = daily_Ts['Ts_min_day_with_buffer_C_80'].mean()
print("Average supply temperature (°C):", average_supply_temperature)
average_supply_temperature_original = data_hourly['AVR_ARN_TA'].mean()
print("Average supply temperature original (°C):", average_supply_temperature_original)
average_supply_temperature_with_cap = maximum_daily_temp_with_cap.mean()
print("Average supply temperature with cap (°C):", average_supply_temperature_with_cap)

Total Heat loss

In [ ]:
yearly_heat_loss_gj_with_buffer = lookup_heat_loss_year(average_supply_temperature) * 8760  * 3.6
yearly_heat_loss_gj_orginal = lookup_heat_loss_year(average_supply_temperature_original) * 8760  * 3.6
yearly_heat_loss_gj_with_cap = lookup_heat_loss_year(average_supply_temperature_with_cap) * 8760  * 3.6
print("Estimated yearly heat loss with buffer (GJ):", yearly_heat_loss_gj_with_buffer)
print("Estimated yearly heat loss original (GJ):", yearly_heat_loss_gj_orginal)
print("Estimated yearly heat loss with cap (GJ):", yearly_heat_loss_gj_with_cap)

In [ ]:
#create a for loop that calculates heat loss for every hour of the year using ts_day and lookup_heat_loss
hourly_heat_loss_mw_buffer = year_res["Ts_day_C"].apply(lookup_heat_loss_year)
hourly_heat_loss_original_mw = data_hourly['AVR_ARN_TA'].apply(lookup_heat_loss_year)
hourly_heat_loss_base_mw = maximum_daily_temp_filled.apply(lookup_heat_loss_year)

yearly_heat_loss_looped_buffer = hourly_heat_loss_mw_buffer.sum()  # MW * 1h = MWh
print("Estimated yearly heat loss buffer:", yearly_heat_loss_looped_buffer)
yearly_heat_loss_looped_base = hourly_heat_loss_base_mw.sum() # MW * 1h = MWh
print("Estimated yearly heat loss base:", yearly_heat_loss_looped_base)
yearly_heat_loss_looped_original = hourly_heat_loss_original_mw.sum() # MW * 1h = MWh
print("Estimated yearly heat loss real:", yearly_heat_loss_looped_original)
delta_yearly_heat_loss = {}
delta_yearly_heat_loss['real_base']=hourly_heat_loss_original_mw.sum()-hourly_heat_loss_base_mw.sum()
delta_yearly_heat_loss['real_buffer']=hourly_heat_loss_original_mw.sum()-hourly_heat_loss_mw_buffer.sum()

print(delta_yearly_heat_loss['real_base'], delta_yearly_heat_loss['real_buffer'])

plt.figure(figsize=(20, 6))

plt.plot(
    hourly_heat_loss_mw_buffer.index,
    hourly_heat_loss_mw_buffer.values,
    label=f'Heat loss with buffer: {yearly_heat_loss_looped_buffer:,.0f} MWh/year',
    color='orange'
)

plt.plot(
    hourly_heat_loss_original_mw.index,
    hourly_heat_loss_original_mw.values,
    label=f'Heat loss 2024: {yearly_heat_loss_looped_original:,.0f} MWh/year',
    color='blue'
)

plt.plot(
    hourly_heat_loss_base_mw.index,
    hourly_heat_loss_base_mw.values,
    label=f'Heat loss base scenario: {yearly_heat_loss_looped_base:,.0f} MWh/year',
    color='red'
)

plt.title('Hourly Heat Loss Over the Year')
plt.xlabel('Time')
plt.ylabel('Heat Loss (MW)')
plt.legend()
plt.grid(True)
save_current_plot_from_title()
plt.show()

#plot heat loss per day in mwh
daily_heat_loss_buffer = hourly_heat_loss_mw_buffer.resample('D').sum()
daily_heat_loss_original = hourly_heat_loss_original_mw.resample('D').sum()
daily_heat_loss_base = hourly_heat_loss_base_mw.resample('D').sum()

plt.figure(figsize=(20, 6))
plt.plot(daily_heat_loss_buffer.index, daily_heat_loss_buffer.values, label='Heat loss with TES', color='orange')
plt.plot(daily_heat_loss_original.index, daily_heat_loss_original.values, label='Heat loss real', color='blue')
plt.plot(daily_heat_loss_base.index, daily_heat_loss_base.values, label='Heat loss base', color='red')
plt.xlabel('Date',fontsize=14)
plt.ylabel('Heat Loss (MWh)',fontsize=14)
plt.legend(fontsize=14)
plt.grid(True)
plot_title = "Daily_heat_loss_over_year_primary_DHN"
save_current_plot_from_title(plot_title)
plt.show()


In [ ]:
#sum of total useful demand yearly
total_useful_demand_mwh = data_hourly['useful_demand_mw'].sum()  # MW * 1h = MWh
print("Total useful demand in 2024 (MWh):", total_useful_demand_mwh)

ECONOMIC ANALYSIS 
1. calculate pumping energy for all the scenarios
    for the real scenario by hand, for the simulated ones with functions
2. define function for operating costs per year, including hp electricity costs
3. plotting and comparison



PUMPING

In [ ]:
#PUMP ENERGY FOR THE REAL SCENARIO
pump_data = pd.read_excel("xx.xlsx")
pump_data["DateTime"] = pd.to_datetime(pump_data["DateTime"])
pump_data = pump_data.set_index("DateTime")
pump_hourly = pump_data.resample("H").mean()
pump_hourly = pump_hourly.clip(lower=0)
delta_p_pa = (pump_hourly['AVR_kPa_supply'] - pump_hourly['AVR_kPa_return']) * 1000
delta_p_pa = delta_p_pa.clip(lower=0)
mdot_AVR_pump = pump_hourly['AVR_KG_S'].clip(lower=0)

pump_power_w = delta_p_pa * mdot_AVR_pump / (rho * pump_efficiency)
pump_power_mw = pump_power_w / 1e6
pump_energy_mwh = pump_power_mw
pump_energy_mwh = pump_energy_mwh.sum()

print(f"Total pumping energy: {pump_energy_mwh:.2f} MWh")
data_hourly['pumping energy']=pump_power_mw

In [ ]:
#calculate the costs for the base_scenario 
#add pumping energy to the base_scneario

dfP_supply_base = pressure_timeseries_year(
    net,
    mass_flow_hourly_capped,
    store_boundaries=False,
    store_flows=False
)

dfP_supply_return_base = dfP_supply_base.copy()
dfP_supply_return_base["dP_total_bar"] = dfP_supply_return_base["dP_total_bar"] * 2 + delta_P_customer
plt.figure(figsize=(14,5))
plt.plot(dfP_supply_return_base.index, dfP_supply_return_base["dP_total_bar"], label="Total Pressure Drop (Supply + Return)")
plt.title("Total Pressure Drop Over the Year")
plt.ylabel("ΔP_total [bar]")
plt.xlabel("Time")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
Ts_base_hourly = maximum_daily_temp_with_cap.reindex(
    data_hourly.index,
    method="ffill"
).clip(lower=80, upper=max_temp)

heat_loss_base_hourly = pd.Series(
    lookup_heat_loss_year(Ts_base_hourly),
    index=data_hourly.index
)

network_load_base_mw = data_hourly["useful_demand_mw"] + heat_loss_base_hourly

q_cap_base_mw = (
    m_dot_max * cp * (Ts_base_hourly - data_hourly["AVR_ARN_TR_monthly"])
) / 1e6

gas_base = (network_load_base_mw - q_cap_base_mw).clip(lower=0)

avr_base = network_load_base_mw - gas_base

In [ ]:
delta_p_pa_base = dfP_supply_return_base["dP_total_bar"] * 1e5
pump_energy_base_w = delta_p_pa_base * mass_flow_hourly_capped / (rho * pump_efficiency)

pump_energy_base_mw = pump_energy_base_w / 1e6
pump_energy_base_mwh = pump_energy_base_mw
pump_energy_mwh_base_total = pump_energy_base_mwh.sum()

print(f"Total pumping energy: {pump_energy_mwh_base_total:.2f} MWh")

data_hourly_base = pd.DataFrame({
    "AVR_base": avr_base,
    "GAS_base": gas_base,
    "TOT_ARN_MW": data_hourly["TOT_ARN_MW"],
    "useful_demand_mw": data_hourly["useful_demand_mw"],
    "heat_loss_mw": heat_loss_base_hourly,
    "total_network_load_mw": network_load_base_mw,
    "pump_energy_base_mw": pump_energy_base_mw
})

In [ ]:
def calculate_hourly_pumping_power(
    net,
    year_policy_res,
    rho,
    pump_efficiency,
    delta_P_customer,
    pressure_timeseries_year_func=pressure_timeseries_year,
    source_mdot_col="m_dot_used_kg_s",
    charge_mdot_col="charge_mdot_kg_s",
    discharge_mdot_col="discharge_mdot_kg_s",
    supply_pressure_offset_bar=3.0,
    return_pressure_drop_bar=1.5,
    buffer_reference_pressure_bar=6.0
):


    # Pressure drop in the network based on source mass flow
    dfP_supply = pressure_timeseries_year_func(
        net,
        year_policy_res[source_mdot_col],
        store_boundaries=False,
        store_flows=False
    )

    dfP_supply_return = dfP_supply.copy()
    dfP_supply_return["dP_total_bar"] = (
        dfP_supply_return["dP_total_bar"] * 2 + delta_P_customer
    )

    # Source pump power
    dfP_supply_return["pump_power_W"] = (
        dfP_supply_return["dP_total_bar"] * 1e5
        * year_policy_res[source_mdot_col]
        / (rho * pump_efficiency)
    )
    dfP_supply_return["pump_power_MW"] = dfP_supply_return["pump_power_W"] / 1e6

    # Pressure levels for buffer pumps
    pressure_at_end_supply = (
        dfP_supply_return["dP_total_bar"] + supply_pressure_offset_bar - delta_P_customer
    ) / 2

    pressure_at_begin_return = pressure_at_end_supply - return_pressure_drop_bar

    # Discharging pump power
    pumping_energy_discharging_W = (
        (pressure_at_end_supply - buffer_reference_pressure_bar) * 1e5
        * year_policy_res[discharge_mdot_col]
        / (rho * pump_efficiency)
    )
    pumping_energy_discharging_MW = pumping_energy_discharging_W / 1e6

    # Charging pump power
    pumping_energy_charging_W = (
        (pressure_at_begin_return - buffer_reference_pressure_bar) * 1e5
        * year_policy_res[charge_mdot_col]
        / (rho * pump_efficiency)
    )
    pumping_energy_charging_MW = pumping_energy_charging_W / 1e6

    # Total hourly pumping power
    total_pumping_power_MW = (
        dfP_supply_return["pump_power_MW"]
        + pumping_energy_charging_MW
        + pumping_energy_discharging_MW
    )

    # Daily energies assuming hourly timestep
    daily_source_pump_energy_MWh = dfP_supply_return["pump_power_MW"].resample("D").sum()
    daily_charge_pump_energy_MWh = pumping_energy_charging_MW.resample("D").sum()
    daily_discharge_pump_energy_MWh = pumping_energy_discharging_MW.resample("D").sum()
    daily_total_pump_energy_MWh = total_pumping_power_MW.resample("D").sum()

    # Year totals
    total_source_pump_energy_MWh = daily_source_pump_energy_MWh.sum()
    total_charge_pump_energy_MWh = daily_charge_pump_energy_MWh.sum()
    total_discharge_pump_energy_MWh = daily_discharge_pump_energy_MWh.sum()
    total_pump_energy_MWh = daily_total_pump_energy_MWh.sum()

    results = {
        "dfP_supply": dfP_supply,
        "dfP_supply_return": dfP_supply_return,
        "pressure_at_end_supply_bar": pressure_at_end_supply,
        "pressure_at_begin_return_bar": pressure_at_begin_return,
        "source_pump_power_MW": dfP_supply_return["pump_power_MW"],
        "charge_pump_power_MW": pumping_energy_charging_MW,
        "discharge_pump_power_MW": pumping_energy_discharging_MW,
        "total_pumping_power_MW": total_pumping_power_MW,
        "daily_source_pump_energy_MWh": daily_source_pump_energy_MWh,
        "daily_charge_pump_energy_MWh": daily_charge_pump_energy_MWh,
        "daily_discharge_pump_energy_MWh": daily_discharge_pump_energy_MWh,
        "daily_total_pump_energy_MWh": daily_total_pump_energy_MWh,
        "total_source_pump_energy_MWh": total_source_pump_energy_MWh,
        "total_charge_pump_energy_MWh": total_charge_pump_energy_MWh,
        "total_discharge_pump_energy_MWh": total_discharge_pump_energy_MWh,
        "total_pump_energy_MWh": total_pump_energy_MWh
    }

    return results


In [ ]:
pump_results_perfect = calculate_hourly_pumping_power(
    net=net,
    year_policy_res=year_res,
    rho=rho,
    pump_efficiency=pump_efficiency,
    delta_P_customer=delta_P_customer
)

year_res["pumping energy"]=pump_results_perfect["total_pumping_power_MW"]

pump_results_6h = calculate_hourly_pumping_power(
    net=net,
    year_policy_res=year_policy_res,
    rho=rho,
    pump_efficiency=pump_efficiency,
    delta_P_customer=delta_P_customer
)
year_policy_res["pumping energy"]=pump_results_6h["total_pumping_power_MW"]
pump_results_6h_overdracht = calculate_hourly_pumping_power(
    net=net,
    year_policy_res=year_policy_res_with_overdracht,
    rho=rho,
    pump_efficiency=pump_efficiency,
    delta_P_customer=delta_P_customer
)
year_policy_res_with_overdracht["pumping energy"]=pump_results_6h_overdracht["total_pumping_power_MW"]

print(year_policy_res_with_overdracht["pumping energy"].sum())
print(year_res["pumping energy"].sum())
print(year_policy_res["pumping energy"].sum())

COPS

In [ ]:
cop_table_geo_80 = pd.DataFrame(
    data=[
        [8.69, 7.28, 6.14, 5.26, 4.56],
        [7.45, 6.69, 5.64, 4.87, 4.32],
        [5.85, 5.56, 5.23, 4.62, 4.14],
        [5.14, 4.90, 4.65, 4.38, 4.07],
        [4.58, 4.39, 4.18, 3.97, 3.78]
    ],
    index=[80, 90, 100, 110, 120], 
    columns=[45, 50, 55, 60, 65]     
)

cop_table_geo_65 = pd.DataFrame(
    data=[
        [4.37, 3.92, 3.54, 3.22, 2.97],
        [4.04, 3.67, 3.34, 3.06, 2.86],
        [3.77, 3.54, 3.26, 3.01, 2.84],
        [3.55, 3.37, 3.18, 2.96, 2.81],
        [3.36, 3.21, 3.05, 2.87, 2.74]
    ],
    index=[80, 90, 100, 110, 120], 
    columns=[45, 50, 55, 60, 65]     
)

print(cop_table_geo_80), print(cop_table_geo_65)
def get_cop(Ts, Tr, cop_table):
    Ts_vals = np.sort(np.asarray(cop_table.index.values, dtype=float))
    Tr_vals = np.sort(np.asarray(cop_table.columns.values, dtype=float))

    # Clip to bounds so we never search outside the table
    Ts = float(np.clip(Ts, Ts_vals.min(), Ts_vals.max()))
    Tr = float(np.clip(Tr, Tr_vals.min(), Tr_vals.max()))

    # Find bounding values
    Ts_low = Ts_vals[Ts_vals <= Ts].max()
    Ts_high = Ts_vals[Ts_vals >= Ts].min()
    Tr_low = Tr_vals[Tr_vals <= Tr].max()
    Tr_high = Tr_vals[Tr_vals >= Tr].min()

    # 1. Exact grid point
    if Ts_low == Ts_high and Tr_low == Tr_high:
        return float(cop_table.loc[Ts_low, Tr_low])

    # 2. Exact Ts grid line -> interpolate only along Tr
    if Ts_low == Ts_high:
        Q1 = cop_table.loc[Ts_low, Tr_low]
        Q2 = cop_table.loc[Ts_low, Tr_high]

        if Tr_high == Tr_low:
            return float(Q1)

        return float(Q1 + (Q2 - Q1) * (Tr - Tr_low) / (Tr_high - Tr_low))

    # 3. Exact Tr grid line -> interpolate only along Ts
    if Tr_low == Tr_high:
        Q1 = cop_table.loc[Ts_low, Tr_low]
        Q2 = cop_table.loc[Ts_high, Tr_low]

        if Ts_high == Ts_low:
            return float(Q1)

        return float(Q1 + (Q2 - Q1) * (Ts - Ts_low) / (Ts_high - Ts_low))

    # 4. Full bilinear interpolation
    Q11 = cop_table.loc[Ts_low, Tr_low]
    Q12 = cop_table.loc[Ts_low, Tr_high]
    Q21 = cop_table.loc[Ts_high, Tr_low]
    Q22 = cop_table.loc[Ts_high, Tr_high]

    denom = (Ts_high - Ts_low) * (Tr_high - Tr_low)

    if denom == 0 or pd.isna(denom):
        return np.nan

    cop = (
        Q11 * (Ts_high - Ts) * (Tr_high - Tr) +
        Q21 * (Ts - Ts_low) * (Tr_high - Tr) +
        Q12 * (Ts_high - Ts) * (Tr - Tr_low) +
        Q22 * (Ts - Ts_low) * (Tr - Tr_low)
    ) / denom

    return float(cop)

OPERATING COSTS

In [ ]:
def calculate_operating_costs(
    df,
    avr_col,
    gas_col,
    demand_col,
    pump_col,
    cost_params,
    scenario_name="scenario",
    hp_col=None,
    Ts_col=None,
    Tr_col=None,
    cop_table=None,
    delta_yearly_heat_loss=0
):
    df = df.copy()
    df.index = pd.to_datetime(df.index)

    avr_heat_mw = df[avr_col].fillna(0)
    gas_heat_mw = df[gas_col].fillna(0)
    total_demand_mw = df[demand_col].fillna(0)
    pump_power_mw = df[pump_col].fillna(0)

    #HP blok
    if hp_col is not None and Ts_col is not None and Tr_col is not None and cop_table is not None:
        hp_heat_mw = df[hp_col].fillna(0)

        df["COP"] = [
            get_cop(Ts, Tr, cop_table)
            for Ts, Tr in zip(df[Ts_col], df[Tr_col])
        ]

        df["hp_electricity_mw"] = np.where(
            df["COP"] > 0,
            hp_heat_mw / df["COP"],
            0
        )

        hp_heat_mwh = hp_heat_mw.sum()
        hp_electricity_mwh = df["hp_electricity_mw"].sum()
        hp_electricity_cost_eur = hp_electricity_mwh * cost_params["electricity_cost_eur_per_mwh"]
    else:
        hp_heat_mw = pd.Series(0, index=df.index)
        df["COP"] = np.nan
        df["hp_electricity_mw"] = 0
        hp_heat_mwh = 0
        hp_electricity_mwh = 0
        hp_electricity_cost_eur = 0

    # Heat losses are already included inside each scenario's hourly source heat.
    # Do not subtract delta_yearly_heat_loss here, otherwise heat-loss savings are double counted.
    avr_heat_mwh = avr_heat_mw.sum()
    gas_heat_mwh = gas_heat_mw.sum()
    total_heat_delivered_mwh = total_demand_mw.sum()
    pump_energy_mwh = pump_power_mw.sum()

    avr_cost_eur = avr_heat_mwh * cost_params["avr_heat_cost_eur_per_mwh"]
    gas_cost_eur = gas_heat_mwh * cost_params["gas_heat_cost_eur_per_mwh"]
    pump_cost_eur = pump_energy_mwh * cost_params["electricity_cost_eur_per_mwh"]
    variable_om_cost_eur = total_heat_delivered_mwh * cost_params["variable_om_eur_per_mwh"]
    fixed_om_cost_eur = cost_params["fixed_om_eur_per_year"]

    total_cost_eur = (
        avr_cost_eur
        + gas_cost_eur
        + pump_cost_eur
        + hp_electricity_cost_eur
        + variable_om_cost_eur
        + fixed_om_cost_eur
    )

    specific_cost_eur_per_mwh = (
        total_cost_eur / total_heat_delivered_mwh
        if total_heat_delivered_mwh > 0 else float("nan")
    )
    
    summary = {
        "scenario": scenario_name,
        "avr_heat_mwh": avr_heat_mwh,
        "gas_heat_mwh": gas_heat_mwh,
        "hp_heat_mwh": hp_heat_mwh,
        "total_heat_delivered_mwh": total_heat_delivered_mwh,
        "pump_energy_mwh": pump_energy_mwh,
        "hp_electricity_mwh": hp_electricity_mwh,
        "avr_cost_eur": avr_cost_eur,
        "gas_cost_eur": gas_cost_eur,
        "pump_cost_eur": pump_cost_eur,
        "hp_electricity_cost_eur": hp_electricity_cost_eur,
        "variable_om_cost_eur": variable_om_cost_eur,
        "fixed_om_cost_eur": fixed_om_cost_eur,
        "total_cost_eur": total_cost_eur,
        "specific_cost_eur_per_mwh": specific_cost_eur_per_mwh,
    }

    monthly = pd.DataFrame(index=df.index)
    monthly["avr_cost_eur"] = avr_heat_mw * cost_params["avr_heat_cost_eur_per_mwh"]
    monthly["gas_cost_eur"] = gas_heat_mw * cost_params["gas_heat_cost_eur_per_mwh"]
    monthly["pump_cost_eur"] = pump_power_mw * cost_params["electricity_cost_eur_per_mwh"]
    monthly["hp_electricity_cost_eur"] = df["hp_electricity_mw"] * cost_params["electricity_cost_eur_per_mwh"]
    monthly["variable_om_eur"] = total_demand_mw * cost_params["variable_om_eur_per_mwh"]
    monthly = monthly.resample("MS").sum().sort_index()

    cost_breakdown = pd.DataFrame({
        "Cost component": [
            "AVR heat",
            "Gas heat",
            "Pumping electricity",
            "HP electricity",
            "Variable O&M",
            "Fixed O&M"
        ],
        "Cost (EUR/year)": [
            avr_cost_eur,
            gas_cost_eur,
            pump_cost_eur,
            hp_electricity_cost_eur,
            variable_om_cost_eur,
            fixed_om_cost_eur
        ],
        "Scenario": scenario_name
    })

    return summary, cost_breakdown, monthly

In [ ]:
def plot_costs(cost_breakdown, monthly_costs, scenario_name="Scenario"):
    plt.figure(figsize=(10, 6))

    plt.bar(
        cost_breakdown["Cost component"],
        cost_breakdown["Cost (EUR/year)"]
    )

    plt.title(f"Annual Operating Cost Breakdown ({scenario_name})")
    plt.ylabel("Cost (EUR/year)")
    plt.xlabel("Cost component")
    plt.xticks(rotation=20)
    plt.grid(axis="y")
    plt.tight_layout()
    save_current_plot_from_title()
    plt.show()

    #monthly cost breakdown
    monthly = monthly_costs.copy()
    monthly.index = pd.to_datetime(monthly.index)
    monthly = monthly.sort_index()

    if monthly.index.freq is None:
        monthly = monthly.resample("MS").sum()

    bar_width = 23  # days

    plt.figure(figsize=(12, 6))

    plt.bar(
        monthly.index,
        monthly["avr_cost_eur"],
        label="AVR heat",
        width=bar_width
    )

    plt.bar(
        monthly.index,
        monthly["gas_cost_eur"],
        bottom=monthly["avr_cost_eur"],
        label="Gas heat",
        width=bar_width
    )

    plt.bar(
        monthly.index,
        monthly["pump_cost_eur"],
        bottom=monthly["avr_cost_eur"] + monthly["gas_cost_eur"],
        label="Pumping",
        width=bar_width
    )

    plt.bar(
        monthly.index,
        monthly["hp_electricity_cost_eur"],
        bottom=monthly["avr_cost_eur"] + monthly["gas_cost_eur"] + monthly["pump_cost_eur"],
        label="HP electricity",
        width=bar_width
    )

    plt.bar(
        monthly.index,
        monthly["variable_om_eur"],
        bottom=monthly["avr_cost_eur"]
             + monthly["gas_cost_eur"]
             + monthly["pump_cost_eur"]
             + monthly["hp_electricity_cost_eur"],
        label="Variable O&M",
        width=bar_width
    )

    ax = plt.gca()
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

    plt.title(f"Monthly Operating Cost Breakdown ({scenario_name})")
    plt.ylabel("Cost (EUR/month)")
    plt.xlabel("Month")
    plt.legend()
    plt.grid(axis="y")
    plt.tight_layout()
    save_current_plot_from_title()
    plt.show()



def plot_all_scenarios_stacked(
    cost_dict,
    title="Annual Operating Cost Breakdown by Scenario"
):
    combined = []

    # Save scenario order exactly as typed in cost_dict
    scenario_order = list(cost_dict.keys())

    for scenario, df in cost_dict.items():
        temp = df.copy()
        temp["Scenario"] = scenario
        combined.append(temp)

    combined = pd.concat(combined, ignore_index=True)

    pivot = combined.pivot_table(
        index="Scenario",
        columns="Cost component",
        values="Cost (EUR/year)",
        aggfunc="sum",
        fill_value=0
    )

    # Fixed order of cost components
    preferred_components = [
        "AVR heat",
        "Gas heat",
        "Pumping",
        "HP electricity",
        "Variable O&M",
        "Fixed O&M",
        "Pumping electricity"
    ]

    existing_cols = [col for col in preferred_components if col in pivot.columns]
    remaining_cols = [col for col in pivot.columns if col not in existing_cols]
    pivot = pivot[existing_cols + remaining_cols]

    # Preserve the order in which scenarios were typed
    existing_rows = [row for row in scenario_order if row in pivot.index]
    remaining_rows = [row for row in pivot.index if row not in existing_rows]
    pivot = pivot.loc[existing_rows + remaining_rows]

    ax = pivot.plot(
        kind="bar",
        stacked=True,
        figsize=(12, 7),
        width=0.7
    )

    totals = pivot.sum(axis=1)
    for i, total in enumerate(totals):
        ax.text(i, total, f"{total:,.0f}", ha="center", va="bottom")

    plt.title(title)
    plt.ylabel("Cost (EUR/year)")
    plt.xlabel("Scenario")
    plt.xticks(rotation=20)
    plt.grid(axis="y")
    plt.legend(title="Cost component", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    save_current_plot_from_title()
    plt.show()

In [ ]:
summary_real, cost_real, monthly_real = calculate_operating_costs(
    df=data_hourly,
    avr_col="AVR_ARN_MW",
    gas_col="GAS_SCH_MW",
    demand_col="useful_demand_mw",
    pump_col="pumping energy",
    cost_params=cost_params,
    scenario_name="Real 2024",
    delta_yearly_heat_loss=0
)

plot_costs(cost_real, monthly_real, "real 2024")
summary_base, cost_base, monthly_base = calculate_operating_costs(
    df=data_hourly_base,
    avr_col="AVR_base",
    gas_col="GAS_base",
    demand_col="useful_demand_mw",
    pump_col="pump_energy_base_mw",
    cost_params=cost_params,
    scenario_name="Base",
    delta_yearly_heat_loss=0
)

plot_costs(cost_base, monthly_base, "base")

summary_perfect, cost_perfect, monthly_perfect = calculate_operating_costs(
    df=year_res,
    avr_col="q_source_mw",
    gas_col="unmet_mw",
    demand_col="demand_mw",
    pump_col="pumping energy",
    cost_params=cost_params,
    scenario_name="Perfect",
    delta_yearly_heat_loss=0
)

plot_costs(cost_perfect, monthly_perfect, "perfect")

summary_6h, cost_6h, monthly_6h = calculate_operating_costs(
    df=year_policy_res,
    avr_col="q_source_mw",
    gas_col="unmet_mw",
    demand_col="demand_mw",
    pump_col="pumping energy",
    cost_params=cost_params,
    scenario_name="6h",
    delta_yearly_heat_loss=0
)

plot_costs(cost_6h, monthly_6h, "6h" )

summary_6h_overdracht, cost_6h_overdracht, monthly_6h_overdracht = calculate_operating_costs(
    df=year_policy_res_with_overdracht,
    avr_col="q_source_mw",
    gas_col="unmet_mw",
    demand_col="demand_mw",
    pump_col="pumping energy",
    cost_params=cost_params,
    scenario_name="6h overdracht",
    delta_yearly_heat_loss=0
)
plot_costs(cost_6h_overdracht, monthly_6h_overdracht, "6h_overdracht")

plot_all_scenarios_stacked({
    "Real 2024": cost_real,
    "Base": cost_base,
    "Perfect": cost_perfect,
    "6h": cost_6h,
    "6h overdracht": cost_6h_overdracht
})

In [ ]:
comparison = pd.DataFrame({
    "scenario": ["real", "base", "perfect", "6h", "6h_overdracht"],
    "avr_cost_eur": [
        summary_real['avr_cost_eur'].sum(),
        summary_base['avr_cost_eur'].sum(),
        summary_perfect['avr_cost_eur'].sum(),
        summary_6h['avr_cost_eur'].sum(),
        summary_6h_overdracht['avr_cost_eur'].sum()
    ],
    "gas_cost_eur": [
        summary_real['gas_cost_eur'].sum(),
        summary_base['gas_cost_eur'].sum(),
        summary_perfect['gas_cost_eur'].sum(),
        summary_6h['gas_cost_eur'].sum(),
        summary_6h_overdracht['gas_cost_eur'].sum()
    ],
    "pump_cost_eur": [
        summary_real['pump_cost_eur'].sum(),
        summary_base['pump_cost_eur'].sum(),
        summary_perfect['pump_cost_eur'].sum(),
        summary_6h['pump_cost_eur'].sum(),
        summary_6h_overdracht['pump_cost_eur'].sum()
    ],
    "total_cost_eur": [
        summary_real['total_cost_eur'].sum(),
        summary_base['total_cost_eur'].sum(),
        summary_perfect['total_cost_eur'].sum(),
        summary_6h['total_cost_eur'].sum(),
        summary_6h_overdracht['total_cost_eur'].sum()
    ]
})

print(comparison)
comparison_rounded = comparison.copy()

cols = ["avr_cost_eur", "gas_cost_eur", "pump_cost_eur", "total_cost_eur"]

comparison_rounded[cols] = (
    comparison_rounded[cols]
    .div(10000)
    .round(0)
    .mul(10000)
)

print(comparison_rounded)

INTEGRATE HP AND RECALCUALTE COSTS

In [ ]:
def calculate_avr_hp_mw(avr_col, hp_thermal_power):
    hp_load = avr_col.clip(upper=hp_thermal_power)
    new_avr_col = avr_col - hp_load

    return new_avr_col, hp_load

In [ ]:
new_base_avr, base_hp = calculate_avr_hp_mw(data_hourly_base["AVR_base"],hp_thermal_power=10)
data_hourly_base["AVR_heat_MW_new"]=new_base_avr
data_hourly_base["hp_heat_MW"]=base_hp
data_hourly_base["AVR_ARN_TR_monthly"]=data_hourly["AVR_ARN_TR_monthly"]
data_hourly_base["supply_temp_base"]=maximum_daily_temp_filled.clip(lower=80.0001, upper=119.999)

new_perfect_avr, perfect_hp = calculate_avr_hp_mw(year_res["q_source_mw"],hp_thermal_power=10)
year_res["AVR_heat_MW_new"]=new_perfect_avr
year_res["hp_heat_MW"]= perfect_hp
year_res["Ts_day_C"]=year_res["Ts_day_C"].clip(lower=80.0001, upper=119.999)

new_real_avr, real_hp = calculate_avr_hp_mw(data_hourly["AVR_ARN_MW"],hp_thermal_power=10)
data_hourly["AVR_heat_MW_new"]= new_real_avr
data_hourly["hp_heat_MW"]= real_hp
data_hourly["supply_temp_real"]=data_hourly["AVR_ARN_TA"].clip(lower=80.0001, upper=119.999)
print(data_hourly.columns)

new_6h_avr, h6_hp = calculate_avr_hp_mw(year_policy_res["q_source_mw"],hp_thermal_power=10)
year_policy_res["AVR_heat_MW_new"]=new_6h_avr
year_policy_res["hp_heat_MW"]= h6_hp
year_policy_res["Ts_day_C"]=year_policy_res["Ts_day_C"].clip(lower=80.0001, upper=119.999)

new_6h_overdracht_avr, h6_overdracht_hp = calculate_avr_hp_mw(year_policy_res_with_overdracht["q_source_mw"],hp_thermal_power=10)
year_policy_res_with_overdracht["AVR_heat_MW_new"]=new_6h_overdracht_avr
year_policy_res_with_overdracht["hp_heat_MW"]= h6_overdracht_hp
year_policy_res_with_overdracht["Ts_day_C"]=year_policy_res_with_overdracht["Ts_day_C"].clip(lower=80.0001, upper=119.999)

In [ ]:
summary_hp_real_temp, cost_breakdown_hp_real_temp, monthly_hp_real_temp = calculate_operating_costs(
    df=data_hourly,
    avr_col="AVR_heat_MW_new",
    gas_col="GAS_SCH_MW",
    demand_col="useful_demand_mw",
    pump_col="pumping energy",
    cost_params=cost_params,
    scenario_name="real_with_hp",
    hp_col="hp_heat_MW",
    Ts_col="supply_temp_real",
    Tr_col="AVR_ARN_TR_monthly",
    cop_table=cop_table_geo_80,
    delta_yearly_heat_loss=0
)

summary_hp_base_temp, cost_breakdown_hp_base_temp, monthly_hp_base_temp = calculate_operating_costs(
    df=data_hourly_base,
    avr_col="AVR_heat_MW_new",
    gas_col="GAS_base",
    demand_col="useful_demand_mw",
    pump_col="pump_energy_base_mw",
    cost_params=cost_params,
    scenario_name="base_with_hp",
    hp_col="hp_heat_MW",
    Ts_col="supply_temp_base",
    Tr_col="AVR_ARN_TR_monthly",
    cop_table=cop_table_geo_80,
    delta_yearly_heat_loss=0
)

summary_hp_perfect_temp, cost_breakdown_hp_perfect_temp, monthly_hp_perfect_temp = calculate_operating_costs(
    df=year_res,
    avr_col="AVR_heat_MW_new",
    gas_col="unmet_mw",
    demand_col="demand_mw",
    pump_col="pumping energy",
    cost_params=cost_params,
    scenario_name="perfect_with_hp",
    hp_col="hp_heat_MW",
    Ts_col="Ts_day_C",
    Tr_col="Tret_C",
    cop_table=cop_table_geo_80,
    delta_yearly_heat_loss=0
)
summary_hp_6h_temp, cost_breakdown_hp_6h_temp, monthly_hp_6h_temp = calculate_operating_costs(
    df=year_policy_res,
    avr_col="AVR_heat_MW_new",
    gas_col="unmet_mw",
    demand_col="demand_mw",
    pump_col="pumping energy",
    cost_params=cost_params,
    scenario_name="6h_with_hp",
    hp_col="hp_heat_MW",
    Ts_col="Ts_day_C",
    Tr_col="Tret_C",
    cop_table=cop_table_geo_80,
    delta_yearly_heat_loss=0
)

summary_hp_6h_overdracht_temp, cost_breakdown_hp_6h_overdracht_temp, monthly_hp_6h_overdracht_temp = calculate_operating_costs(
    df=year_policy_res_with_overdracht,
    avr_col="AVR_heat_MW_new",
    gas_col="unmet_mw",
    demand_col="demand_mw",
    pump_col="pumping energy",
    cost_params=cost_params,
    scenario_name="6h_overdracht_with_hp",
    hp_col="hp_heat_MW",
    Ts_col="Ts_day_C",
    Tr_col="Tret_C",
    cop_table=cop_table_geo_80,
    delta_yearly_heat_loss=0
)

In [ ]:
comparison2 = pd.DataFrame({
    "scenario": ["hp_real_temp", "hp_base_temp", "hp_perfect_temp"],
    "avr_cost_eur": [
        summary_hp_real_temp["avr_cost_eur"],
        summary_hp_base_temp["avr_cost_eur"],
        summary_hp_perfect_temp["avr_cost_eur"]
    ],
    "gas_cost_eur": [
        summary_hp_real_temp["gas_cost_eur"],
        summary_hp_base_temp["gas_cost_eur"],
        summary_hp_perfect_temp["gas_cost_eur"]
    ],
    "pump_cost_eur": [
        summary_hp_real_temp["pump_cost_eur"],
        summary_hp_base_temp["pump_cost_eur"],
        summary_hp_perfect_temp["pump_cost_eur"]
    ],
    "hp_electricity_cost_eur": [
        summary_hp_real_temp["hp_electricity_cost_eur"],
        summary_hp_base_temp["hp_electricity_cost_eur"],
        summary_hp_perfect_temp["hp_electricity_cost_eur"]
    ],
    "variable_om_cost_eur": [
        summary_hp_real_temp["variable_om_cost_eur"],
        summary_hp_base_temp["variable_om_cost_eur"],
        summary_hp_perfect_temp["variable_om_cost_eur"]
    ],
    "fixed_om_cost_eur": [
        summary_hp_real_temp["fixed_om_cost_eur"],
        summary_hp_base_temp["fixed_om_cost_eur"],
        summary_hp_perfect_temp["fixed_om_cost_eur"]
    ],
    "total_cost_eur": [
        summary_hp_real_temp["total_cost_eur"],
        summary_hp_base_temp["total_cost_eur"],
        summary_hp_perfect_temp["total_cost_eur"]
    ]
})

comparison2["total_electricity_cost_eur"] = (
    comparison2["pump_cost_eur"] + comparison2["hp_electricity_cost_eur"]
)

print(comparison2.round(2))

In [ ]:
plot_costs(cost_breakdown_hp_real_temp, monthly_hp_real_temp, scenario_name="real with hp")
plot_costs(cost_breakdown_hp_base_temp, monthly_hp_base_temp, scenario_name="with_hp_base_temp")
plot_costs(cost_breakdown_hp_perfect_temp, monthly_hp_perfect_temp, scenario_name="with_hp_perfect_temp")

plot_all_scenarios_stacked({
    "Real 2024_hp": cost_breakdown_hp_real_temp,
    "Base_hp": cost_breakdown_hp_base_temp,
    "Perfect_hp": cost_breakdown_hp_perfect_temp,
    "6h_hp": cost_breakdown_hp_6h_temp,
    "6h_overdracht_hp": cost_breakdown_hp_6h_overdracht_temp
})


In [ ]:
print(summary_hp_base_temp["total_cost_eur"]-summary_hp_perfect_temp["total_cost_eur"])


In [ ]:
comparison_hp = pd.DataFrame({
    "metric": summary_hp_base_temp.keys(),
    "base_temp": summary_hp_base_temp.values(),
    "perfect_temp": summary_hp_perfect_temp.values(),
    "6h_temp": summary_hp_6h_temp.values(),
})

print(comparison_hp)


In [ ]:
#make a table with all scenarios, and the costs of gas, avr heat, pumping, hp electricity, total cost
final_comparison = pd.DataFrame({
    "scenario": [
        "real",
        "base",
        "perfect",
        "6h",
        "6h_overdracht",
        "real_with_hp",
        "base_with_hp",
        "perfect_with_hp",
        "6h_with_hp",
        "6h_overdracht_with_hp"
    ],
    "avr_cost_eur": [
        summary_real['avr_cost_eur'],
        summary_base['avr_cost_eur'],
        summary_perfect['avr_cost_eur'],
        summary_6h['avr_cost_eur'],
        summary_6h_overdracht['avr_cost_eur'],
        summary_hp_real_temp['avr_cost_eur'],
        summary_hp_base_temp['avr_cost_eur'],
        summary_hp_perfect_temp['avr_cost_eur'],
        summary_hp_6h_temp['avr_cost_eur'],
        summary_hp_6h_overdracht_temp['avr_cost_eur']
    ],
    "gas_cost_eur": [
        summary_real['gas_cost_eur'],
        summary_base['gas_cost_eur'],
        summary_perfect['gas_cost_eur'],
        summary_6h['gas_cost_eur'],
        summary_6h_overdracht['gas_cost_eur'],
        summary_hp_real_temp['gas_cost_eur'],
        summary_hp_base_temp['gas_cost_eur'],
        summary_hp_perfect_temp['gas_cost_eur'],
        summary_hp_6h_temp['gas_cost_eur'],
        summary_hp_6h_overdracht_temp['gas_cost_eur']
    ],
    "pump_cost_eur": [
        summary_real['pump_cost_eur'],
        summary_base['pump_cost_eur'],
        summary_perfect['pump_cost_eur'],
        summary_6h['pump_cost_eur'],
        summary_6h_overdracht['pump_cost_eur'],
        summary_hp_real_temp['pump_cost_eur'],
        summary_hp_base_temp['pump_cost_eur'],
        summary_hp_perfect_temp['pump_cost_eur'],
        summary_hp_6h_temp['pump_cost_eur'],
        summary_hp_6h_overdracht_temp['pump_cost_eur']
    ],
    "hp_electricity_cost_eur": [
        summary_real['hp_electricity_cost_eur'],
        summary_base['hp_electricity_cost_eur'],
        summary_perfect['hp_electricity_cost_eur'],
        summary_6h['hp_electricity_cost_eur'],
        summary_6h_overdracht['hp_electricity_cost_eur'],
        summary_hp_real_temp['hp_electricity_cost_eur'],
        summary_hp_base_temp['hp_electricity_cost_eur'],
        summary_hp_perfect_temp['hp_electricity_cost_eur'],
        summary_hp_6h_temp['hp_electricity_cost_eur'],
        summary_hp_6h_overdracht_temp['hp_electricity_cost_eur']
    ],
    "total_cost_eur": [
        summary_real['total_cost_eur'],
        summary_base['total_cost_eur'],
        summary_perfect['total_cost_eur'],
        summary_6h['total_cost_eur'],
        summary_6h_overdracht['total_cost_eur'],
        summary_hp_real_temp['total_cost_eur'],
        summary_hp_base_temp['total_cost_eur'],
        summary_hp_perfect_temp['total_cost_eur'],
        summary_hp_6h_temp['total_cost_eur'],
        summary_hp_6h_overdracht_temp['total_cost_eur']
    ]

})
#round to euros
final_comparison[["avr_cost_eur", "gas_cost_eur", "pump_cost_eur", "hp_electricity_cost_eur", "total_cost_eur"]] = (
    final_comparison[["avr_cost_eur", "gas_cost_eur", "pump_cost_eur", "hp_electricity_cost_eur", "total_cost_eur"]]
    .round(0)
)
print(final_comparison)

CAPEX & DELTA LCOH CALCULATION
This block calculates the CAPEX of the used buffer, and the change in the LCOH that it creates

In [ ]:
capex_buffer = price_buffer * buffer_volume_m3
print(capex_buffer)
print(buffer_volume_m3)


# --- Inputs ---
capex = capex_buffer  # € (total investment)
delta_opex =  summary_perfect["total_cost_eur"] - summary_base["total_cost_eur"]
heat_delivered = summary_base["total_heat_delivered_mwh"]  # MWh/year

r = 0.06   # discount rate
n = 40     # lifetime (years)

# --- Capital Recovery Factor ---
crf = (r * (1 + r)**n) / ((1 + r)**n - 1)

# --- Annualized CAPEX ---
annualized_capex = capex * crf

# --- ΔLCOH ---
delta_lcoh = (annualized_capex + delta_opex) / heat_delivered

print(f"ΔLCOH: {delta_lcoh:.2f} €/MWh")

delta_opex_real = summary_perfect["total_cost_eur"] - summary_real["total_cost_eur"]
delta_lcoh_real = (annualized_capex + delta_opex_real) / heat_delivered
print(f"ΔLCOH vs Real: {delta_lcoh_real:.2f} €/MWh")

In [ ]:

capex_buffer = price_buffer * buffer_volume_m3
print(capex_buffer)
print(buffer_volume_m3)


# --- Inputs ---
capex = capex_buffer  # € (total investment)
delta_opex_hp =  summary_hp_perfect_temp["total_cost_eur"] - summary_hp_base_temp["total_cost_eur"]
heat_delivered_hp = summary_hp_base_temp["total_heat_delivered_mwh"]  # MWh/year

r = 0.06   # discount rate
n = 40     # lifetime (years)

# --- Capital Recovery Factor ---
crf = (r * (1 + r)**n) / ((1 + r)**n - 1)

# --- Annualized CAPEX ---
annualized_capex = capex * crf

# --- ΔLCOH ---
delta_lcoh_hp = (annualized_capex + delta_opex_hp) / heat_delivered_hp

print(f"ΔLCOH_HPS: {delta_lcoh_hp:.2f} €/MWh")

delta_opex_real = summary_hp_perfect_temp["total_cost_eur"] - summary_hp_real_temp["total_cost_eur"]
delta_lcoh_real = (annualized_capex + delta_opex_real) / heat_delivered_hp
print(f"ΔLCOH vs Real: {delta_lcoh_real:.2f} €/MWh")


EMISSION CALCULATIONS


In [ ]:
import matplotlib.ticker as mticker

def plot_all_scenarios_stacked_label(
    cost_dict,
    title="Annual Operating Cost Breakdown by Scenario",
    min_label_value=120_000
):
    combined = []

    # Save scenario order exactly as typed in cost_dict
    scenario_order = list(cost_dict.keys())

    for scenario, df in cost_dict.items():
        temp = df.copy()
        temp["Scenario"] = scenario
        combined.append(temp)

    combined = pd.concat(combined, ignore_index=True)

    pivot = combined.pivot_table(
        index="Scenario",
        columns="Cost component",
        values="Cost (EUR/year)",
        aggfunc="sum",
        fill_value=0
    )

    # Fixed order of cost components
    preferred_components = [
        "AVR heat",
        "Gas heat",
        "HP electricity",
        "Variable O&M",
        "Fixed O&M",
        "Pumping electricity"
    ]

    existing_cols = [col for col in preferred_components if col in pivot.columns]
    remaining_cols = [col for col in pivot.columns if col not in existing_cols]
    pivot = pivot[existing_cols + remaining_cols]

    # Preserve scenario order
    existing_rows = [row for row in scenario_order if row in pivot.index]
    remaining_rows = [row for row in pivot.index if row not in existing_rows]
    pivot = pivot.loc[existing_rows + remaining_rows]

    ax = pivot.plot(
        kind="bar",
        stacked=True,
        figsize=(12, 7),
        width=0.7
    )

    # Remove scientific notation from y-axis
    ax.ticklabel_format(style="plain", axis="y")
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, pos: f"{x:,.0f}")
    )

    # Add labels inside bar segments
    for container in ax.containers:
        component_name = container.get_label()

        labels = []
        for bar in container:
            value = bar.get_height()

            # Always show gas heat labels, even if small
            if component_name == "Gas heat" and value > 0:
                labels.append(f"{value:,.0f}")

            # Show other components only if large enough
            elif value >= min_label_value:
                labels.append(f"{value:,.0f}")

            else:
                labels.append("")

        ax.bar_label(
            container,
            labels=labels,
            label_type="center",
            fontsize=8
        )

    # Add total labels on top of each bar
    totals = pivot.sum(axis=1)
    for i, total in enumerate(totals):
        ax.text(
            i,
            total + totals.max() * 0.01,
            f"{total:,.0f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

    
    plt.ylabel("Cost [€/year]")
    plt.xlabel("Scenario")
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.4)

    plt.legend(
        title="Cost component",
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )
    plot_title = title
    plt.tight_layout()
    save_current_plot_from_title(plot_title)
    plt.show()

plot_all_scenarios_stacked_label({
    "Real 2024": cost_real,
    "Base": cost_base,
    "Perfect": cost_perfect,
    "Real 2024_hp": cost_breakdown_hp_real_temp,
    "Base_hp": cost_breakdown_hp_base_temp,
    "Perfect_hp": cost_breakdown_hp_perfect_temp
})

In [ ]:
co2_real    =   summary_real["gas_heat_mwh"]*co2_gas_mwh +summary_real["avr_heat_mwh"]*co2_avr_mwh +summary_real["pump_energy_mwh"]*co2_electricity_mwh
co2_base    =   summary_base["gas_heat_mwh"]*co2_gas_mwh +summary_base["avr_heat_mwh"]*co2_avr_mwh +summary_base["pump_energy_mwh"]*co2_electricity_mwh
co2_perfect =   summary_perfect["gas_heat_mwh"]*co2_gas_mwh +summary_perfect["avr_heat_mwh"]*co2_avr_mwh +summary_perfect["pump_energy_mwh"]*co2_electricity_mwh
co2_6h      =   summary_6h["gas_heat_mwh"]*co2_gas_mwh +summary_6h["avr_heat_mwh"]*co2_avr_mwh +summary_6h["pump_energy_mwh"]*co2_electricity_mwh
co2_6h_overdracht = summary_6h_overdracht["gas_heat_mwh"]*co2_gas_mwh +summary_6h_overdracht["avr_heat_mwh"]*co2_avr_mwh +summary_6h_overdracht["pump_energy_mwh"]*co2_electricity_mwh

co2_real_hp    =   summary_hp_real_temp["gas_heat_mwh"]*co2_gas_mwh + summary_hp_real_temp["avr_heat_mwh"]*co2_avr_mwh + summary_hp_real_temp["pump_energy_mwh"]*co2_electricity_mwh + summary_hp_real_temp["hp_electricity_mwh"]*co2_electricity_mwh
co2_base_hp    =   summary_hp_base_temp["gas_heat_mwh"]*co2_gas_mwh + summary_hp_base_temp["avr_heat_mwh"]*co2_avr_mwh + summary_hp_base_temp["pump_energy_mwh"]*co2_electricity_mwh + summary_hp_base_temp["hp_electricity_mwh"]*co2_electricity_mwh
co2_perfect_hp =   summary_hp_perfect_temp["gas_heat_mwh"]*co2_gas_mwh + summary_hp_perfect_temp["avr_heat_mwh"]*co2_avr_mwh + summary_hp_perfect_temp["pump_energy_mwh"]*co2_electricity_mwh + summary_hp_perfect_temp["hp_electricity_mwh"]*co2_electricity_mwh
co2_6h_hp      =   summary_hp_6h_temp["gas_heat_mwh"]*co2_gas_mwh + summary_hp_6h_temp["avr_heat_mwh"]*co2_avr_mwh + summary_hp_6h_temp["pump_energy_mwh"]*co2_electricity_mwh + summary_hp_6h_temp["hp_electricity_mwh"]*co2_electricity_mwh
co2_6h_overdracht_hp = summary_hp_6h_overdracht_temp["gas_heat_mwh"]*co2_gas_mwh + summary_hp_6h_overdracht_temp["avr_heat_mwh"]*co2_avr_mwh + summary_hp_6h_overdracht_temp["pump_energy_mwh"]*co2_electricity_mwh + summary_hp_6h_overdracht_temp["hp_electricity_mwh"]*co2_electricity_mwh


print(co2_real)
print(co2_base)
print(co2_perfect)
print(co2_6h)
print(co2_6h_overdracht)

print(co2_real_hp)
print(co2_base_hp)
print(co2_perfect_hp)
print(co2_6h_hp)
print(co2_6h_overdracht_hp)

In [ ]:
co2_df = pd.DataFrame({
    "Scenario": [
        "Real",
        "Base",
        "Perfect",
        "6h",
        "6h_overdracht",
        "Real + HP",
        "Base + HP",
        "Perfect + HP",
        "6h + HP",
        "6h_overdracht + HP"
    ],

    "Gas": [
        summary_real["gas_heat_mwh"] * co2_gas_mwh,
        summary_base["gas_heat_mwh"] * co2_gas_mwh,
        summary_perfect["gas_heat_mwh"] * co2_gas_mwh,
        summary_6h["gas_heat_mwh"] * co2_gas_mwh,
        summary_6h_overdracht["gas_heat_mwh"] * co2_gas_mwh,

        summary_hp_real_temp["gas_heat_mwh"] * co2_gas_mwh,
        summary_hp_base_temp["gas_heat_mwh"] * co2_gas_mwh,
        summary_hp_perfect_temp["gas_heat_mwh"] * co2_gas_mwh,
        summary_hp_6h_temp["gas_heat_mwh"] * co2_gas_mwh,
        summary_hp_6h_overdracht_temp["gas_heat_mwh"] * co2_gas_mwh
    ],

    "AVR": [
        summary_real["avr_heat_mwh"] * co2_avr_mwh,
        summary_base["avr_heat_mwh"] * co2_avr_mwh,
        summary_perfect["avr_heat_mwh"] * co2_avr_mwh,
        summary_6h["avr_heat_mwh"] * co2_avr_mwh,
        summary_6h_overdracht["avr_heat_mwh"] * co2_avr_mwh,

        summary_hp_real_temp["avr_heat_mwh"] * co2_avr_mwh,
        summary_hp_base_temp["avr_heat_mwh"] * co2_avr_mwh,
        summary_hp_perfect_temp["avr_heat_mwh"] * co2_avr_mwh,
        summary_hp_6h_temp["avr_heat_mwh"] * co2_avr_mwh,
        summary_hp_6h_overdracht_temp["avr_heat_mwh"] * co2_avr_mwh
    ],

    "Pump": [
        summary_real["pump_energy_mwh"] * co2_electricity_mwh,
        summary_base["pump_energy_mwh"] * co2_electricity_mwh,
        summary_perfect["pump_energy_mwh"] * co2_electricity_mwh,
        summary_6h["pump_energy_mwh"] * co2_electricity_mwh,
        summary_6h_overdracht["pump_energy_mwh"] * co2_electricity_mwh,

        summary_hp_real_temp["pump_energy_mwh"] * co2_electricity_mwh,
        summary_hp_base_temp["pump_energy_mwh"] * co2_electricity_mwh,
        summary_hp_perfect_temp["pump_energy_mwh"] * co2_electricity_mwh,
        summary_hp_6h_temp["pump_energy_mwh"] * co2_electricity_mwh,
        summary_hp_6h_overdracht_temp["pump_energy_mwh"] * co2_electricity_mwh
    ],

    "HP electricity": [
        0,
        0,
        0,
        0,
        0,

        summary_hp_real_temp["hp_electricity_mwh"] * co2_electricity_mwh,
        summary_hp_base_temp["hp_electricity_mwh"] * co2_electricity_mwh,
        summary_hp_perfect_temp["hp_electricity_mwh"] * co2_electricity_mwh,
        summary_hp_6h_temp["hp_electricity_mwh"] * co2_electricity_mwh,
        summary_hp_6h_overdracht_temp["hp_electricity_mwh"] * co2_electricity_mwh
    ]
})

co2_df.set_index("Scenario", inplace=True)

# Split scenarios
no_hp_scenarios = [
    "Real",
    "Base",
    "Perfect",
    "6h",
    "6h_overdracht"
]

hp_scenarios = [
    "Real + HP",
    "Base + HP",
    "Perfect + HP",
    "6h + HP",
    "6h_overdracht + HP"
]

co2_no_hp = co2_df.loc[no_hp_scenarios]
co2_hp = co2_df.loc[hp_scenarios]

# Optional: remove HP electricity column from no-HP chart because it is all zero
co2_no_hp_plot = co2_no_hp.drop(columns=["HP electricity"], errors="ignore")
co2_hp_plot = co2_hp.copy()

# Create two plots below each other
fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(14, 10),
    sharey=True
)

# =========================
# Plot 1: without HP
# =========================
ax1 = co2_no_hp_plot.plot(
    kind="bar",
    stacked=True,
    ax=axes[0],
    width=0.75
)

totals_no_hp = co2_no_hp_plot.sum(axis=1)

for i, total in enumerate(totals_no_hp):
    ax1.text(
        i,
        total,
        f"{total:,.0f}",
        ha="center",
        va="bottom"
    )

ax1.set_title("CO₂ Emissions per Scenario — without HP")
ax1.set_ylabel("CO₂ emissions kg")
ax1.set_xlabel("")
ax1.grid(axis="y")
ax1.tick_params(axis="x", rotation=0)
ax1.legend(title="Source", bbox_to_anchor=(1.02, 1), loc="upper left")

# =========================
# Plot 2: with HP
# =========================
ax2 = co2_hp_plot.plot(
    kind="bar",
    stacked=True,
    ax=axes[1],
    width=0.75
)

totals_hp = co2_hp_plot.sum(axis=1)

for i, total in enumerate(totals_hp):
    ax2.text(
        i,
        total,
        f"{total:,.0f}",
        ha="center",
        va="bottom"
    )

ax2.set_title("CO₂ Emissions per Scenario — with HP")
ax2.set_ylabel("CO₂ emissions kg")
ax2.set_xlabel("Scenario")
ax2.grid(axis="y")
ax2.tick_params(axis="x", rotation=0)
ax2.legend(title="Source", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
save_current_plot_from_title()
plt.show()

In [ ]:
year_res["COP"] = year_res.apply(
    lambda row: get_cop(
        Ts=row["Ts_day_C"],
        Tr=row["Tret_C"],
        cop_table=cop_table_geo_80
    ),
    axis=1
)

data_hourly_base["COP"] = data_hourly_base.apply(
    lambda row: get_cop(
        Ts=row["supply_temp_base"],
        Tr=row["AVR_ARN_TR_monthly"],
        cop_table=cop_table_geo_80
    ),
    axis=1
)

plt.figure(figsize=(12, 6))
plt.grid(True)
year_res["COP"].plot(label="With TES")
data_hourly_base["COP"].plot(label="Base case", alpha=0.7)

plt.xlabel("Time")
plt.ylabel("COP (-)")
plt.title("COP comparison")
plt.legend()
plt.grid(True)
save_current_plot_from_title()
plt.show()


In [ ]:


# --- Store all results ---
results = []

# --- Loop over years ---
for year, ef in emission_factors.items():
    
    co2_gas_mwh = ef["gas_GJ"] * 3.6
    co2_avr_mwh = ef["avr_GJ"] * 3.6
    co2_electricity_mwh = ef["elec_kWh"] * 1000

    scenarios = {
        "Real": summary_real,
        "Base": summary_base,
        "Perfect": summary_perfect,
        "Real + HP": summary_hp_real_temp,
        "Base + HP": summary_hp_base_temp,
        "Perfect + HP": summary_hp_perfect_temp,
    }

    for name, s in scenarios.items():
        
        gas = s["gas_heat_mwh"] * co2_gas_mwh
        avr = s["avr_heat_mwh"] * co2_avr_mwh
        pump = s["pump_energy_mwh"] * co2_electricity_mwh
        
        hp = 0
        if "hp_electricity_mwh" in s:
            hp = s["hp_electricity_mwh"] * co2_electricity_mwh

        total_emissions_kg = gas + avr + pump + hp

        # MWh heat delivered -> GJ heat delivered
        heat_delivered_gj = s["total_heat_delivered_mwh"] * 3.6

        kg_co2_per_gj_heat_delivered = (
            total_emissions_kg / heat_delivered_gj
            if heat_delivered_gj > 0
            else np.nan
        )

        results.append({
            "Year": year,
            "Scenario": name,
            "Gas": gas,
            "AVR": avr,
            "Pump electricity": pump,
            "HP electricity": hp,
            "Total emissions kg": total_emissions_kg,
            "Heat delivered GJ": heat_delivered_gj,
            "kg CO2 per GJ heat delivered": kg_co2_per_gj_heat_delivered,
        })

# --- Create dataframe ---
co2_df = pd.DataFrame(results)

# Keep intensity values separately before plotting
intensity_df = co2_df.set_index(["Year", "Scenario"])[
    "kg CO2 per GJ heat delivered"
]

# Only emission components for stacked bar plot
plot_df = co2_df.set_index(["Year", "Scenario"])[
    ["Gas", "AVR", "Pump electricity", "HP electricity"]
]

# --- Plot per year ---
years = plot_df.index.get_level_values(0).unique()

fig, axes = plt.subplots(len(years), 1, figsize=(12, 12), sharex=True)

# make sure axes is iterable if there is only one year
if len(years) == 1:
    axes = [axes]

for i, year in enumerate(years):
    year_df = plot_df.loc[year]
    year_intensity = intensity_df.loc[year]
    
    ax = year_df.plot(
        kind="bar",
        stacked=True,
        ax=axes[i]
    )
    
    totals = year_df.sum(axis=1)
    offset = 0.02 * totals.max()

    for j, scenario in enumerate(year_df.index):
        total = totals.loc[scenario]
        intensity = year_intensity.loc[scenario]

        # 1) totaal boven de bar
        ax.text(
            j,
            total + offset,
            f"{total:,.0f}",
            ha="center",
            va="bottom",
            fontsize=10
        )

        # 2) kg/GJ in het midden van de totale bar
        ax.text(
            j,
            total * 0.55,
            f"{intensity:.1f} kg/GJ",
            ha="center",
            va="center",
            fontsize=10,
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=1.5)
        )

    ax.set_title(f"CO₂ Emissions in {year}")
    ax.set_ylabel("kg CO₂")
    ax.grid(True)
    ax.set_ylim(0, totals.max() * 1.12)

plot_title = "CO₂ Emissions per Scenario and Year"
plt.xticks(rotation=0)
plt.tight_layout()
save_current_plot_from_title(plot_title)
plt.show()

In [ ]:


# Scenario and year order
scenario_order = [
    "Real",
    "Base",
    "Perfect",
    "Real + HP",
    "Base + HP",
    "Perfect + HP"
]

year_order = sorted(co2_df["Year"].unique())

# Pivot data for plotting
intensity_pivot = co2_df.pivot(
    index="Year",
    columns="Scenario",
    values="kg CO2 per GJ heat delivered"
).reindex(index=year_order, columns=scenario_order)

# Create figure
fig, ax = plt.subplots(figsize=(10, 6))

# Manual label offsets to avoid overlap
# Values are vertical offsets in kg CO2/GJ
label_offsets = {
    "Real": 0.30,
    "Base": -0.35,
    "Perfect": 0.25,
    "Real + HP": 0.22,
    "Base + HP": 0.22,
    "Perfect + HP": 0.22
}

# Plot each scenario
for scenario in scenario_order:
    ax.plot(
        intensity_pivot.index,
        intensity_pivot[scenario],
        marker="o",
        linewidth=2,
        label=scenario
    )

    # Add value labels to each point with scenario-specific offset
    for year, value in intensity_pivot[scenario].items():
        offset = label_offsets.get(scenario, 0.2)

        ax.text(
            year,
            value + offset,
            f"{value:.1f}",
            ha="center",
            va="center",
            fontsize=11,
            bbox=dict(
                facecolor="white",
                edgecolor="none",
                alpha=0.8,
                pad=1.2
            )
        )

# Titles and labels

ax.set_xlabel("Year", fontsize=14)
ax.set_ylabel("kg CO₂/GJ heat delivered", fontsize=14)

# Axis settings
ax.set_xticks(year_order)

# Add extra space at the top and bottom for labels
y_min = intensity_pivot.min().min()
y_max = intensity_pivot.max().max()
ax.set_ylim(y_min - 0.8, y_max + 0.8)

ax.grid(True, alpha=0.3)

# Legend
ax.legend(
    title="Scenario",
    fontsize=12,
    title_fontsize=12,
    loc="lower left"
)

plot_title = "CO2 emissions intensity per scenario line chart"
plt.tight_layout()
save_current_plot_from_title(plot_title)
plt.show()

PIE CHARTS


In [ ]:

scenario_order = [
    "Real",
    "Base",
    "Perfect",
    "Real + HP",
    "Base + HP",
    "Perfect + HP"
]

year_order = sorted(co2_df["Year"].unique())

component_cols = [
    "Gas",
    "AVR",
    "Pump electricity",
    "HP electricity"
]


intensity_pivot = co2_df.pivot(
    index="Year",
    columns="Scenario",
    values="kg CO2 per GJ heat delivered"
).reindex(index=year_order, columns=scenario_order)


pie_year = 2030

pie_df = (
    co2_df[co2_df["Year"] == pie_year]
    .set_index("Scenario")[component_cols]
    .reindex(scenario_order)
)


fig = plt.figure(figsize=(17, 8.8))

gs = fig.add_gridspec(
    nrows=3,
    ncols=3,
    width_ratios=[2.9, 1.05, 1.05],
    wspace=0.18,
    hspace=0.34
)

# Left: line chart spanning all rows
ax_line = fig.add_subplot(gs[:, 0])

# Right: 6 pie charts, 3 rows x 2 columns
pie_axes = [
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[0, 2]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[1, 2]),
    fig.add_subplot(gs[2, 1]),
    fig.add_subplot(gs[2, 2]),
]

# ------------------------------------------------------------
# Plot line chart
# ------------------------------------------------------------
for scenario in scenario_order:
    ax_line.plot(
        intensity_pivot.index,
        intensity_pivot[scenario],
        marker="o",
        linewidth=2,
        label=scenario
    )

ax_line.set_xlabel("Year", fontsize=14)
ax_line.set_ylabel("kg CO₂/GJ heat delivered", fontsize=14)

ax_line.tick_params(axis="x", labelsize=14)
ax_line.tick_params(axis="y", labelsize=14)

ax_line.set_xticks(year_order)

y_min = intensity_pivot.min().min()
y_max = intensity_pivot.max().max()
ax_line.set_ylim(y_min - 0.8, y_max + 0.8)

ax_line.grid(True, alpha=0.3)

ax_line.legend(
    title="Scenario",
    fontsize=12,
    title_fontsize=12,
    loc="lower left"
)

# ------------------------------------------------------------
# Add non-overlapping value labels to line chart
# ------------------------------------------------------------
def add_non_overlapping_line_labels(ax, pivot, scenarios, min_gap=1.5):
    """
    Adds value labels to a line chart while avoiding vertical overlap
    between labels at the same x-position.
    """

    y_low, y_high = ax.get_ylim()
    margin = 0.15

    for year in pivot.index:
        labels = []

        for scenario in scenarios:
            value = pivot.loc[year, scenario]

            if np.isfinite(value):
                labels.append({
                    "scenario": scenario,
                    "value": value,
                    "y_text": value
                })

        # Sort labels from low to high
        labels = sorted(labels, key=lambda x: x["y_text"])

        # Push labels upward if they are too close
        for i in range(1, len(labels)):
            if labels[i]["y_text"] - labels[i - 1]["y_text"] < min_gap:
                labels[i]["y_text"] = labels[i - 1]["y_text"] + min_gap

        # If the highest label goes outside the plot, shift all labels down
        overflow = labels[-1]["y_text"] - (y_high - margin)
        if overflow > 0:
            for label in labels:
                label["y_text"] -= overflow

        # If the lowest label goes outside the plot, shift all labels up
        underflow = (y_low + margin) - labels[0]["y_text"]
        if underflow > 0:
            for label in labels:
                label["y_text"] += underflow

        # Add the labels
        for label in labels:
            value = label["value"]
            y_text = label["y_text"]

            if abs(y_text - value) > 0.05:
                arrowprops = dict(
                    arrowstyle="-",
                    lw=0.6,
                    color="black",
                    shrinkA=0,
                    shrinkB=3
                )
            else:
                arrowprops = None

            ax.annotate(
                f"{value:.1f}",
                xy=(year, value),
                xytext=(year, y_text),
                textcoords="data",
                ha="center",
                va="center",
                fontsize=12,
                bbox=dict(
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.85,
                    pad=1.2
                ),
                arrowprops=arrowprops
            )

add_non_overlapping_line_labels(
    ax=ax_line,
    pivot=intensity_pivot,
    scenarios=scenario_order,
    min_gap=0.42
)

# ------------------------------------------------------------
# Pie chart settings
# ------------------------------------------------------------
source_colors = {
    "Gas": "#1f77b4",
    "AVR": "#ff7f0e",
    "Pump electricity": "#2ca02c",
    "HP electricity": "#d62728"
}

pie_colors = [source_colors[col] for col in component_cols]

# ------------------------------------------------------------
# Helper function: external percentage labels with leader lines
# ------------------------------------------------------------
def add_pie_percentage_labels(ax, wedges, values, fontsize=12):
    total = np.sum(values)

    for wedge, value in zip(wedges, values):
        if value <= 0 or total <= 0:
            continue

        pct = value / total * 100

        angle = (wedge.theta2 + wedge.theta1) / 2
        angle_rad = np.deg2rad(angle)

        # Point on the pie edge
        x = np.cos(angle_rad)
        y = np.sin(angle_rad)

        # Label position outside the pie
        label_radius = 1.18
        label_x = label_radius * x
        label_y = label_radius * y

        # Push very top labels sideways/down slightly to avoid subplot titles
        if label_y > 0.85:
            label_y -= 0.10
            label_x += 0.10 if label_x >= 0 else -0.10

        # Push very bottom labels slightly upward to avoid neighbouring charts
        if label_y < -0.90:
            label_y += 0.08

        ha = "left" if label_x >= 0 else "right"

        ax.annotate(
            f"{pct:.1f}%",
            xy=(0.92 * x, 0.92 * y),
            xytext=(label_x, label_y),
            ha=ha,
            va="center",
            fontsize=fontsize,
            arrowprops=dict(
                arrowstyle="-",
                lw=0.8,
                shrinkA=0,
                shrinkB=0,
                connectionstyle="arc3,rad=0.15"
            )
        )

# ------------------------------------------------------------
# Plot 6 pie charts
# ------------------------------------------------------------
for ax, scenario in zip(pie_axes, scenario_order):
    values = pie_df.loc[scenario, component_cols].values.astype(float)

    # Remove zero values from the pie itself, but keep source categories in legend
    nonzero_mask = values > 0
    values_nonzero = values[nonzero_mask]
    colors_nonzero = np.array(pie_colors)[nonzero_mask]

    wedges, _ = ax.pie(
        values_nonzero,
        colors=colors_nonzero,
        startangle=90,
        counterclock=False,
        radius=0.92,
        wedgeprops={
            "edgecolor": "white",
            "linewidth": 0.8
        }
    )

    add_pie_percentage_labels(
        ax=ax,
        wedges=wedges,
        values=values_nonzero,
        fontsize=12
    )

    ax.set_title(scenario, fontsize=12, pad=16)
    ax.set_aspect("equal")

    # Extra room for outside labels
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.25, 1.25)

# ------------------------------------------------------------
# Aligned titles for left and right plot sections
# ------------------------------------------------------------
fig.text(
    0.255,
    0.965,
    "CO₂ emissions intensity per scenario",
    ha="center",
    va="top",
    fontsize=12
)

fig.text(
    0.745,
    0.965,
    "Emission share per source for 2030",
    ha="center",
    va="top",
    fontsize=12
)

# ------------------------------------------------------------
# Shared legend for emission sources
# ------------------------------------------------------------
legend_handles = [
    Patch(facecolor=source_colors[col], edgecolor="white", label=col)
    for col in component_cols
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=12,
    bbox_to_anchor=(0.68, 0.015)
)

# ------------------------------------------------------------
# Layout and save
# ------------------------------------------------------------
plt.tight_layout(rect=[0, 0.065, 1, 0.93])

save_current_plot_from_title("CO2 line chart with fixed 2030 emission source pie charts")

plt.show()

In [ ]:
# Reset index
co2_table = co2_df.reset_index()

# Add total column
co2_table["Total"] = (
    co2_table["Gas"] +
    co2_table["AVR"] +
    co2_table["Pump electricity"] +
    co2_table["HP electricity"]
)

# Round all emission values
cols_to_round = ["Gas", "AVR", "Pump electricity", "HP electricity", "Total"]
co2_table[cols_to_round] = co2_table[cols_to_round].round(0).astype(int)

tables_per_year = {}

for year in co2_table["Year"].unique():
    
    df_year = co2_table[co2_table["Year"] == year].copy()
    
    # Pivot: emissions as rows, scenarios as columns
    table = df_year.set_index("Scenario")[cols_to_round].T
    
    tables_per_year[year] = table
    
    print(f"\n=== CO₂ Emissions in {year} ===")
    print(table)

In [ ]:

def plot_daily_supply_temp_difference_heatmap(
    df1, df2, col1, col2, 
    label1="Scenario 1", 
    label2="Scenario 2",
    vmin=-20,
    vmax=0
):
    # daily series
    s1 = df1[col1].resample("D").first()
    s2 = df2[col2].resample("D").first()

    delta = s2 - s1

    plot_df = delta.to_frame(name="delta_Ts")
    plot_df["month"] = plot_df.index.month
    plot_df["day"] = plot_df.index.day

    heatmap = plot_df.pivot(index="day", columns="month", values="delta_Ts")

    plt.figure(figsize=(10, 6))

    im = plt.imshow(
        heatmap,
        aspect="auto",
        origin="lower",
        cmap="viridis_r",
        vmin=vmin,
        vmax=vmax
    )

    cbar = plt.colorbar(
        im,
        ticks=[-20, -15, -10, -5, 0],
        label=f"Daily supply temperature reduction\n{label1} - {label2}"
    )

    plt.xticks(
        range(12),
        ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
         "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    )

    plt.yticks([0, 9, 19, 29], [1, 10, 20, 30])

    plt.xlabel("Month",fontsize=14)
    plt.ylabel("Day of month",fontsize=14)
    
    plot_title = f"Daily Supply Temperature Reduction - {label1} vs {label2}"
    plt.tight_layout()
    save_current_plot_from_title(plot_title)
    plt.show()

plot_daily_supply_temp_difference_heatmap(
    data_hourly_base,
    year_res,
    col1="supply_temp_base",
    col2="Ts_day_C",
    label1="Base",
    label2="Perfect",
    vmin=-20,
    vmax=0
)
    

In [ ]:

def plot_monthly_boxplots_two_scenarios(df1, df2, col1, col2,
                                        label1="Scenario 1",
                                        label2="Scenario 2"):
    s1 = df1[col1].resample("D").first()
    s2 = df2[col2].resample("D").first()

    plot_df = pd.DataFrame({
        label1: s1,
        label2: s2
    })
    plot_df["month"] = plot_df.index.strftime("%b")

    months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    data = []
    positions = []
    pos = 1

    for m in months:
        vals1 = plot_df.loc[plot_df["month"] == m, label1].dropna()
        vals2 = plot_df.loc[plot_df["month"] == m, label2].dropna()
        if len(vals1) > 0 and len(vals2) > 0:
            data.extend([vals1, vals2])
            positions.extend([pos, pos + 0.35])
            pos += 1

    plt.figure(figsize=(13, 6))

    cmap = plt.cm.viridis_r
    color1 = cmap(0.85)   # purple
    color2 = cmap(0.20)   # yellow/green

    bp = plt.boxplot(
        data,
        positions=positions,
        widths=0.25,
        patch_artist=True,
        flierprops=dict(
            markersize=3,
            markerfacecolor="gray",
            markeredgecolor="gray"
        )
    )

    for i, patch in enumerate(bp["boxes"]):
        if i % 2 == 0:
            patch.set_facecolor(color1)
            patch.set_alpha(0.75)
        else:
            patch.set_facecolor(color2)
            patch.set_alpha(0.75)

    plt.legend(
        handles=[
            Patch(facecolor=color1, alpha=0.95, label=label1),
            Patch(facecolor=color2, alpha=0.95, label=label2)
        ]
    )

    plt.xticks([i + 0.175 for i in range(1, 13)], months)
    plt.ylabel("Minimum supply temperature (°C)", fontsize=14)
    plt.xlabel("Month", fontsize=14)

    plt.grid(axis="y")
    plt.legend(
    handles=[
        Patch(facecolor=color1, alpha=0.75, label=label1),
        Patch(facecolor=color2, alpha=0.75, label=label2)
    ],
    loc="best")
    plt.tight_layout()
    plot_title = f"Monthly Boxplots of Daily Supply Temperature - {label1} vs {label2}"
    save_current_plot_from_title(plot_title)
    plt.show()
    

In [ ]:
plot_monthly_boxplots_two_scenarios(
    data_hourly_base,
    year_res,
    col1="supply_temp_base",
    col2="Ts_day_C",
    label1="Base",
    label2="Perfect"
)

In [ ]:
def prepare_temperature_series(s, reduce_to_daily=False, daily_method="first"):
    """
    Prepare a temperature series for comparison.

    Parameters
    ----------
    s : pd.Series
        Datetime-indexed temperature series.
    reduce_to_daily : bool
        If True, convert to one value per day.
    daily_method : str
        'first', 'mean', 'max', 'min'

    Returns
    -------
    pd.Series
    """
    s = s.copy()
    s.index = pd.to_datetime(s.index)
    s = s.sort_index()

    if reduce_to_daily:
        if daily_method == "first":
            s = s.resample("D").first()
        elif daily_method == "mean":
            s = s.resample("D").mean()
        elif daily_method == "max":
            s = s.resample("D").max()
        elif daily_method == "min":
            s = s.resample("D").min()
        else:
            raise ValueError("daily_method must be one of: first, mean, max, min")

    return s.dropna()


def temperature_stability_metrics(s):
    """
    Calculate variability / stability metrics for one temperature series.
    """
    delta = s.diff().dropna()

    metrics = {
        "mean_temp_C": s.mean(),
        "median_temp_C": s.median(),
        "std_temp_C": s.std(),
        "variance_temp_C2": s.var(),
        "iqr_temp_C": s.quantile(0.75) - s.quantile(0.25),
        "range_temp_C": s.max() - s.min(),
        "mad_from_median_C": np.median(np.abs(s - s.median())),
        "mean_abs_change_C": delta.abs().mean(),
        "std_change_C": delta.std(),
        "p95_abs_change_C": delta.abs().quantile(0.95),
        "max_abs_change_C": delta.abs().max(),
    }

    if abs(s.mean()) > 1e-9:
        metrics["coef_of_variation"] = s.std() / s.mean()
    else:
        metrics["coef_of_variation"] = np.nan

    return pd.Series(metrics)


def compare_temperature_stability(
    s1,
    s2,
    label1="Scenario 1",
    label2="Scenario 2",
    reduce_to_daily=False,
    daily_method="first"
):
    """
    Compare stability metrics between two yearly temperature series.

    Returns
    -------
    comparison_df : pd.DataFrame
    reduction_df : pd.DataFrame
    """
    s1 = prepare_temperature_series(s1, reduce_to_daily=reduce_to_daily, daily_method=daily_method)
    s2 = prepare_temperature_series(s2, reduce_to_daily=reduce_to_daily, daily_method=daily_method)

    # align on common timestamps
    common_index = s1.index.intersection(s2.index)
    s1 = s1.loc[common_index]
    s2 = s2.loc[common_index]

    m1 = temperature_stability_metrics(s1)
    m2 = temperature_stability_metrics(s2)

    comparison_df = pd.concat([m1, m2], axis=1)
    comparison_df.columns = [label1, label2]

    # reduction = how much lower label2 is than label1
    reduction_pct = (comparison_df[label1] - comparison_df[label2]) / comparison_df[label1] * 100

    reduction_df = pd.DataFrame({
        "metric": comparison_df.index,
        f"{label1}": comparison_df[label1].values,
        f"{label2}": comparison_df[label2].values,
        f"reduction_pct_{label2}_vs_{label1}": reduction_pct.values
    })

    return comparison_df, reduction_df


def print_stability_interpretation(reduction_df, label1="Scenario 1", label2="Scenario 2"):
    """
    Prints a short interpretation of the main stability metrics.
    """
    key_metrics = [
        "std_temp_C",
        "iqr_temp_C",
        "mean_abs_change_C",
        "p95_abs_change_C",
        "max_abs_change_C"
    ]

    print(f"Temperature stability comparison: {label2} versus {label1}\n")

    for metric in key_metrics:
        row = reduction_df.loc[reduction_df["metric"] == metric]
        if not row.empty:
            val = row.iloc[0][f"reduction_pct_{label2}_vs_{label1}"]
            if pd.notna(val):
                if val > 0:
                    print(f"- {metric}: {val:.1f}% lower in {label2} -> more stable")
                elif val < 0:
                    print(f"- {metric}: {abs(val):.1f}% higher in {label2} -> less stable")
                else:
                    print(f"- {metric}: no difference")

In [ ]:
comparison_df, reduction_df = compare_temperature_stability(
    s1=data_hourly_base["supply_temp_base"],
    s2=year_res["Ts_day_C"],
    label1="Base",
    label2="perfect",
    reduce_to_daily=False
)

print(comparison_df.round(2))
print(reduction_df.round(2))
print_stability_interpretation(reduction_df, label1="Base", label2="perfect")

#calculate the average daily reduction in supply temperature (base - buffer) across the year
daily_base = data_hourly_base["supply_temp_base"].resample("D").first()
daily_buffer = year_res["Ts_day_C"].resample("D").first()
daily_delta = daily_base - daily_buffer
average_daily_reduction_C = daily_delta.mean()
print(f"Average daily reduction in supply temperature (Base - Buffer): {average_daily_reduction_C:.2f} °C")
#average reduction without the months may-september
daily_delta_cold_season = daily_delta[daily_delta.index.month.isin([10,11,12,1,2,3,4])]
average_reduction_cold_season_C = daily_delta_cold_season.mean()
print(f"Average daily reduction in supply temperature during cold season (Base - Buffer): {average_reduction_cold_season_C:.2f} °C")

In [ ]:
comparison_real_df, reduction_real_df = compare_temperature_stability(
    s1=data_hourly["AVR_ARN_TA"],
    s2=year_res["Ts_day_C"],
    label1="Real",
    label2="Perfect",
    reduce_to_daily=False
)
print(comparison_real_df.round(2))
print(reduction_real_df.round(2))

In [ ]:
def seasonal_performance_factor(df, hp_heat_col, cop_col):
    q = df[hp_heat_col].fillna(0)          # MW or MWh/h
    cop = df[cop_col].replace(0, pd.NA)

    elec = q / cop
    spf = q.sum() / elec.sum()  

    return spf

In [ ]:
spf_base = seasonal_performance_factor(data_hourly_base, "hp_heat_MW", "COP")
spf_new  = seasonal_performance_factor(year_res, "hp_heat_MW", "COP")

spf_improvement_pct = (spf_new - spf_base) / spf_base * 100
electricity_saving_pct = (1 - spf_base / spf_new) * 100

print("SPF base:", round(spf_base, 2))
print("SPF new:", round(spf_new, 2))
print("SPF improvement (%):", round(spf_improvement_pct, 1))
print("Electricity saving for same heat output (%):", round(electricity_saving_pct, 1))

In [ ]:
#print data hourly colums
print(data_hourly.columns)

In [ ]:
#plot dot chart with T_buiten on x axis and supply_temp_real on y axis of data_hourly, add Ts_min_no_buffer in different color, decrease dot size
data_hourly["Ts_day_C"]=year_res["Ts_day_C"]
plt.figure(figsize=(10, 6))
#plt.scatter(data_hourly["T_buiten"], data_hourly["supply_temp_real"], label="Supply Temp (Real)", alpha=0.5, s=3)
plt.scatter(data_hourly["T_buiten"], data_hourly["Ts_min_no_buffer"], label="Supply temperature base", alpha=0.5, s=3)
plt.scatter(data_hourly["T_buiten"], data_hourly["Ts_day_C"], label="Supply temperature with TES", alpha=0.5, s=3)
plt.xlabel("Outdoor Temperature (°C)")
plt.ylabel("Supply Temperature (°C)")
plt.title("Supply Temperature vs Outdoor Temperature")
plt.legend()
plt.grid(True)
save_current_plot_from_title()
plt.show()

In [ ]:
totalyearlydemandarn = data_hourly["useful_demand_mw"].sum()
print("Total yearly useful demand ARN (MWh):", round(totalyearlydemandarn, 2))

In [ ]:
#print coloms data_hourly
print(data_hourly.columns)

In [ ]:
# --- Use one common 8740-hour index ---
common_index = data_hourly.index

# Optional check
print("Expected hours:", len(common_index))

# --- Extract series and force all to full hourly index ---
ta = pd.Series(data_hourly['AVR_ARN_TA'], index=data_hourly.index, name='TA')
tb = pd.Series(maximum_daily_temp_filled, index=data_hourly.index, name='Base')
ts = pd.Series(year_res['Ts_day_C'], name='With TES')

# --- Reindex TES to the same 8740 hours ---
ts = ts.reindex(common_index)

# --- Fill if needed ---
# Use forward/backward fill only if Ts_day_C is daily or has small gaps
ts = ts.ffill().bfill()

# --- Cap base temperature at 120 °C ---
tb = tb.clip(upper=120)

# --- Align all series to exactly the same index ---
ta = ta.reindex(common_index)
tb = tb.reindex(common_index)
ts = ts.reindex(common_index)

# --- Optional: check lengths ---
print("TA length:", len(ta))
print("Base length:", len(tb))
print("TES length:", len(ts))

print("TA missing:", ta.isna().sum())
print("Base missing:", tb.isna().sum())
print("TES missing:", ts.isna().sum())

# --- If there are still missing values, fill them ---
ta = ta.ffill().bfill()
tb = tb.ffill().bfill()
ts = ts.ffill().bfill()

# --- Sort descending for duration curves ---
ta_sorted = np.sort(ta.values)[::-1]
tb_sorted = np.sort(tb.values)[::-1]
ts_sorted = np.sort(ts.values)[::-1]

# --- X-axis: all 8740 cumulative hours ---
hours = np.arange(1, len(common_index) + 1)

# --- Plot ---
plt.figure(figsize=(10, 6))

plt.plot(hours, ta_sorted, label="Real 2024 supply temperature", linewidth=2)
plt.plot(hours, tb_sorted, label="Base", linewidth=2)
plt.plot(hours, ts_sorted, label="With TES", linewidth=2)

plt.xlabel('Cumulative hours', fontsize=14)
plt.ylabel('Temperature (°C)', fontsize=14)
plt.ylim(80, 120)
plt.xlim(0, len(common_index))
plt.grid(True, alpha=0.3)
plt.legend(fontsize=14)
plt.tight_layout()
plot_title = "Temperature duration curve"
save_current_plot_from_title(plot_title)
plt.show()

In [ ]:
Total_supplied_gas = summary_hp_perfect_temp["gas_heat_mwh"]
Total_supplied_avr = summary_hp_perfect_temp["avr_heat_mwh"]
Total_supplied_heat = Total_supplied_gas + Total_supplied_avr
Gas_share = Total_supplied_gas / Total_supplied_heat * 100
AVR_share = Total_supplied_avr / Total_supplied_heat * 100
print(f"Total supplied heat (MWh): {Total_supplied_heat:.2f}")
print(f"Gas share: {Gas_share:.1f}%")
print(f"AVR share: {AVR_share:.1f}%")
total_gas_supplied = data_hourly["GAS_SCH_MW"].sum()
gas_share_old = total_gas_supplied / totalyearlydemandarn * 100
print(f"Total gas supplied (MWh): {total_gas_supplied:.2f}")
print(f"Gas share old: {gas_share_old:.1f}%")

HEAT LOSS & TES HEAT LOSS


In [ ]:
#visualise heat loss per temperature from heat_loss_table
#plot coloms heat_loss_table
print(heat_loss_table.columns)
plt.figure(figsize=(10, 6))
plt.plot(heat_loss_table["Ts_C"], heat_loss_table["Heat_loss_mw"], label="Heat loss (MW)")
plt.xlabel("Supply Temperature (°C)", fontsize=14)
plt.ylabel("Heat loss (MW)", fontsize=14)
plt.grid(True)
plt.tight_layout()
plot_title="Heat loss as function of supply temperature in the primary network"
save_current_plot_from_title(plot_title)
plt.show()


In [ ]:
from thermal_storage import calculate_tes_heat_loss

In [ ]:

year_res["SOC"]=year_res["soc_mwh"]/(buffer_capacity_mwh_from_volume(buffer_volume_m3, T_hot_C=120, T_cold_C=55, rho=RHO_WATER, cp=CP_WATER))
print(year_res.columns)

In [ ]:
df_with_tes_loss, tes_loss_mwh, tes_loss_gj = calculate_tes_heat_loss(
    df=year_res,
    tes_size_m3=buffer_volume_m3,
    soc_col="SOC",
    T_hot_col="Ts_day_C",
    T_cold_col="Tret_C",
    T_ambient_col="T_ambient",
    k_wall=0.04,
    wall_thickness_m=0.3,
    h_out=8.0,
    ratio_height_diameter=2.0
)

In [ ]:
print(f"Annual TES heat loss: {tes_loss_mwh:.2f} MWh/year")
print(f"Annual TES heat loss: {tes_loss_gj:.2f} GJ/year")
plt.figure(figsize=(20, 4))
plt.plot(df_with_tes_loss.index, df_with_tes_loss["TES_heat_loss_MWh"])
plt.ylabel("Heat loss [MWh/h]")
plt.xlabel("Time")
plt.title("Hourly TES Heat Loss Over the Year")
plt.grid(True)
save_current_plot_from_title()
plt.show()

#daily heat loss of the TES, resample to daily and plot
daily_tes_loss = df_with_tes_loss["TES_heat_loss_MWh"].resample("D").sum()
plt.figure(figsize=(12, 6))
plt.plot(daily_tes_loss.index, daily_tes_loss.values)
plt.ylabel("Daily TES heat loss MWh", fontsize=14)
plt.xlabel("Date",fontsize=14)

plt.grid(True)
plt.tight_layout()
plot_title = "Daily TES heat loss over the year"
save_current_plot_from_title(plot_title)
plt.show()


UNCERTAINTY AND SENSITIVITY ANALYSIS DELTA LCOH

In [ ]:
def capital_recovery_factor(r, n):
    return (r * (1 + r)**n) / ((1 + r)**n - 1)


def recalculate_total_cost_from_summary(summary, cost_params):
    """
    cALCULATE COSTS BASED ON SUMMARY AND COST PARAMETERS
    """

    avr_cost = summary["avr_heat_mwh"] * cost_params["avr_heat_cost_eur_per_mwh"]
    gas_cost = summary["gas_heat_mwh"] * cost_params["gas_heat_cost_eur_per_mwh"]
    pump_cost = summary["pump_energy_mwh"] * cost_params["electricity_cost_eur_per_mwh"]
    hp_cost = summary["hp_electricity_mwh"] * cost_params["electricity_cost_eur_per_mwh"]
    variable_om = summary["total_heat_delivered_mwh"] * cost_params["variable_om_eur_per_mwh"]
    fixed_om = cost_params["fixed_om_eur_per_year"]

    total_cost = (
        avr_cost
        + gas_cost
        + pump_cost
        + hp_cost
        + variable_om
        + fixed_om
    )

    return total_cost

def calculate_delta_lcoh_from_summaries(
    summary_base,
    summary_tes,
    cost_params,
    buffer_volume_m3,
    price_buffer_eur_per_m3,
    r=0.06,
    n=40,
    heat_delivered_factor=1.0
):


    base_opex = recalculate_total_cost_from_summary(summary_base, cost_params)
    tes_opex = recalculate_total_cost_from_summary(summary_tes, cost_params)

    delta_opex = tes_opex - base_opex

    capex = buffer_volume_m3 * price_buffer_eur_per_m3
    crf = capital_recovery_factor(r, n)
    annualized_capex = capex * crf

    heat_delivered = summary_base["total_heat_delivered_mwh"] * heat_delivered_factor

    delta_lcoh = (annualized_capex + delta_opex) / heat_delivered

    return {
        "base_opex": base_opex,
        "tes_opex": tes_opex,
        "delta_opex": delta_opex,
        "capex": capex,
        "annualized_capex": annualized_capex,
        "heat_delivered": heat_delivered,
        "heat_delivered_factor": heat_delivered_factor,
        "delta_lcoh": delta_lcoh
    }

In [ ]:
def one_at_a_time_sensitivity(
    summary_base,
    summary_tes,
    base_cost_params,
    buffer_volume_m3,
    base_price_buffer_eur_per_m3,
    base_r=0.06,
    base_n=40
):
    results = []

    base_case = calculate_delta_lcoh_from_summaries(
        summary_base=summary_base,
        summary_tes=summary_tes,
        cost_params=base_cost_params,
        buffer_volume_m3=buffer_volume_m3,
        price_buffer_eur_per_m3=base_price_buffer_eur_per_m3,
        r=base_r,
        n=base_n
    )

    base_delta_lcoh = base_case["delta_lcoh"]

    sensitivity_ranges = {
        "AVR heat price": {
            "type": "cost_param",
            "key": "avr_heat_cost_eur_per_mwh",
            "low": 0.7,
            "high": 1.3
        },
        "Gas heat price": {
            "type": "cost_param",
            "key": "gas_heat_cost_eur_per_mwh",
            "low": 0.7,
            "high": 1.3
        },
        "Electricity price": {
            "type": "cost_param",
            "key": "electricity_cost_eur_per_mwh",
            "low": 0.7,
            "high": 1.3
        },
        "Buffer CAPEX": {
            "type": "buffer_price",
            "low": 0.7,
            "high": 1.3
        },
        "Discount rate": {
            "type": "discount_rate",
            "low": 0.03,
            "high": 0.09
        },
        "Lifetime": {
            "type": "lifetime",
            "low": 25,
            "high": 55
        }
    }

    for parameter, settings in sensitivity_ranges.items():

        for case in ["low", "high"]:

            cost_params = base_cost_params.copy()
            price_buffer = base_price_buffer_eur_per_m3
            r = base_r
            n = base_n

            if settings["type"] == "cost_param":
                key = settings["key"]
                cost_params[key] = base_cost_params[key] * settings[case]

            elif settings["type"] == "buffer_price":
                price_buffer = base_price_buffer_eur_per_m3 * settings[case]

            elif settings["type"] == "discount_rate":
                r = settings[case]

            elif settings["type"] == "lifetime":
                n = settings[case]

            output = calculate_delta_lcoh_from_summaries(
                summary_base=summary_base,
                summary_tes=summary_tes,
                cost_params=cost_params,
                buffer_volume_m3=buffer_volume_m3,
                price_buffer_eur_per_m3=price_buffer,
                r=r,
                n=n
            )

            results.append({
                "parameter": parameter,
                "case": case,
                "delta_lcoh": output["delta_lcoh"],
                "change_from_base": output["delta_lcoh"] - base_delta_lcoh
            })

    results_df = pd.DataFrame(results)

    return base_delta_lcoh, results_df

base_delta_lcoh, sensitivity_df = one_at_a_time_sensitivity(
    summary_base=summary_hp_base_temp,
    summary_tes=summary_hp_perfect_temp,
    base_cost_params=cost_params,
    buffer_volume_m3=buffer_volume_m3,
    base_price_buffer_eur_per_m3=price_buffer,
    base_r=0.06,
    base_n=40
)

print(f"Base ΔLCOH: {base_delta_lcoh:.2f} €/MWh")
display(sensitivity_df)

In [ ]:
def plot_tornado(sensitivity_df, base_delta_lcoh):
    tornado = sensitivity_df.pivot(
        index="parameter",
        columns="case",
        values="delta_lcoh"
    )

    tornado["low_effect"] = tornado["low"] - base_delta_lcoh
    tornado["high_effect"] = tornado["high"] - base_delta_lcoh

    tornado["min_effect"] = tornado[["low_effect", "high_effect"]].min(axis=1)
    tornado["max_effect"] = tornado[["low_effect", "high_effect"]].max(axis=1)
    tornado["range"] = tornado["max_effect"] - tornado["min_effect"]

    tornado = tornado.sort_values("range", ascending=True)

    plt.figure(figsize=(9, 6))

    for i, row in enumerate(tornado.itertuples()):
        plt.plot(
            [row.min_effect, row.max_effect],
            [i, i],
            linewidth=14
        )

    plt.axvline(0, linestyle="--")
    plt.yticks(range(len(tornado.index)), tornado.index,fontsize=14)
    plt.xlabel("Change in ΔLCOH relative to base case (€/MWh)", fontsize=14)
    

    plt.grid(axis="x")
    plt.tight_layout()
    plot_title = "One-at-a-time Sensitivity of ΔLCOH"
    save_current_plot_from_title(plot_title)
    plt.show()

    return tornado
tornado_df = plot_tornado(sensitivity_df, base_delta_lcoh)
display(tornado_df)

In [ ]:
def truncated_normal(rng, mean, std, lower, upper):
    value = rng.normal(mean, std)
    while value < lower or value > upper:
        value = rng.normal(mean, std)
    return value

In [ ]:
def truncated_normal(rng, mean, std, lower, upper):
    value = rng.normal(mean, std)
    while value < lower or value > upper:
        value = rng.normal(mean, std)
    return value
def monte_carlo_delta_lcoh(
    summary_base,
    summary_tes,
    base_cost_params,
    buffer_volume_m3,
    base_price_buffer_eur_per_m3,
    n_runs=5000,
    seed=42
):
    rng = np.random.default_rng(seed)

    results = []

    for i in range(n_runs):

        cost_params = base_cost_params.copy()

        # Example uncertainty assumptions
        cost_params["avr_heat_cost_eur_per_mwh"] = rng.triangular(
            0.9 * base_cost_params["avr_heat_cost_eur_per_mwh"],
            base_cost_params["avr_heat_cost_eur_per_mwh"],
            1.2 * base_cost_params["avr_heat_cost_eur_per_mwh"]
        )

        cost_params["gas_heat_cost_eur_per_mwh"] = rng.triangular(
            0.8 * base_cost_params["gas_heat_cost_eur_per_mwh"],
            base_cost_params["gas_heat_cost_eur_per_mwh"],
            1.4 * base_cost_params["gas_heat_cost_eur_per_mwh"]
        )

        cost_params["electricity_cost_eur_per_mwh"] = rng.triangular(
            0.8 * base_cost_params["electricity_cost_eur_per_mwh"],
            base_cost_params["electricity_cost_eur_per_mwh"],
            1.3 * base_cost_params["electricity_cost_eur_per_mwh"]
        )

        price_buffer = truncated_normal(
            rng,
            mean=base_price_buffer_eur_per_m3,
            std=0.10 * base_price_buffer_eur_per_m3,
            lower=0.70 * base_price_buffer_eur_per_m3,
            upper=1.30 * base_price_buffer_eur_per_m3
        )
        heat_delivered_factor = rng.uniform(0.9, 1.1)

        r = truncated_normal(
            rng,
            mean=0.06,
            std=(0.08 - 0.06) / 3,
            lower=0.04,
            upper=0.08
        )
        n = rng.triangular(left=35, mode=40, right=55)

        output = calculate_delta_lcoh_from_summaries(
            summary_base=summary_base,
            summary_tes=summary_tes,
            cost_params=cost_params,
            buffer_volume_m3=buffer_volume_m3,
            price_buffer_eur_per_m3=price_buffer,
            r=r,
            n=n,
            heat_delivered_factor=heat_delivered_factor
        )

        results.append({
            "delta_lcoh": output["delta_lcoh"],
            "delta_opex": output["delta_opex"],
            "annualized_capex": output["annualized_capex"],
            "avr_price": cost_params["avr_heat_cost_eur_per_mwh"],
            "gas_price": cost_params["gas_heat_cost_eur_per_mwh"],
            "electricity_price": cost_params["electricity_cost_eur_per_mwh"],
            "buffer_price": price_buffer,
            "discount_rate": r,
            "lifetime": n
        })

    return pd.DataFrame(results)
mc_df = monte_carlo_delta_lcoh(
    summary_base=summary_base,
    summary_tes=summary_perfect,
    base_cost_params=cost_params,
    buffer_volume_m3=buffer_volume_m3,
    base_price_buffer_eur_per_m3=price_buffer,
    n_runs=10000
)

display(mc_df.describe())

plt.figure(figsize=(9, 5))
plt.hist(mc_df["delta_lcoh"], bins=50)
plt.axvline(0, linestyle="--", label="Break-even", color="red")
plt.axvline(mc_df["delta_lcoh"].median(), linestyle="-", label="Median", color="red")
plt.xlabel("ΔLCOH (€/MWh)", fontsize=14)
plt.ylabel("Frequency", fontsize=14)

plt.legend(fontsize=14)
plt.grid(axis="y")
plt.tight_layout()
plot_title = "Monte Carlo Uncertainty Distribution of ΔLCOH"
save_current_plot_from_title(plot_title)
plt.show()

p_negative = (mc_df["delta_lcoh"] < 0).mean()

print(f"Mean ΔLCOH: {mc_df['delta_lcoh'].mean():.2f} €/MWh")
print(f"Median ΔLCOH: {mc_df['delta_lcoh'].median():.2f} €/MWh")
print(f"5th percentile: {mc_df['delta_lcoh'].quantile(0.05):.2f} €/MWh")
print(f"95th percentile: {mc_df['delta_lcoh'].quantile(0.95):.2f} €/MWh")
print(f"Probability ΔLCOH < 0: {p_negative:.1%}")

In [ ]:


scenarios = {
    "Real": summary_real,
    "Base": summary_base,
    "Perfect": summary_perfect,

    "Real + HP": summary_hp_real_temp,
    "Base + HP": summary_hp_base_temp,
    "Perfect + HP": summary_hp_perfect_temp,
}



def calculate_emissions_from_summary(
    summary,
    co2_gas_mwh,
    co2_avr_mwh,
    co2_electricity_mwh
):
    gas = summary["gas_heat_mwh"] * co2_gas_mwh
    avr = summary["avr_heat_mwh"] * co2_avr_mwh
    pump = summary["pump_energy_mwh"] * co2_electricity_mwh

    hp = 0
    if "hp_electricity_mwh" in summary:
        hp = summary["hp_electricity_mwh"] * co2_electricity_mwh

    total = gas + avr + pump + hp

    heat_delivered_gj = summary["total_heat_delivered_mwh"] * 3.6

    intensity = (
        total / heat_delivered_gj
        if heat_delivered_gj > 0
        else np.nan
    )

    return {
        "Gas": gas,
        "AVR": avr,
        "Pump": pump,
        "HP electricity": hp,
        "Total emissions kg": total,
        "Heat delivered GJ": heat_delivered_gj,
        "kg CO2 per GJ heat delivered": intensity,
    }



sensitivity_results = []

for year, ef in emission_factors.items():

    # Base emission factors
    base_co2_gas_mwh = ef["gas_GJ"] * 3.6
    base_co2_avr_mwh = ef["avr_GJ"] * 3.6
    base_co2_electricity_mwh = ef["elec_kWh"] * 1000

    for scenario_name, summary in scenarios.items():

        # Base-case emissions
        base_output = calculate_emissions_from_summary(
            summary=summary,
            co2_gas_mwh=base_co2_gas_mwh,
            co2_avr_mwh=base_co2_avr_mwh,
            co2_electricity_mwh=base_co2_electricity_mwh
        )

        base_total = base_output["Total emissions kg"]
        base_intensity = base_output["kg CO2 per GJ heat delivered"]

        # Sensitivity cases
        sensitivity_cases = {
            "AVR emission factor -20%": {
                "parameter": "AVR emission factor",
                "case": "-20%",
                "co2_gas_mwh": base_co2_gas_mwh,
                "co2_avr_mwh": base_co2_avr_mwh * 0.8,
                "co2_electricity_mwh": base_co2_electricity_mwh,
            },
            "AVR emission factor +20%": {
                "parameter": "AVR emission factor",
                "case": "+20%",
                "co2_gas_mwh": base_co2_gas_mwh,
                "co2_avr_mwh": base_co2_avr_mwh * 1.2,
                "co2_electricity_mwh": base_co2_electricity_mwh,
            },
            "Electricity emission factor -20%": {
                "parameter": "Electricity emission factor",
                "case": "-20%",
                "co2_gas_mwh": base_co2_gas_mwh,
                "co2_avr_mwh": base_co2_avr_mwh,
                "co2_electricity_mwh": base_co2_electricity_mwh * 0.8,
            },
            "Electricity emission factor +20%": {
                "parameter": "Electricity emission factor",
                "case": "+20%",
                "co2_gas_mwh": base_co2_gas_mwh,
                "co2_avr_mwh": base_co2_avr_mwh,
                "co2_electricity_mwh": base_co2_electricity_mwh * 1.2,
            },
        }

        for sensitivity_name, params in sensitivity_cases.items():

            output = calculate_emissions_from_summary(
                summary=summary,
                co2_gas_mwh=params["co2_gas_mwh"],
                co2_avr_mwh=params["co2_avr_mwh"],
                co2_electricity_mwh=params["co2_electricity_mwh"]
            )

            sensitivity_results.append({
                "Year": year,
                "Scenario": scenario_name,
                "Parameter": params["parameter"],
                "Case": params["case"],
                "Base total emissions kg": base_total,
                "Sensitivity total emissions kg": output["Total emissions kg"],
                "Change in total emissions kg": output["Total emissions kg"] - base_total,
                "Change in total emissions %": (
                    (output["Total emissions kg"] - base_total) / base_total * 100
                    if base_total > 0 else np.nan
                ),
                "Base kg CO2 per GJ": base_intensity,
                "Sensitivity kg CO2 per GJ": output["kg CO2 per GJ heat delivered"],
                "Change in kg CO2 per GJ": (
                    output["kg CO2 per GJ heat delivered"] - base_intensity
                ),
            })


sensitivity_emissions_df = pd.DataFrame(sensitivity_results)

display(sensitivity_emissions_df)

In [ ]:

compact_rows = []

for year, ef in emission_factors.items():

    base_co2_gas_mwh = ef["gas_GJ"] * 3.6
    base_co2_avr_mwh = ef["avr_GJ"] * 3.6
    base_co2_electricity_mwh = ef["elec_kWh"] * 1000

    for scenario_name, summary in scenarios.items():

        # Base case
        base_output = calculate_emissions_from_summary(
            summary=summary,
            co2_gas_mwh=base_co2_gas_mwh,
            co2_avr_mwh=base_co2_avr_mwh,
            co2_electricity_mwh=base_co2_electricity_mwh
        )

        base_total = base_output["Total emissions kg"]
        base_intensity = base_output["kg CO2 per GJ heat delivered"]

        # One-at-a-time sensitivity cases
        sensitivity_outputs = []

        cases = {
            "AVR -20%": {
                "co2_gas_mwh": base_co2_gas_mwh,
                "co2_avr_mwh": base_co2_avr_mwh * 0.8,
                "co2_electricity_mwh": base_co2_electricity_mwh
            },
            "AVR +20%": {
                "co2_gas_mwh": base_co2_gas_mwh,
                "co2_avr_mwh": base_co2_avr_mwh * 1.2,
                "co2_electricity_mwh": base_co2_electricity_mwh
            },
            "Electricity -20%": {
                "co2_gas_mwh": base_co2_gas_mwh,
                "co2_avr_mwh": base_co2_avr_mwh,
                "co2_electricity_mwh": base_co2_electricity_mwh * 0.8
            },
            "Electricity +20%": {
                "co2_gas_mwh": base_co2_gas_mwh,
                "co2_avr_mwh": base_co2_avr_mwh,
                "co2_electricity_mwh": base_co2_electricity_mwh * 1.2
            }
        }

        for case_name, params in cases.items():
            out = calculate_emissions_from_summary(
                summary=summary,
                co2_gas_mwh=params["co2_gas_mwh"],
                co2_avr_mwh=params["co2_avr_mwh"],
                co2_electricity_mwh=params["co2_electricity_mwh"]
            )

            sensitivity_outputs.append({
                "case": case_name,
                "total_kg": out["Total emissions kg"],
                "intensity_kg_per_gj": out["kg CO2 per GJ heat delivered"]
            })

        sens_df = pd.DataFrame(sensitivity_outputs)

        low_total = sens_df["total_kg"].min()
        high_total = sens_df["total_kg"].max()
        low_intensity = sens_df["intensity_kg_per_gj"].min()
        high_intensity = sens_df["intensity_kg_per_gj"].max()

        compact_rows.append({
            "Year": year,
            "Scenario": scenario_name,
            "Base emissions kt CO2": base_total / 1000,
            "Low emissions kt CO2": low_total / 1000,
            "High emissions kt CO2": high_total / 1000,
            "Range kt CO2": (high_total - low_total) / 1000,
            "Base kg CO2/GJ": base_intensity,
            "Low kg CO2/GJ": low_intensity,
            "High kg CO2/GJ": high_intensity,
            "Range kg CO2/GJ": high_intensity - low_intensity
        })


compact_emission_sensitivity_df = pd.DataFrame(compact_rows)

# Optional: order scenarios
scenario_order = [
    "Real",
    "Base",
    "Perfect",
    "6h",
    "6h_overdracht",
    "Real + HP",
    "Base + HP",
    "Perfect + HP",
    "6h + HP",
    "6h_overdracht + HP"
]

compact_emission_sensitivity_df["Scenario"] = pd.Categorical(
    compact_emission_sensitivity_df["Scenario"],
    categories=scenario_order,
    ordered=True
)

compact_emission_sensitivity_df = compact_emission_sensitivity_df.sort_values(
    ["Year", "Scenario"]
).reset_index(drop=True)

display(compact_emission_sensitivity_df)

In [ ]:
def plot_emission_sensitivity_ranges(
    df,
    selected_scenarios=None,
    title="Emission sensitivity range by scenario and year"
):
    plot_df = df.copy()

    if selected_scenarios is not None:
        plot_df = plot_df[plot_df["Scenario"].isin(selected_scenarios)].copy()


    base_col = "Base emissions kt CO2"
    low_col = "Low emissions kt CO2"
    high_col = "High emissions kt CO2"

    years = sorted(plot_df["Year"].unique())

    scenario_order = [
        "Real",
        "Base",
        "Perfect",
        "Real + HP",
        "Base + HP",
        "Perfect + HP"
    ]

    plot_df["Scenario"] = pd.Categorical(
        plot_df["Scenario"],
        categories=scenario_order,
        ordered=True
    )

    fig, axes = plt.subplots(
        nrows=len(years),
        ncols=1,
        figsize=(10, 2.8 * len(years)),
        sharex=True
    )

    if len(years) == 1:
        axes = [axes]

    color_no_hp = "#1f77b4"
    color_hp = "#ff7f0e"

    for ax, year in zip(axes, years):

        year_df = plot_df[plot_df["Year"] == year].copy()
        year_df = year_df.sort_values("Scenario", ascending=True)

        y_pos = np.arange(len(year_df))

        base = year_df[base_col].values
        low = year_df[low_col].values
        high = year_df[high_col].values

        lower_error = base - low
        upper_error = high - base

        colors = [
            color_hp if "HP" in str(scenario) else color_no_hp
            for scenario in year_df["Scenario"]
        ]

        for i, scenario in enumerate(year_df["Scenario"]):
            ax.errorbar(
                base[i],
                y_pos[i],
                xerr=np.array([[lower_error[i]], [upper_error[i]]]),
                fmt="o",
                color=colors[i],
                ecolor=colors[i],
                elinewidth=5,
                capsize=5,
                markersize=7
            )

        ax.set_yticks(y_pos)
        ax.set_yticklabels(year_df["Scenario"])
        ax.set_title(str(year))
        ax.grid(axis="x", alpha=0.35)
        ax.set_ylabel("Scenario")

        # Optional: remove top/right borders for cleaner thesis style
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[-1].set_xlabel("Total CO₂ emissions (t CO₂/year)")

    # Manual legend
    handles = [
        plt.Line2D([0], [0], marker="o", color=color_no_hp, linewidth=5, label="Without HP"),
        plt.Line2D([0], [0], marker="o", color=color_hp, linewidth=5, label="With HP")
    ]

    fig.legend(
        handles=handles,
        loc="upper center",
        ncol=2,
        bbox_to_anchor=(0.5, 1.02)
    )

    

    plt.tight_layout()
    plot_title = "Emission sensitivity range by scenario and year"
    save_current_plot_from_title(plot_title)
    plt.show()


selected_scenarios = [
    "Real",
    "Base",
    "Perfect",
    "Real + HP",
    "Base + HP",
    "Perfect + HP"
]

plot_emission_sensitivity_ranges(
    compact_emission_sensitivity_df,
    selected_scenarios=selected_scenarios
)

In [ ]:

table_compact = compact_emission_sensitivity_df.copy()

table_compact = table_compact[[
    "Year",
    "Scenario",
    "Base emissions kt CO2",
    "Low emissions kt CO2",
    "High emissions kt CO2",
    "Range kt CO2",
    "Base kg CO2/GJ",
    "Low kg CO2/GJ",
    "High kg CO2/GJ",
    "Range kg CO2/GJ"
]]

table_compact = table_compact.round({
    "Base emissions kt CO2": 1,
    "Low emissions kt CO2": 1,
    "High emissions kt CO2": 1,
    "Range kt CO2": 1,
    "Base kg CO2/GJ": 2,
    "Low kg CO2/GJ": 2,
    "High kg CO2/GJ": 2,
    "Range kg CO2/GJ": 2
})

display(table_compact)


In [ ]:
def plot_emission_sensitivity_ranges_intensity(
    df,
    selected_scenarios=None,
    title="Emission intensity sensitivity range by scenario and year"
):
    plot_df = df.copy()

    if selected_scenarios is not None:
        plot_df = plot_df[plot_df["Scenario"].isin(selected_scenarios)].copy()

    base_col = "Base kg CO2/GJ"
    low_col = "Low kg CO2/GJ"
    high_col = "High kg CO2/GJ"

    years = sorted(plot_df["Year"].unique())

    scenario_order = [
        "Real",
        "Base",
        "Perfect",
        "Real + HP",
        "Base + HP",
        "Perfect + HP"
    ]

    plot_df["Scenario"] = pd.Categorical(
        plot_df["Scenario"],
        categories=scenario_order,
        ordered=True
    )

    fig, axes = plt.subplots(
        nrows=len(years),
        ncols=1,
        figsize=(10, 2.8 * len(years)),
        sharex=True
    )

    if len(years) == 1:
        axes = [axes]

    color_no_hp = "#1f77b4"
    color_hp = "#ff7f0e"

    for ax, year in zip(axes, years):

        year_df = plot_df[plot_df["Year"] == year].copy()
        year_df = year_df.sort_values("Scenario", ascending=True)

        y_pos = np.arange(len(year_df))

        base = year_df[base_col].values
        low = year_df[low_col].values
        high = year_df[high_col].values

        lower_error = base - low
        upper_error = high - base

        colors = [
            color_hp if "HP" in str(scenario) else color_no_hp
            for scenario in year_df["Scenario"]
        ]

        for i, scenario in enumerate(year_df["Scenario"]):
            ax.errorbar(
                base[i],
                y_pos[i],
                xerr=np.array([[lower_error[i]], [upper_error[i]]]),
                fmt="o",
                color=colors[i],
                ecolor=colors[i],
                elinewidth=5,
                capsize=5,
                markersize=7
            )

        ax.set_yticks(y_pos)
        ax.set_yticklabels(year_df["Scenario"])
        ax.set_title(str(year))
        ax.grid(axis="x", alpha=0.35)
        ax.set_ylabel("Scenario")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[-1].set_xlabel("Emission intensity (kg CO₂/GJ heat delivered)")

    handles = [
        plt.Line2D(
            [0], [0],
            marker="o",
            color=color_no_hp,
            linewidth=5,
            label="Without HP"
        ),
        plt.Line2D(
            [0], [0],
            marker="o",
            color=color_hp,
            linewidth=5,
            label="With HP"
        )
    ]

    fig.legend(
        handles=handles,
        loc="upper center",
        ncol=2,
        bbox_to_anchor=(0.5, 1.02)
    )

    

    plt.tight_layout()
    plot_title = "Emission intensity sensitivity range by scenario and year"
    save_current_plot_from_title(plot_title)
    plt.show()


plot_emission_sensitivity_ranges_intensity(
    compact_emission_sensitivity_df,
    selected_scenarios=selected_scenarios
)


In [ ]:

dispatch_summaries = {}
for scenario_name, summary in scenarios.items():

    dispatch_summary = {
        "Gas heat MWh": summary["gas_heat_mwh"],
        "AVR heat MWh": summary["avr_heat_mwh"],
        "Pump energy MWh": summary["pump_energy_mwh"],
        "HP electricity MWh": summary.get("hp_electricity_mwh", 0)
    }

    dispatch_summaries[scenario_name] = dispatch_summary
    #round to 2 decimals
dispatch_df = pd.DataFrame(dispatch_summaries).T
dispatch_df = dispatch_df.round(2)
display(dispatch_df)


In [ ]:


scenario_dispatch = {
    "Real": summary_real,
    "Base": summary_base,
    "Perfect": summary_perfect,
    "6h": summary_6h,
    "6h_overdracht": summary_6h_overdracht,
    "Real + HP": summary_hp_real_temp,
    "Base + HP": summary_hp_base_temp,
    "Perfect + HP": summary_hp_perfect_temp,
    "6h + HP": summary_hp_6h_temp,
    "6h_overdracht + HP": summary_hp_6h_overdracht_temp
}
dispatch_summaries = {}
for scenario_name, summary in scenario_dispatch.items():

    dispatch_summary = {
        "Gas heat MWh": summary["gas_heat_mwh"],
        "AVR heat MWh": summary["avr_heat_mwh"],
        "Pump energy MWh": summary["pump_energy_mwh"],
        "HP electricity MWh": summary.get("hp_electricity_mwh", 0)
    }

    dispatch_summaries[scenario_name] = dispatch_summary
    #round to 2 decimals
dispatch_df = pd.DataFrame(dispatch_summaries).T
dispatch_df = dispatch_df.round(2)
display(dispatch_df)